In [ ]:
#@title LCMS Method Summary (ZIP Upload)
#@markdown # LCMS Method Summary (ZIP Upload)
#@markdown **Quick Instructions**
#@markdown 1. On your local machine, zip your method folder(s).
#@markdown    - You can zip each `*.m` folder separately, or
#@markdown    - put multiple `*.m` folders into one ZIP file.
#@markdown 2. Run this cell (hit Runtime -> Run all).
#@markdown 3. Click the **Choose Files** button and select your `.zip` file(s).
#@markdown    - Note: it may take a few seconds for the button to appear.
#@markdown 4. The script runs automatically and writes:
#@markdown    - `/content/sample_data/lc_method_summary.csv`
#@markdown    - `/content/sample_data/lc_method_summary.html`
#@markdown 5. HTML preview + file downloads are triggered automatically.
#@markdown ---
#@markdown ZIPs must contain one or more `*.m` method folders.

# Colab one-cell workflow: includes viewer code inline + ZIP upload run
import sys
import subprocess
import tempfile
import zipfile
from pathlib import Path


def _ensure_package(pkg: str) -> None:
    try:
        __import__(pkg)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


for _pkg in ("pandas", "plotly"):
    _ensure_package(_pkg)

try:
    from google.colab import files  # type: ignore
except Exception as exc:
    raise RuntimeError(f"This notebook is intended for Google Colab: {exc}")

from __future__ import annotations

import html
import hashlib
import json
import re
import struct
import sys
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any
import xml.etree.ElementTree as ET

import pandas as pd


REQUIRED_XML_FILES = ("info.xml", "BinPump_1.xml", "ALS_1.xml", "TCC_1.xml")
OUTPUT_CSV = "lc_method_summary.csv"
OUTPUT_HTML = "lc_method_summary.html"
VIEWER_VERSION = "0.1.0"
QTOF_REPORT_PATTERN = re.compile(r"<\?xml[^>]*\?>\s*<QTOFTuneReport[\s\S]*?</QTOFTuneReport>")
CFBF_SIGNATURE = b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1"
CFBF_BYTE_ORDER_LE = 0xFFFE
CFBF_FREE_SECTOR = 0xFFFFFFFF
CFBF_END_OF_CHAIN = 0xFFFFFFFE
CFBF_FAT_SECTOR = 0xFFFFFFFD
CFBF_DIFAT_SECTOR = 0xFFFFFFFC
CFBF_NO_STREAM = 0xFFFFFFFF
CFBF_DIRECTORY_ENTRY_SIZE = 128
MSMETHOD_FIXED_HEADER_SIZE = 56
MSMETHOD_EXPECTED_FIXED_VALUE_300 = 300
MSMETHOD_TIME_SEGMENT_HEADER_SIZE = 16
MSMETHOD_PRIMARY_FIELD_COUNT = MSMETHOD_EXPECTED_FIXED_VALUE_300
MSMETHOD_TRAILING_WRAP_COLUMNS = 12
MSMETHOD_TRAILING_SPECIAL_WRAP_COLUMNS = 3
MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS = (
    MSMETHOD_TRAILING_WRAP_COLUMNS - MSMETHOD_TRAILING_SPECIAL_WRAP_COLUMNS
)
MSMETHOD_TRAILING_COMPACTION_RULES: tuple[tuple[tuple[int, ...], int, int], ...] = (
    # (signature, insert_blank_after_n_values, blank_cell_count)
    ((65535, 65535, 0, 0), 6, 1),
    ((65535, 65535, 65535, 0), 6, 0),
    ((65535, 65535, 0, 65535), 6, 1),
)
MSMETHOD_TRAILING_TYPE_BY_CODE: dict[int, str] = {
    0: "TIC",
    1: "EIC",
    2: "Setpoint",
    3: "Actual",
}
MSMETHOD_TRAILING_EXPT_TYPE_BY_CODE: dict[int, str] = {
    0: "Both",
    1: "MS",
    2: "MS/MS",
}
MSMETHOD_TRAILING_POLARITY_BY_CODE: dict[int, str] = {
    0: "Positive",
    1: "Negative",
    2: "Both",
}
MSMETHOD_PRIMARY_COLUMN_CATEGORY_RULES: tuple[tuple[str, tuple[str, ...]], ...] = (
    ("Core", (
        "Model",
        "ion_source",
        "no_limit_as_pump",
        "stop_time [min]",
    )),
    ("General", (
        "polarity", 
        "lc_stream",
        "data_storage",
        "centroid_abs_threshold (MS)",
        "centroid_rel_threshold (MS) [%]",
        "centroid_abs_threshold (MS/MS)",
        "centroid_rel_threshold (MS/MS) [%]",
        "profile_threshold (MS)",
        "profile_threshold (MS/MS)",
        "do_not_wait_for_setpoints_to_equilibrate",
        "bioconfirm_max_entropy_mode",
    )),
    ("Source", (
        "gas_temp [ºC]",
        "drying_gas [L/min]",
        "nebulizer [psi]",
        "sheath_gas_temp [ºC]",
        "sheath_gas_flow [L/min]",
        "vcap [V]",
        "nozzle_voltage [V]",
        "fragmentor [V]",
        "skimmer [V]",
        "oct_1_rf_vpp [V]",
    )),
    ("Acquisition", (
        "range_min [m/z]",
        "range_max [m/z]",
        "rate [spectra/s]",
        "[transients/spectrum]",
        "collision_energy [V]",
    )),
    ("Ref Mass", (
        "correction",
        "use_bottle_a",
        "detection_window [ppm]",
        "minimum_height [counts]",
    )),
    ("misc", (
        "targetedlist",
        "calibration_xml [pos]",
        "calibration_xml [neg]",
        "xml_comment",
    ))
)
MSMETHOD_PRIMARY_METHOD_CATEGORY_NAME = "method"
MSMETHOD_PRIMARY_OTHER_CATEGORY_NAME = "other"
# Generic slot index (0-based within trailing generic row values) hidden from display.
# slot 1 corresponds to legacy "col_2".
MSMETHOD_TRAILING_HIDDEN_GENERIC_SLOT_INDEXES = frozenset({1})
MSMETHOD_UNKNOWN_FIELD_NAME_OVERRIDES: dict[int, str] = {
    6: "skimmer [V]",
    7: "oct_1_rf_vpp [V]",
    11: "gas_temp [ºC]",
    13: "drying_gas [L/min]",
    14: "nebulizer [psi]",
    15: "polarity",
    16: "fragmentor [V]",
    17: "stop_time [min]",
    19: "ion_source",
    20: "no_limit_as_pump",
    21: "data_storage",
    25: "lc_stream",
    26: "centroid_abs_threshold (MS)",
    27: "centroid_rel_threshold (MS) [%]",
    29: "use_bottle_a",
    30: "correction",
    32: "detection_window [ppm]",
    33: "minimum_height [counts]",
    35: "do_not_wait_for_setpoints_to_equilibrate",
    41: "centroid_abs_threshold (MS/MS)",
    42: "centroid_rel_threshold (MS/MS) [%]",
    129: "range_min [m/z]",     # [lower_limit, user_defined_lower_range (used value), ???]
    130: "range_max [m/z]",     # [upper_limit, user_defined_upper_range (used value), ???]
    164: "vcap [V]",
    166: "sheath_gas_flow [L/min]",
    168: "sheath_gas_temp [ºC]",
    170: "nozzle_voltage [V]",
    173: "rate [spectra/s]",
    206: "collision_energy [V]",
    257: "[transients/spectrum]",
    266: "profile_threshold (MS)",
    267: "profile_threshold (MS/MS)",
    273: "Model",
    277: "targetedlist",
    278: "bioconfirm_max_entropy_mode",
    279: "calibration_xml [pos]",
    283: "calibration_xml [neg]",
    284: "xml_comment"
}
MSMETHOD_TRAILING_UNKNOWN_FIELD_NAME_OVERRIDES: dict[int, str] = {
    13: "ref_mass (pos)",
    14: "ref_mass (neg)",
}
MSMETHOD_REF_MASS_POS_FIELD_INDEX = 13
MSMETHOD_REF_MASS_NEG_FIELD_INDEX = 14
MSMETHOD_REF_MASS_FIELD_INDEXES = frozenset(
    {MSMETHOD_REF_MASS_POS_FIELD_INDEX, MSMETHOD_REF_MASS_NEG_FIELD_INDEX}
)
MSMETHOD_PRIMARY_POLARITY_FIELD_INDEXES = frozenset({15})
MSMETHOD_ZERO_ONE_BOOL_FIELD_INDEXES = frozenset({29, 30, 35})
MSMETHOD_ENUM_BY_FIELD_INDEX: dict[int, dict[int, str]] = {
    19: {
        0: "esi",
        1: "hplc_chip",
        2: "nanoesi",
        3: "maldi",
        4: "multimode",
        5: "dual_esi",
        6: "dual_nanoesi",
        7: "apci",
        8: "appi",
        9: "ajs_esi",
        10: "gc_apci",
        11: "dual_ajs_esi",
        12: "desi",
    },
    20: {
        0: "stop_time",
        1: "no_limit_as_pump",
    },
    25: {
        0: "waste",
        1: "ms",
    },
    21: {
        0: "none",
        1: "profile",
        2: "centroid",
        3: "both",
    },
    278: {
        0: "false",
        4: "true",
    }
}
MSMETHOD_STRICT_ENUM_FIELD_INDEXES = frozenset({278})
MSMETHOD_CALIBRATION_XML_FIELD_INDEXES = frozenset({279, 283})
MSMETHOD_MASS_RANGE_TRIPLET_FIELD_INDEXES = frozenset({129, 130})
MSMETHOD_CALIBRATION_XML_CELL_TEXT = "See details below"
MSMETHOD_XML_PREFIX_PATTERN = re.compile(r"^\s*(<\?xml\b|<[A-Za-z_][\w:.\-]*)")
PLOTLY_COLORWAY = (
    "#636EFA",
    "#EF553B",
    "#00CC96",
    "#AB63FA",
    "#FFA15A",
    "#19D3F3",
    "#FF6692",
    "#B6E880",
    "#FF97FF",
    "#FECB52",
)
DEFAULT_UNCHECKED_LC_COLUMNS = {
    # "method_dir",
    "description",
    "created_at",
    "modified_at",
    "estimated_run_time_min",
    # "flow_mL_min",
    # "stop_time_min",
    # "post_time_min",
    "low_pressure_limit_bar",
    "high_pressure_limit_bar",
    "max_flow_ramp",
    # "solvent_a_name",
    # "solvent_b_name",
    "initial_percent_a",
    "initial_percent_b",
    "gradient_profile",
    "gradient_profile_json",
    "injection_mode",
    # "injection_volume_uL",
    "needle_wash_location",
    "needle_wash_time_s",
    "draw_speed",
    "eject_speed",
    "wait_time_after_draw_s",
    "draw_position_offset",
    "sample_flush_out_factor",
    "valve_to_bypass_after_injection",
    # "column_valve_position",
    "left_temp_mode",
    # "left_temp_C",
    "left_not_ready_limit",
    "right_temp_mode",
}

# MS Method Details table (method_dir + unknown_* columns).
# Add any column names here to hide them by default.
# Unknown fields use names like "unknown_0", "unknown_1", ... unless overridden.
DEFAULT_UNCHECKED_MS_METHOD_DETAILS_COLUMNS = {
    # "unknown_0",
    # "unknown_1",
    "calibration_xml [pos]",
    "calibration_xml [neg]",
}
DEFAULT_UNCHECKED_MS_METHOD_DETAILS_EXTENSION_COLUMNS = {
    # "unknown_0",
}
DEFAULT_UNCHECKED_MS_METHOD_DETAILS_TRAILING_COLUMNS = {
    # "unknown_0",
    "unknown_1",
    "unknown_2",
}

DEFAULT_UNCHECKED_TUNE_COLUMNS = {
    # "method_dir",
    # "ms_tune_datetime",
    "ms_model",
    "ms_serial_number",
    "ms_firmware_version",
    # "ms_source_type",
    # "ms_mass_range",
    # "ms_instrument_mode",
    "ms_suremass",
    # "ms_min_range_mz",
    # "ms_max_range_mz",
    # "ms_acquisition_rate_hz",
    "ms_acquisition_time_ms",
    # "ms_set_src_gas_temp_C",
    # "ms_set_src_drying_gas_L_min",
    # "ms_set_src_neb_psi",
    # "ms_set_src_vcap_V",
    # "ms_set_src_nozzle_voltage_V",
    # "ms_set_src_sheath_gas_temp_C",
    # "ms_set_src_sheath_gas_flow_L_min",
    # "ms_act_src_gas_temp_C",
    # "ms_act_src_vapor_temp_C",
    # "ms_act_src_drying_gas_L_min",
    # "ms_act_src_neb_psi",
    # "ms_act_src_vcap_V",
    "ms_tune_report_type",
    "ms_tune_report_title",
    "ms_tune_data_path",
    "ms_tune_user",
    "ms_tune_last_user",
    "ms_tune_slicer_mode",
    "ms_tune_state_save_delta",
    "ms_tune_state_gain_channel",
    "ms_tune_state_high_resolution_enabled",
    "ms_tune_cal_a",
    "ms_tune_cal_b",
    "ms_tune_cal_c",
    "ms_tune_cal_d",
    "ms_tune_cal_e",
    "ms_tune_cal_f",
    "ms_tune_cal_poly_terms_flag",
    "ms_tune_cal_trad_weighting",
    "ms_tune_cal_poly_weighting",
    "ms_tune_vac_quad_temp_C",
    "ms_tune_vac_rough_torr",
    "ms_tune_vac_high_torr",
    "ms_tune_vac_octknee_torr",
    "ms_tune_vac_turbo1_pwr_w",
    "ms_tune_vac_turbo1_spd_pct",
    "ms_tune_vac_turbo2_pwr_w",
    "ms_tune_vac_turbo2_spd_pct",
    "ms_tune_cal_point_count",
    "ms_tune_expected_mass_min_mz",
    "ms_tune_expected_mass_max_mz",
    "ms_tune_resolution_mean",
    "ms_tune_resolution_min",
    "ms_tune_resolution_max",
    # "ms_set_opt1_fragment_V",
    # "ms_set_opt1_skim1_V",
    # "ms_set_opt1_oct_peak_V",
    "ms_set_opt1_lens1_V",
    "ms_set_opt1_lens2_V",
    "ms_set_quad_mass_dac_amu",
    "ms_set_quad_quad_dc_V",
    "ms_set_cell_cc_flow_psi",
    "ms_set_cell_hex_rf_V",
    "ms_set_cell_skim2_V",
    "ms_set_opt2_extractor_dc_V",
    "ms_set_opt2_ion_focus_V",
    "ms_set_opt2_slit_left_V",
    "ms_set_opt2_slit_right_V",
    "ms_set_tof_vpulse_V",
    "ms_set_tof_pusher_offset_mV",
    "ms_set_tof_vliner_V",
    "ms_set_acq_num_pts",
    "ms_set_acq_num_trans",
    "ms_set_det_vmcpback_V",
    "ms_set_det_preamp_offset",
    "ms_set_det_dualgain_ratio",
    "ms_act_src_corona_vol_V",
    "ms_act_src_cap_cur",
    "ms_act_src_cham_cur",
}

DEFAULT_UNCHECKED_BY_TABLE: dict[str, set[str]] = {
    "lc_method_table": DEFAULT_UNCHECKED_LC_COLUMNS,
    "ms_method_details_table": DEFAULT_UNCHECKED_MS_METHOD_DETAILS_COLUMNS,
    "ms_method_details_extension_table": DEFAULT_UNCHECKED_MS_METHOD_DETAILS_EXTENSION_COLUMNS,
    "ms_method_details_trailing_table": DEFAULT_UNCHECKED_MS_METHOD_DETAILS_TRAILING_COLUMNS,
    "ms_tune_positive_table": DEFAULT_UNCHECKED_TUNE_COLUMNS,
    "ms_tune_negative_table": DEFAULT_UNCHECKED_TUNE_COLUMNS,
}


@dataclass(frozen=True)
class GradientPoint:
    time_min: float
    percent_b: float


@dataclass(frozen=True)
class CfbfHeader:
    minor_version: int
    major_version: int
    byte_order: int
    sector_shift: int
    mini_sector_shift: int
    sector_size: int
    mini_sector_size: int
    num_dir_sectors: int
    num_fat_sectors: int
    first_dir_sector: int
    transaction_signature: int
    mini_stream_cutoff_size: int
    first_mini_fat_sector: int
    num_mini_fat_sectors: int
    first_difat_sector: int
    num_difat_sectors: int
    difat_header: list[int]


@dataclass(frozen=True)
class CfbfDirectoryEntry:
    entry_id: int
    name: str
    object_type: int
    color_flag: int
    left_sibling_id: int | None
    right_sibling_id: int | None
    child_id: int | None
    clsid_hex: str
    state_bits: int
    creation_time: int
    modified_time: int
    starting_sector: int
    stream_size: int


@dataclass(frozen=True)
class CfbfStreamRecord:
    entry_id: int
    name: str
    chain_type: str
    sector_chain: list[int]
    data: bytes


@dataclass(frozen=True)
class CfbfDecodeResult:
    header: CfbfHeader
    fat_sector_ids: list[int]
    fat: list[int]
    mini_fat: list[int]
    directory_sector_chain: list[int]
    directory_entries: list[CfbfDirectoryEntry]
    mini_stream: bytes
    streams: list[CfbfStreamRecord]


@dataclass(frozen=True)
class MsMethodMetadata:
    parse_status: str
    compobj_parse_status: str
    propertyset_parse_status: str
    com_class: str | None
    progid: str | None
    type_id: str | None
    method_id: str | None
    revision_number: int | None


@dataclass(frozen=True)
class MsMethodFixedHeader:
    unk0: int
    unk1: int
    n_time_segments: int
    fixed_value_300: int
    unk2: int
    unk3: int
    n_time_segments_2: int
    unk4: int
    unk5: int
    unk6: int
    unk7: int
    unk8: int
    unk9: int
    unk10: int


@dataclass(frozen=True)
class MsMethodTimeSegment:
    index: int
    header_offset: int
    n_fields: int
    n_experimental_segments: int
    start_time_raw_u64: int
    start_time: float
    payload_start_offset: int
    payload_end_offset: int
    tagged_values: tuple["MsMethodTaggedValue", ...]
    field_major_values: tuple[tuple["MsMethodTaggedValue", ...], ...]
    trailing_header_offset: int | None
    trailing_n_fields: int
    trailing_n_experimental_segments: int
    trailing_payload_start_offset: int | None
    trailing_payload_end_offset: int | None
    trailing_tagged_values: tuple["MsMethodTaggedValue", ...]
    trailing_field_major_values: tuple[tuple["MsMethodTaggedValue", ...], ...]
    primary_unknown_fields: tuple["MsMethodUnknownField", ...]
    extension_unknown_fields: tuple["MsMethodUnknownField", ...]


@dataclass(frozen=True)
class MsMethodArrayValue:
    rank: int
    dims: tuple[int, ...]
    values: tuple[float, ...] | tuple[int, ...]


@dataclass(frozen=True)
class MsMethodTaggedValue:
    offset: int
    type_code: int
    value: float | int | str | bool | MsMethodArrayValue | None


@dataclass(frozen=True)
class MsMethodUnknownField:
    index: int
    name: str
    values: tuple[MsMethodTaggedValue, ...]


@dataclass(frozen=True)
class MsMethodPostamble:
    start_offset: int
    end_offset: int
    raw_bytes: bytes
    tagged_values: tuple[MsMethodTaggedValue, ...]
    unknown_fields: tuple[MsMethodUnknownField, ...]


@dataclass(frozen=True)
class MsMethodParseResult:
    fixed_header: MsMethodFixedHeader
    time_segments: tuple[MsMethodTimeSegment, ...]
    postamble: MsMethodPostamble


@dataclass
class MethodSummary:
    method_dir: str
    description: str | None
    created_at: str | None
    modified_at: str | None
    estimated_run_time_min: float | None
    flow_mL_min: float | None
    stop_time_min: float | None
    post_time_min: float | None
    low_pressure_limit_bar: float | None
    high_pressure_limit_bar: float | None
    max_flow_ramp: float | None
    solvent_a_name: str | None
    solvent_b_name: str | None
    initial_percent_a: float | None
    initial_percent_b: float | None
    gradient_profile: str
    gradient_profile_json: str
    injection_mode: str | None
    injection_volume_uL: float | None
    needle_wash_location: str | None
    needle_wash_time_s: float | None
    draw_speed: float | None
    eject_speed: float | None
    wait_time_after_draw_s: float | None
    draw_position_offset: float | None
    sample_flush_out_factor: float | None
    valve_to_bypass_after_injection: str | None
    column_valve_position: str | None
    left_temp_mode: str | None
    left_temp_C: float | None
    left_not_ready_limit: float | None
    right_temp_mode: str | None
    ms_parse_status: str
    stg_size_bytes: int | None
    stg_md5: str | None
    ms_report_count: int | None
    ms_available_polarities: str | None
    ms_selected_polarity: str | None
    ms_tune_datetime: str | None
    ms_model: str | None
    ms_serial_number: str | None
    ms_firmware_version: str | None
    ms_source_type: str | None
    ms_mass_range: str | None
    ms_instrument_mode: str | None
    ms_suremass: str | None
    ms_min_range_mz: float | None
    ms_max_range_mz: float | None
    ms_acquisition_rate_hz: float | None
    ms_acquisition_time_ms: float | None
    ms_set_src_gas_temp_C: float | None
    ms_set_src_drying_gas_L_min: float | None
    ms_set_src_neb_psi: float | None
    ms_set_src_vcap_V: float | None
    ms_set_src_nozzle_voltage_V: float | None
    ms_set_src_sheath_gas_temp_C: float | None
    ms_set_src_sheath_gas_flow_L_min: float | None
    ms_act_src_gas_temp_C: float | None
    ms_act_src_vapor_temp_C: float | None
    ms_act_src_drying_gas_L_min: float | None
    ms_act_src_neb_psi: float | None
    ms_act_src_vcap_V: float | None
    ms_tune_report_type: str | None
    ms_tune_report_title: str | None
    ms_tune_data_path: str | None
    ms_tune_user: str | None
    ms_tune_last_user: str | None
    ms_tune_slicer_mode: str | None
    ms_tune_state_save_delta: str | None
    ms_tune_state_gain_channel: str | None
    ms_tune_state_high_resolution_enabled: str | None
    ms_tune_cal_a: float | None
    ms_tune_cal_b: float | None
    ms_tune_cal_c: float | None
    ms_tune_cal_d: float | None
    ms_tune_cal_e: float | None
    ms_tune_cal_f: float | None
    ms_tune_cal_poly_terms_flag: str | None
    ms_tune_cal_trad_weighting: float | None
    ms_tune_cal_poly_weighting: float | None
    ms_tune_vac_quad_temp_C: float | None
    ms_tune_vac_rough_torr: float | None
    ms_tune_vac_high_torr: float | None
    ms_tune_vac_octknee_torr: float | None
    ms_tune_vac_turbo1_pwr_w: float | None
    ms_tune_vac_turbo1_spd_pct: float | None
    ms_tune_vac_turbo2_pwr_w: float | None
    ms_tune_vac_turbo2_spd_pct: float | None
    ms_tune_cal_point_count: int | None
    ms_tune_expected_mass_min_mz: float | None
    ms_tune_expected_mass_max_mz: float | None
    ms_tune_resolution_mean: float | None
    ms_tune_resolution_min: float | None
    ms_tune_resolution_max: float | None
    ms_mode_hint: str
    ms_set_opt1_fragment_V: float | None
    ms_set_opt1_skim1_V: float | None
    ms_set_opt1_oct_peak_V: float | None
    ms_set_opt1_lens1_V: float | None
    ms_set_opt1_lens2_V: float | None
    ms_set_quad_mass_dac_amu: float | None
    ms_set_quad_quad_dc_V: float | None
    ms_set_cell_cc_flow_psi: float | None
    ms_set_cell_hex_rf_V: float | None
    ms_set_cell_skim2_V: float | None
    ms_set_opt2_extractor_dc_V: float | None
    ms_set_opt2_ion_focus_V: float | None
    ms_set_opt2_slit_left_V: float | None
    ms_set_opt2_slit_right_V: float | None
    ms_set_tof_vpulse_V: float | None
    ms_set_tof_pusher_offset_mV: float | None
    ms_set_tof_vliner_V: float | None
    ms_set_acq_num_pts: int | None
    ms_set_acq_num_trans: float | None
    ms_set_det_vmcpback_V: float | None
    ms_set_det_preamp_offset: float | None
    ms_set_det_dualgain_ratio: float | None
    ms_act_src_corona_vol_V: float | None
    ms_act_src_cap_cur: float | None
    ms_act_src_cham_cur: float | None
    ms_method_parse_status: str
    ms_method_progid: str | None
    ms_method_com_class: str | None
    ms_method_type_id: str | None
    ms_method_default_tune_file: str | None
    ms_method_targeted_list_file: str | None
    ms_method_profile_token: str | None
    ms_method_model_token: str | None
    ms_method_polarity: str | None
    ms_method_source_type: str | None
    ms_method_mass_range: str | None
    ms_method_instrument_mode: str | None
    ms_method_acquisition_rate_hz: float | None
    ms_method_acquisition_time_ms: float | None
    ms_method_cc_gas: str | None
    ms_method_acq_holdoff_delay_ms: float | None
    ms_method_time_segments_json: str | None
    ms_method_postamble_json: str | None
    ms_embedded_xml_count: int | None
    ms_embedded_xml_headers: str | None
    ms_embedded_xml_payload_json: str | None
    ms_bin_candidate_parse_status: str
    ms_bin_gap_lengths: str | None
    ms_bin_tagged_double_count: int | None
    ms_bin_tagged_double_values: str | None
    ms_bin_method_extraction_note: str | None
    ms_bin_method_gas_temp_C: float | None
    ms_bin_method_drying_gas_L_min: float | None
    ms_bin_method_nebulizer_psi: float | None
    ms_bin_method_nebulizer_source: str | None
    ms_bin_method_sheath_gas_temp_C: float | None
    ms_bin_method_sheath_gas_flow_L_min: float | None
    ms_bin_method_vcap_V: float | None
    ms_bin_method_nozzle_voltage_V: float | None
    ms_bin_method_fragmentor_V: float | None
    ms_bin_method_skimmer_V: float | None
    ms_bin_method_oct1_rf_vpp_V: float | None
    ms_tune_positive_json: str | None
    ms_tune_negative_json: str | None
    ms_tune_positive_count: int | None
    ms_tune_negative_count: int | None
    ms_tune_positive_unique_hash_count: int | None
    ms_tune_negative_unique_hash_count: int | None
    ms_tune_positive_selected_hash: str | None
    ms_tune_negative_selected_hash: str | None
    ms_tune_positive_all_hashes: str | None
    ms_tune_negative_all_hashes: str | None
    ms_tune_positive_hash_consistency: str | None
    ms_tune_negative_hash_consistency: str | None


def warn(message: str) -> None:
    print(f"[WARN] {message}", file=sys.stderr)


def parse_xml(path: Path) -> ET.Element:
    return ET.parse(path).getroot()


def text_at(root: ET.Element, path: str, default: str | None = None) -> str | None:
    node = root.find(".//" + path)
    if node is None:
        return default
    text = (node.text or "").strip()
    return text if text else default


def to_float(value: str | None) -> float | None:
    if value is None:
        return None
    try:
        normalized = value.strip().replace(",", "")
        return float(normalized)
    except ValueError:
        return None


def mean_or_none(values: list[float]) -> float | None:
    if not values:
        return None
    return sum(values) / len(values)


def parse_expected_polarity(method_dir_name: str) -> str | None:
    name = method_dir_name.lower()
    if "_pos" in name:
        return "Positive"
    if "_neg" in name:
        return "Negative"
    return None


def normalize_polarity_key(value: str | None) -> str:
    text = (value or "").strip().lower()
    if text.startswith("pos"):
        return "positive"
    if text.startswith("neg"):
        return "negative"
    if text:
        return text
    return "unknown"


def _read_le_uint(buffer: bytes, offset: int, size: int, label: str) -> int:
    chunk = buffer[offset : offset + size]
    if len(chunk) != size:
        raise ValueError(f"{label}: truncated while reading {size} bytes at 0x{offset:x}")
    return int.from_bytes(chunk, "little", signed=False)


def _u16le(buffer: bytes, offset: int, label: str) -> int:
    return _read_le_uint(buffer, offset, 2, label)


def _u32le(buffer: bytes, offset: int, label: str) -> int:
    return _read_le_uint(buffer, offset, 4, label)


def _u64le(buffer: bytes, offset: int, label: str) -> int:
    return _read_le_uint(buffer, offset, 8, label)


def _normalize_stream_id(value: int) -> int | None:
    return None if value == CFBF_NO_STREAM else value


def parse_cfbf_header(stg_bytes: bytes) -> CfbfHeader:
    if len(stg_bytes) < 512:
        raise ValueError("CFBF header is truncated (< 512 bytes)")
    if stg_bytes[:8] != CFBF_SIGNATURE:
        raise ValueError("Not a CFBF/OLE compound file (signature mismatch)")

    byte_order = _u16le(stg_bytes, 0x1C, "CFBF header")
    if byte_order != CFBF_BYTE_ORDER_LE:
        raise ValueError(f"Unsupported CFBF byte order: 0x{byte_order:04x}")

    sector_shift = _u16le(stg_bytes, 0x1E, "CFBF header")
    mini_sector_shift = _u16le(stg_bytes, 0x20, "CFBF header")

    if sector_shift < 9:
        raise ValueError(f"Invalid CFBF sector shift: {sector_shift}")
    if mini_sector_shift < 3:
        raise ValueError(f"Invalid CFBF mini sector shift: {mini_sector_shift}")

    difat_header = [_u32le(stg_bytes, 0x4C + idx * 4, "CFBF header DIFAT") for idx in range(109)]

    return CfbfHeader(
        minor_version=_u16le(stg_bytes, 0x18, "CFBF header"),
        major_version=_u16le(stg_bytes, 0x1A, "CFBF header"),
        byte_order=byte_order,
        sector_shift=sector_shift,
        mini_sector_shift=mini_sector_shift,
        sector_size=(1 << sector_shift),
        mini_sector_size=(1 << mini_sector_shift),
        num_dir_sectors=_u32le(stg_bytes, 0x28, "CFBF header"),
        num_fat_sectors=_u32le(stg_bytes, 0x2C, "CFBF header"),
        first_dir_sector=_u32le(stg_bytes, 0x30, "CFBF header"),
        transaction_signature=_u32le(stg_bytes, 0x34, "CFBF header"),
        mini_stream_cutoff_size=_u32le(stg_bytes, 0x38, "CFBF header"),
        first_mini_fat_sector=_u32le(stg_bytes, 0x3C, "CFBF header"),
        num_mini_fat_sectors=_u32le(stg_bytes, 0x40, "CFBF header"),
        first_difat_sector=_u32le(stg_bytes, 0x44, "CFBF header"),
        num_difat_sectors=_u32le(stg_bytes, 0x48, "CFBF header"),
        difat_header=difat_header,
    )


def _cfbf_sector_offset(header: CfbfHeader, sector_id: int) -> int:
    if sector_id < 0:
        raise ValueError(f"Negative sector id is invalid: {sector_id}")
    return (sector_id + 1) * header.sector_size


def _cfbf_read_sector(stg_bytes: bytes, header: CfbfHeader, sector_id: int) -> bytes:
    offset = _cfbf_sector_offset(header, sector_id)
    end = offset + header.sector_size
    if end > len(stg_bytes):
        raise ValueError(
            f"Sector {sector_id} points outside file boundaries "
            f"(offset={offset}, end={end}, file_size={len(stg_bytes)})"
        )
    return stg_bytes[offset:end]


def _cfbf_unpack_u32_list(buffer: bytes, label: str) -> list[int]:
    if len(buffer) % 4 != 0:
        raise ValueError(f"{label}: expected 4-byte alignment, got {len(buffer)} bytes")
    return [int.from_bytes(buffer[idx : idx + 4], "little", signed=False) for idx in range(0, len(buffer), 4)]


def _cfbf_collect_fat_sector_ids(stg_bytes: bytes, header: CfbfHeader) -> list[int]:
    fat_sector_ids = [sid for sid in header.difat_header if sid != CFBF_FREE_SECTOR]
    entries_per_difat_sector = (header.sector_size // 4) - 1
    difat_sector_id = header.first_difat_sector
    remaining_difat_sectors = header.num_difat_sectors
    seen_difat_sectors: set[int] = set()

    while remaining_difat_sectors > 0:
        if difat_sector_id in (CFBF_END_OF_CHAIN, CFBF_FREE_SECTOR):
            raise ValueError("DIFAT chain ended before reading all DIFAT sectors")
        if difat_sector_id in seen_difat_sectors:
            raise ValueError(f"DIFAT chain contains a loop at sector {difat_sector_id}")
        seen_difat_sectors.add(difat_sector_id)

        difat_sector = _cfbf_read_sector(stg_bytes, header, difat_sector_id)
        difat_entries = _cfbf_unpack_u32_list(difat_sector, "DIFAT sector")
        for fat_sid in difat_entries[:entries_per_difat_sector]:
            if fat_sid != CFBF_FREE_SECTOR:
                fat_sector_ids.append(fat_sid)
        difat_sector_id = difat_entries[entries_per_difat_sector]
        remaining_difat_sectors -= 1

    if len(fat_sector_ids) < header.num_fat_sectors:
        raise ValueError(
            f"DIFAT provided only {len(fat_sector_ids)} FAT sector ids, "
            f"header expects {header.num_fat_sectors}"
        )
    return fat_sector_ids[: header.num_fat_sectors]


def _cfbf_load_fat(stg_bytes: bytes, header: CfbfHeader, fat_sector_ids: list[int]) -> list[int]:
    fat_entries: list[int] = []
    for fat_sector_id in fat_sector_ids:
        sector_bytes = _cfbf_read_sector(stg_bytes, header, fat_sector_id)
        fat_entries.extend(_cfbf_unpack_u32_list(sector_bytes, f"FAT sector {fat_sector_id}"))
    return fat_entries


def _cfbf_follow_chain(
    start_sector_id: int,
    allocation_table: list[int],
    *,
    table_name: str,
    max_chain_length: int | None = None,
) -> list[int]:
    if start_sector_id in (CFBF_END_OF_CHAIN, CFBF_FREE_SECTOR):
        return []
    if start_sector_id in (CFBF_FAT_SECTOR, CFBF_DIFAT_SECTOR):
        raise ValueError(f"{table_name}: invalid chain start marker 0x{start_sector_id:08x}")

    chain: list[int] = []
    seen: set[int] = set()
    current = start_sector_id
    step_limit = max_chain_length if max_chain_length is not None else (len(allocation_table) + 1)

    while current not in (CFBF_END_OF_CHAIN, CFBF_FREE_SECTOR):
        if current in seen:
            raise ValueError(f"{table_name}: chain loop detected at sector {current}")
        if current < 0 or current >= len(allocation_table):
            raise ValueError(f"{table_name}: sector {current} is outside allocation table bounds")
        if current in (CFBF_FAT_SECTOR, CFBF_DIFAT_SECTOR):
            raise ValueError(f"{table_name}: chain jumped to reserved marker 0x{current:08x}")

        seen.add(current)
        chain.append(current)
        if len(chain) > step_limit:
            raise ValueError(f"{table_name}: chain exceeded safety limit ({step_limit})")
        current = allocation_table[current]

    return chain


def _cfbf_read_chained_sectors(
    stg_bytes: bytes,
    header: CfbfHeader,
    sector_chain: list[int],
    expected_size: int,
) -> bytes:
    if expected_size < 0:
        raise ValueError(f"Expected size must be non-negative, got {expected_size}")
    if not sector_chain or expected_size == 0:
        return b""

    data = bytearray()
    for sector_id in sector_chain:
        data.extend(_cfbf_read_sector(stg_bytes, header, sector_id))
    return bytes(data[:expected_size])


def _decode_cfbf_entry_name(entry_bytes: bytes, name_length: int) -> str:
    raw_name = entry_bytes[:64]
    if 2 <= name_length <= 64 and name_length % 2 == 0:
        name_payload = raw_name[: max(0, name_length - 2)]
    else:
        name_payload = raw_name.rstrip(b"\x00")
    return name_payload.decode("utf-16le", errors="ignore").rstrip("\x00")


def parse_cfbf_directory_entries(
    stg_bytes: bytes, header: CfbfHeader, fat: list[int]
) -> tuple[list[CfbfDirectoryEntry], list[int]]:
    directory_chain = _cfbf_follow_chain(header.first_dir_sector, fat, table_name="FAT(directory)")
    directory_bytes = _cfbf_read_chained_sectors(
        stg_bytes,
        header,
        directory_chain,
        expected_size=len(directory_chain) * header.sector_size,
    )

    entries: list[CfbfDirectoryEntry] = []
    for offset in range(0, len(directory_bytes), CFBF_DIRECTORY_ENTRY_SIZE):
        entry_bytes = directory_bytes[offset : offset + CFBF_DIRECTORY_ENTRY_SIZE]
        if len(entry_bytes) != CFBF_DIRECTORY_ENTRY_SIZE:
            break

        entry_id = offset // CFBF_DIRECTORY_ENTRY_SIZE
        name_length = _u16le(entry_bytes, 0x40, "Directory entry")
        stream_size = _u64le(entry_bytes, 0x78, "Directory entry")
        if header.major_version <= 3:
            stream_size &= 0xFFFFFFFF

        entries.append(
            CfbfDirectoryEntry(
                entry_id=entry_id,
                name=_decode_cfbf_entry_name(entry_bytes, name_length),
                object_type=entry_bytes[0x42],
                color_flag=entry_bytes[0x43],
                left_sibling_id=_normalize_stream_id(_u32le(entry_bytes, 0x44, "Directory entry")),
                right_sibling_id=_normalize_stream_id(_u32le(entry_bytes, 0x48, "Directory entry")),
                child_id=_normalize_stream_id(_u32le(entry_bytes, 0x4C, "Directory entry")),
                clsid_hex=entry_bytes[0x50:0x60].hex(),
                state_bits=_u32le(entry_bytes, 0x60, "Directory entry"),
                creation_time=_u64le(entry_bytes, 0x64, "Directory entry"),
                modified_time=_u64le(entry_bytes, 0x6C, "Directory entry"),
                starting_sector=_u32le(entry_bytes, 0x74, "Directory entry"),
                stream_size=stream_size,
            )
        )

    return entries, directory_chain


def _find_root_entry(directory_entries: list[CfbfDirectoryEntry]) -> CfbfDirectoryEntry | None:
    for entry in directory_entries:
        if entry.object_type == 5:
            return entry
    return None


def _cfbf_load_mini_fat(
    stg_bytes: bytes,
    header: CfbfHeader,
    fat: list[int],
) -> list[int]:
    if (
        header.num_mini_fat_sectors == 0
        or header.first_mini_fat_sector in (CFBF_END_OF_CHAIN, CFBF_FREE_SECTOR)
    ):
        return []

    mini_fat_chain = _cfbf_follow_chain(
        header.first_mini_fat_sector,
        fat,
        table_name="FAT(miniFAT)",
        max_chain_length=header.num_mini_fat_sectors + 8,
    )
    if len(mini_fat_chain) < header.num_mini_fat_sectors:
        raise ValueError(
            f"MiniFAT chain too short: expected {header.num_mini_fat_sectors}, got {len(mini_fat_chain)}"
        )

    mini_fat_chain = mini_fat_chain[: header.num_mini_fat_sectors]
    mini_fat_bytes = _cfbf_read_chained_sectors(
        stg_bytes,
        header,
        mini_fat_chain,
        expected_size=len(mini_fat_chain) * header.sector_size,
    )
    return _cfbf_unpack_u32_list(mini_fat_bytes, "MiniFAT")


def _cfbf_load_mini_stream(
    stg_bytes: bytes,
    header: CfbfHeader,
    fat: list[int],
    directory_entries: list[CfbfDirectoryEntry],
) -> bytes:
    root_entry = _find_root_entry(directory_entries)
    if root_entry is None:
        return b""
    if root_entry.stream_size == 0 or root_entry.starting_sector in (CFBF_END_OF_CHAIN, CFBF_FREE_SECTOR):
        return b""

    expected_root_sectors = (root_entry.stream_size + header.sector_size - 1) // header.sector_size
    root_chain = _cfbf_follow_chain(
        root_entry.starting_sector,
        fat,
        table_name="FAT(root mini stream)",
        max_chain_length=expected_root_sectors + 8,
    )
    if expected_root_sectors and len(root_chain) < expected_root_sectors:
        raise ValueError(
            f"Root mini stream chain too short: expected {expected_root_sectors}, got {len(root_chain)}"
        )
    return _cfbf_read_chained_sectors(stg_bytes, header, root_chain, expected_size=root_entry.stream_size)


def _cfbf_get_stream_chain(
    entry: CfbfDirectoryEntry,
    header: CfbfHeader,
    fat: list[int],
    mini_fat: list[int],
    mini_stream: bytes,
) -> tuple[str, list[int]]:
    if entry.stream_size == 0 or entry.starting_sector in (CFBF_END_OF_CHAIN, CFBF_FREE_SECTOR):
        return ("empty", [])

    use_mini_stream = (
        entry.stream_size < header.mini_stream_cutoff_size and len(mini_fat) > 0 and len(mini_stream) > 0
    )
    if use_mini_stream:
        expected_mini_sectors = (entry.stream_size + header.mini_sector_size - 1) // header.mini_sector_size
        mini_chain = _cfbf_follow_chain(
            entry.starting_sector,
            mini_fat,
            table_name=f"MiniFAT(stream:{entry.name or entry.entry_id})",
            max_chain_length=expected_mini_sectors + 8,
        )
        if expected_mini_sectors and len(mini_chain) < expected_mini_sectors:
            raise ValueError(
                f"Mini stream chain too short for entry {entry.entry_id}: "
                f"expected {expected_mini_sectors}, got {len(mini_chain)}"
            )
        return ("mini", mini_chain)

    expected_sectors = (entry.stream_size + header.sector_size - 1) // header.sector_size
    fat_chain = _cfbf_follow_chain(
        entry.starting_sector,
        fat,
        table_name=f"FAT(stream:{entry.name or entry.entry_id})",
        max_chain_length=expected_sectors + 8,
    )
    if expected_sectors and len(fat_chain) < expected_sectors:
        raise ValueError(
            f"FAT stream chain too short for entry {entry.entry_id}: "
            f"expected {expected_sectors}, got {len(fat_chain)}"
        )
    return ("fat", fat_chain)


def _cfbf_read_stream_bytes(
    stg_bytes: bytes,
    header: CfbfHeader,
    entry: CfbfDirectoryEntry,
    fat: list[int],
    mini_fat: list[int],
    mini_stream: bytes,
) -> tuple[bytes, str, list[int]]:
    chain_type, sector_chain = _cfbf_get_stream_chain(entry, header, fat, mini_fat, mini_stream)
    if chain_type == "empty":
        return (b"", chain_type, sector_chain)

    if chain_type == "mini":
        stream_data = bytearray()
        for mini_sector_id in sector_chain:
            begin = mini_sector_id * header.mini_sector_size
            end = begin + header.mini_sector_size
            if end > len(mini_stream):
                raise ValueError(
                    f"Mini stream sector {mini_sector_id} for entry {entry.entry_id} "
                    f"is outside root mini stream bounds"
                )
            stream_data.extend(mini_stream[begin:end])
        return (bytes(stream_data[: entry.stream_size]), chain_type, sector_chain)

    return (
        _cfbf_read_chained_sectors(stg_bytes, header, sector_chain, expected_size=entry.stream_size),
        chain_type,
        sector_chain,
    )


def parse_cfbf_structure(stg_bytes: bytes) -> CfbfDecodeResult:
    header = parse_cfbf_header(stg_bytes)
    fat_sector_ids = _cfbf_collect_fat_sector_ids(stg_bytes, header)
    fat = _cfbf_load_fat(stg_bytes, header, fat_sector_ids)
    directory_entries, directory_sector_chain = parse_cfbf_directory_entries(stg_bytes, header, fat)
    mini_fat = _cfbf_load_mini_fat(stg_bytes, header, fat)
    mini_stream = _cfbf_load_mini_stream(stg_bytes, header, fat, directory_entries)

    streams: list[CfbfStreamRecord] = []
    for entry in directory_entries:
        if entry.object_type != 2:
            continue
        data, chain_type, sector_chain = _cfbf_read_stream_bytes(
            stg_bytes, header, entry, fat, mini_fat, mini_stream
        )
        streams.append(
            CfbfStreamRecord(
                entry_id=entry.entry_id,
                name=entry.name,
                chain_type=chain_type,
                sector_chain=sector_chain,
                data=data,
            )
        )

    return CfbfDecodeResult(
        header=header,
        fat_sector_ids=fat_sector_ids,
        fat=fat,
        mini_fat=mini_fat,
        directory_sector_chain=directory_sector_chain,
        directory_entries=directory_entries,
        mini_stream=mini_stream,
        streams=streams,
    )


def cfbf_get_stream_by_name(parsed: CfbfDecodeResult, stream_name: str) -> CfbfStreamRecord | None:
    needle = stream_name.strip().lower()
    for stream in parsed.streams:
        if stream.name.strip().lower() == needle:
            return stream
    return None


def parse_msmethod_fixed_header(stream_bytes: bytes) -> MsMethodFixedHeader:
    if len(stream_bytes) < MSMETHOD_FIXED_HEADER_SIZE:
        raise ValueError(
            f"MSMETHOD stream is too short for fixed header: "
            f"required={MSMETHOD_FIXED_HEADER_SIZE}, actual={len(stream_bytes)}"
        )

    fields = [
        _u32le(stream_bytes, idx * 4, "MSMETHOD fixed header")
        for idx in range(MSMETHOD_FIXED_HEADER_SIZE // 4)
    ]

    header = MsMethodFixedHeader(
        unk0=fields[0],
        unk1=fields[1],
        n_time_segments=fields[2],
        fixed_value_300=fields[3],
        unk2=fields[4],
        unk3=fields[5],
        n_time_segments_2=fields[6],
        unk4=fields[7],
        unk5=fields[8],
        unk6=fields[9],
        unk7=fields[10],
        unk8=fields[11],
        unk9=fields[12],
        unk10=fields[13],
    )

    if header.n_time_segments != header.n_time_segments_2:
        raise AssertionError(
            "MSMETHOD header mismatch: n_time_segments and n_time_segments_2 differ "
            f"({header.n_time_segments} vs {header.n_time_segments_2})"
        )
    if header.fixed_value_300 != MSMETHOD_EXPECTED_FIXED_VALUE_300:
        raise AssertionError(
            "MSMETHOD header mismatch: fixed_value_300 differs from expected value "
            f"({header.fixed_value_300} vs {MSMETHOD_EXPECTED_FIXED_VALUE_300})"
        )
    return header


def _decode_msmethod_tagged_value(
    stream_bytes: bytes,
    offset: int,
    payload_end: int,
    segment_index: int,
) -> tuple[MsMethodTaggedValue, int]:
    # MSMETHOD tagged value notes (current reverse-engineering status):
    # - type 0x0000: padding-like marker with no payload bytes.
    # - type 0x0002: 2-byte unsigned integer payload.
    # - type 0x0003: 4-byte unsigned integer payload.
    # - type 0x0005: 8-byte IEEE754 little-endian double payload.
    # - type 0x0008: UTF-16LE string payload with u16 code-unit length prefix.
    # - type 0x2002: uint16[] payload; wire format is [u32(rank), rank*u32(dims), data...].
    # - type 0x2003: uint32[] payload; wire format is [u32(rank), rank*u32(dims), data...].
    # - type 0x2005: double[] payload; wire format is [u32(rank), rank*u32(dims), data...].
    # - type 0x000B: 2-byte VARIANT_BOOL payload (0x0000=False, 0xFFFF=True).
    # Unknown codes are intentionally treated as hard errors to avoid silent drift.
    context_label = f"MSMETHOD time segment {segment_index} payload"
    if offset + 2 > payload_end:
        raise ValueError(
            f"{context_label}: truncated before type code at offset {offset} (payload_end={payload_end})"
        )

    type_code = _u16le(stream_bytes, offset, context_label)
    cursor = offset + 2

    if type_code == 0x0000:
        # Padding-like token with no payload bytes.
        value: float | int | str | bool | MsMethodArrayValue | None = None
    elif type_code == 0x0002:
        if cursor + 2 > payload_end:
            raise ValueError(
                f"{context_label}: type 0x0002 is truncated at offset {offset} "
                f"(payload_end={payload_end})"
            )
        value = _u16le(stream_bytes, cursor, context_label)
        cursor += 2
    elif type_code == 0x0003:
        if cursor + 4 > payload_end:
            raise ValueError(
                f"{context_label}: type 0x0003(uint32) is truncated at offset {offset} "
                f"(payload_end={payload_end})"
            )
        value = _u32le(stream_bytes, cursor, context_label)
        cursor += 4
    elif type_code == 0x0005:
        if cursor + 8 > payload_end:
            raise ValueError(
                f"{context_label}: type 0x0005(double) is truncated at offset {offset} "
                f"(payload_end={payload_end})"
            )
        value = struct.unpack("<d", stream_bytes[cursor : cursor + 8])[0]
        cursor += 8
    elif type_code == 0x0008:
        # UTF-16LE string prefixed by u16 code-unit count.
        if cursor + 2 > payload_end:
            raise ValueError(
                f"{context_label}: type 0x0008(string) is truncated before length at offset {offset}"
            )
        code_units = _u16le(stream_bytes, cursor, context_label)
        cursor += 2
        byte_len = code_units * 2
        if cursor + byte_len > payload_end:
            raise ValueError(
                f"{context_label}: type 0x0008(string) exceeds payload bounds at offset {offset} "
                f"(code_units={code_units}, payload_end={payload_end})"
            )
        raw = stream_bytes[cursor : cursor + byte_len]
        value = raw.decode("utf-16le", errors="ignore").rstrip("\x00")
        cursor += byte_len
    elif type_code in (0x2002, 0x2003, 0x2005):
        if cursor + 8 > payload_end:
            raise ValueError(
                f"{context_label}: type 0x{type_code:04x}(array) is truncated before array header "
                f"at offset {offset} (payload_end={payload_end})"
            )
        rank = _u32le(stream_bytes, cursor, context_label)
        cursor += 4
        if rank <= 0:
            raise AssertionError(
                f"{context_label}: type 0x{type_code:04x} rank must be > 0, "
                f"got {rank} at offset {offset}"
            )
        if rank > 8:
            raise AssertionError(
                f"{context_label}: type 0x{type_code:04x} rank is unexpectedly large ({rank}) "
                f"at offset {offset}"
            )

        dims_byte_len = rank * 4
        if cursor + dims_byte_len > payload_end:
            raise ValueError(
                f"{context_label}: type 0x{type_code:04x} dims are truncated at offset {offset} "
                f"(rank={rank}, payload_end={payload_end})"
            )
        dims: list[int] = []
        n_items = 1
        for _ in range(rank):
            dim = _u32le(stream_bytes, cursor, context_label)
            cursor += 4
            dims.append(dim)
            n_items *= dim
        dims_tuple = tuple(dims)

        if type_code in (0x2002, 0x2003):
            element_size = 2 if type_code == 0x2002 else 4
            total_len = n_items * element_size
            type_name = "uint16[]" if type_code == 0x2002 else "uint32[]"
            if cursor + total_len > payload_end:
                raise ValueError(
                    f"{context_label}: type 0x{type_code:04x}({type_name}) exceeds payload bounds "
                    f"at offset {offset} (count={n_items}, payload_end={payload_end})"
                )
            values: list[int] = []
            for _ in range(n_items):
                if type_code == 0x2002:
                    values.append(_u16le(stream_bytes, cursor, context_label))
                else:
                    values.append(_u32le(stream_bytes, cursor, context_label))
                cursor += element_size
            value = MsMethodArrayValue(rank=rank, dims=dims_tuple, values=tuple(values))
        else:
            total_len = n_items * 8
            if cursor + total_len > payload_end:
                raise ValueError(
                    f"{context_label}: type 0x2005(double[]) exceeds payload bounds at offset {offset} "
                    f"(count={n_items}, payload_end={payload_end})"
                )
            values_f: list[float] = []
            for _ in range(n_items):
                values_f.append(struct.unpack("<d", stream_bytes[cursor : cursor + 8])[0])
                cursor += 8
            value = MsMethodArrayValue(rank=rank, dims=dims_tuple, values=tuple(values_f))
    elif type_code == 0x000B:
        if cursor + 2 > payload_end:
            raise ValueError(
                f"{context_label}: type 0x000B(bool) is truncated at offset {offset} "
                f"(payload_end={payload_end})"
            )
        raw_bool = _u16le(stream_bytes, cursor, context_label)
        if raw_bool == 0x0000:
            value = False
        elif raw_bool == 0xFFFF:
            value = True
        else:
            raise AssertionError(
                f"{context_label}: type 0x000B has unexpected raw value 0x{raw_bool:04x} "
                f"at offset {offset} (expected 0x0000 or 0xFFFF)"
            )
        cursor += 2
    else:
        raise AssertionError(
            f"{context_label}: unknown type code 0x{type_code:04x} at offset {offset}"
        )

    return (
        MsMethodTaggedValue(
            offset=offset,
            type_code=type_code,
            value=value,
        ),
        cursor,
    )


def _parse_msmethod_time_segment_payload(
    stream_bytes: bytes,
    payload_start: int,
    segment_index: int,
    n_fields: int,
    n_experimental_segments: int,
) -> tuple[
    tuple[MsMethodTaggedValue, ...], tuple[tuple[MsMethodTaggedValue, ...], ...], int
]:
    if n_fields <= 0:
        raise AssertionError(
            f"MSMETHOD time segment {segment_index}: n_fields must be > 0, got {n_fields}"
        )
    if n_experimental_segments <= 0:
        raise AssertionError(
            f"MSMETHOD time segment {segment_index}: n_experimental_segments must be > 0, "
            f"got {n_experimental_segments}"
        )
    if payload_start < 0 or payload_start > len(stream_bytes):
        raise ValueError(
            f"MSMETHOD time segment {segment_index}: payload start is out of range "
            f"(start={payload_start}, stream_size={len(stream_bytes)})"
        )

    tagged_values: list[MsMethodTaggedValue] = []
    field_major_values: list[tuple[MsMethodTaggedValue, ...]] = []
    cursor = payload_start
    for field_index in range(n_fields):
        field_values: list[MsMethodTaggedValue] = []
        for expt_index in range(n_experimental_segments):
            tagged_value, next_cursor = _decode_msmethod_tagged_value(
                stream_bytes=stream_bytes,
                offset=cursor,
                payload_end=len(stream_bytes),
                segment_index=segment_index,
            )
            if next_cursor <= cursor:
                raise AssertionError(
                    f"MSMETHOD time segment {segment_index}: tagged-value decode did not advance "
                    f"(field={field_index}, expt={expt_index}, offset={cursor}, next={next_cursor})"
                )
            tagged_values.append(tagged_value)
            field_values.append(tagged_value)
            cursor = next_cursor
        field_major_values.append(tuple(field_values))

    return tuple(tagged_values), tuple(field_major_values), cursor


def msmethod_unknown_field_name(field_index: int) -> str:
    return MSMETHOD_UNKNOWN_FIELD_NAME_OVERRIDES.get(field_index, f"unknown_{field_index}")


def msmethod_extension_unknown_field_name(field_index: int) -> str:
    return f"unknown_{field_index}"


def msmethod_trailing_unknown_field_name(field_index: int) -> str:
    return MSMETHOD_TRAILING_UNKNOWN_FIELD_NAME_OVERRIDES.get(
        field_index, f"unknown_{field_index}"
    )


def msmethod_trailing_wrapped_column_name(slot_index: int) -> str:
    if slot_index == 0:
        return "ref_mass (pos)"
    if slot_index == 1:
        return "ref_mass (neg)"
    if slot_index == 2:
        return "type"
    generic_col_index = slot_index - MSMETHOD_TRAILING_SPECIAL_WRAP_COLUMNS + 1
    if generic_col_index == 1:
        return "label"
    if generic_col_index == 3:
        return "offset"
    if generic_col_index == 4:
        return "y_range"
    if generic_col_index == 5:
        return "unknown_1"
    if generic_col_index == 6:
        return "unknown_2"
    if generic_col_index == 8:
        return "expt_type"
    if generic_col_index == 9:
        return "polarity"
    return f"col_{generic_col_index}"


def format_msmethod_trailing_wrapped_cell(
    value: Any,
    *,
    column_name: str | None = None,
) -> str:
    code = _normalize_polarity_code(value)
    if column_name == "expt_type" and code in MSMETHOD_TRAILING_EXPT_TYPE_BY_CODE:
        return MSMETHOD_TRAILING_EXPT_TYPE_BY_CODE[code]
    if column_name == "polarity" and code in MSMETHOD_TRAILING_POLARITY_BY_CODE:
        return MSMETHOD_TRAILING_POLARITY_BY_CODE[code]
    return str(value or "")


def msmethod_trailing_type_label_from_generic_row(generic_row_values: list[Any]) -> str:
    if len(generic_row_values) <= 1:
        return ""
    code = _normalize_polarity_code(generic_row_values[1])
    if code is None:
        return ""
    return MSMETHOD_TRAILING_TYPE_BY_CODE.get(code, "")


def msmethod_ref_mass_indexes_for_n_fields(n_trailing_fields: int) -> tuple[int | None, int | None]:
    n = max(0, int(n_trailing_fields))
    if n <= 0:
        return None, None
    if n == 1:
        return None, 0
    return n - 2, n - 1


def build_msmethod_trailing_slot_layout(n_trailing_fields: int) -> list[list[int | None]]:
    n_trailing_fields = max(0, int(n_trailing_fields))
    ref_pos_index, ref_neg_index = msmethod_ref_mass_indexes_for_n_fields(n_trailing_fields)
    ref_mass_indexes = {idx for idx in (ref_pos_index, ref_neg_index) if idx is not None}
    generic_field_indexes = [
        field_idx
        for field_idx in range(n_trailing_fields)
        if field_idx not in ref_mass_indexes
    ]
    n_rows = max(
        1,
        (
            len(generic_field_indexes) + MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS - 1
        )
        // MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS,
    )
    layout: list[list[int | None]] = []
    for row_index in range(n_rows):
        start = row_index * MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS
        stop = start + MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS
        generic_slice = generic_field_indexes[start:stop]
        row_layout: list[int | None] = list(generic_slice)
        while len(row_layout) < MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS:
            row_layout.append(None)

        if row_index == 0:
            row_layout = [ref_pos_index, ref_neg_index, *row_layout]
        else:
            row_layout = [None, None, *row_layout]
        layout.append(row_layout)
    return layout


def _match_msmethod_trailing_compaction_rule(
    values: list[Any], start_index: int
) -> tuple[int, int, int] | None:
    if start_index < 0:
        return None
    for signature, insert_after_n_values, blank_cell_count in MSMETHOD_TRAILING_COMPACTION_RULES:
        if start_index + len(signature) > len(values):
            continue
        matched = True
        for offset, expected in enumerate(signature):
            actual = _normalize_polarity_code(values[start_index + offset])
            if actual != expected:
                matched = False
                break
        if matched:
            return len(signature), insert_after_n_values, blank_cell_count
    return None


def _build_msmethod_trailing_generic_rows(values: list[Any]) -> list[list[Any]]:
    rows: list[list[Any]] = []
    cursor = 0

    while cursor < len(values):
        row: list[Any] = []
        # Per requested display rule, each row's first numeric value is a row index and skipped.
        cursor += 1

        pending_blank_insertions: list[list[int]] = []

        while len(row) < MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS and cursor < len(values):
            match = _match_msmethod_trailing_compaction_rule(values, cursor)
            if match is not None:
                compact_len, insert_after_n_values, blank_cell_count = match
                # Drop the 4-number signature itself.
                cursor += compact_len
                pending_blank_insertions.append([insert_after_n_values, blank_cell_count])
                continue

            row.append(values[cursor])
            cursor += 1

            if pending_blank_insertions:
                for pending in pending_blank_insertions:
                    pending[0] -= 1
                ready = [pending for pending in pending_blank_insertions if pending[0] <= 0]
                pending_blank_insertions = [
                    pending for pending in pending_blank_insertions if pending[0] > 0
                ]
                for _, blank_cell_count in ready:
                    for _ in range(blank_cell_count):
                        if len(row) >= MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS:
                            break
                        row.append("")

        rows.append(row)

    if not rows:
        rows.append([])
    return rows


def _build_unknown_fields(
    field_major_values: tuple[tuple[MsMethodTaggedValue, ...], ...],
) -> tuple[MsMethodUnknownField, ...]:
    return tuple(
        MsMethodUnknownField(
            index=field_index,
            name=msmethod_unknown_field_name(field_index),
            values=field_values,
        )
        for field_index, field_values in enumerate(field_major_values)
    )


def _build_unknown_fields_from_tagged_values(
    tagged_values: tuple[MsMethodTaggedValue, ...],
) -> tuple[MsMethodUnknownField, ...]:
    return tuple(
        MsMethodUnknownField(
            index=field_index,
            name=msmethod_unknown_field_name(field_index),
            values=(tagged_value,),
        )
        for field_index, tagged_value in enumerate(tagged_values)
    )


def parse_msmethod_time_segments(
    stream_bytes: bytes, fixed_header: MsMethodFixedHeader
) -> list[MsMethodTimeSegment]:
    if fixed_header.n_time_segments <= 0:
        raise AssertionError(
            f"MSMETHOD header invalid: n_time_segments must be > 0, got {fixed_header.n_time_segments}"
        )
    if len(stream_bytes) < MSMETHOD_FIXED_HEADER_SIZE:
        raise ValueError(
            f"MSMETHOD stream is too short for time-segment iteration: "
            f"required>={MSMETHOD_FIXED_HEADER_SIZE}, actual={len(stream_bytes)}"
        )

    segments: list[MsMethodTimeSegment] = []
    cursor = MSMETHOD_FIXED_HEADER_SIZE

    # Primary block: per-time-segment payload with fixed 300 fields and
    # n_experimental_segments-aligned values.
    for segment_index in range(fixed_header.n_time_segments):
        if cursor + MSMETHOD_TIME_SEGMENT_HEADER_SIZE > len(stream_bytes):
            raise ValueError(
                "MSMETHOD primary time segment header is truncated: "
                f"segment={segment_index}, offset={cursor}"
            )

        label = f"MSMETHOD primary time segment {segment_index} header"
        n_fields = _u32le(stream_bytes, cursor, label)
        if n_fields != fixed_header.fixed_value_300:
            raise AssertionError(
                f"{label}: n_fields differs from fixed_value_300 "
                f"({n_fields} vs {fixed_header.fixed_value_300})"
            )

        n_experimental_segments = _u32le(stream_bytes, cursor + 4, label)
        start_time_raw_u64 = _u64le(stream_bytes, cursor + 8, label)
        start_time = struct.unpack(
            "<d",
            stream_bytes[cursor + 8 : cursor + MSMETHOD_TIME_SEGMENT_HEADER_SIZE],
        )[0]

        payload_start_offset = cursor + MSMETHOD_TIME_SEGMENT_HEADER_SIZE
        tagged_values, field_major_values, payload_end_offset = _parse_msmethod_time_segment_payload(
            stream_bytes=stream_bytes,
            payload_start=payload_start_offset,
            segment_index=segment_index,
            n_fields=n_fields,
            n_experimental_segments=n_experimental_segments,
        )
        primary_unknown_fields = _build_unknown_fields(field_major_values)

        segments.append(
            MsMethodTimeSegment(
                index=segment_index,
                header_offset=cursor,
                n_fields=n_fields,
                n_experimental_segments=n_experimental_segments,
                start_time_raw_u64=start_time_raw_u64,
                start_time=start_time,
                payload_start_offset=payload_start_offset,
                payload_end_offset=payload_end_offset,
                tagged_values=tagged_values,
                field_major_values=field_major_values,
                trailing_header_offset=None,
                trailing_n_fields=0,
                trailing_n_experimental_segments=0,
                trailing_payload_start_offset=None,
                trailing_payload_end_offset=None,
                trailing_tagged_values=(),
                trailing_field_major_values=(),
                primary_unknown_fields=primary_unknown_fields,
                extension_unknown_fields=(),
            )
        )
        cursor = payload_end_offset

    # Optional trailing block: per-time-segment metadata payload.
    # In current files this appears after all primary segments and uses n_experimental_segments=1.
    if segments and cursor + MSMETHOD_TIME_SEGMENT_HEADER_SIZE <= len(stream_bytes):
        probe_n_experimental_segments = _u32le(
            stream_bytes, cursor + 4, "MSMETHOD trailing probe header"
        )
        probe_start_time_raw_u64 = _u64le(
            stream_bytes, cursor + 8, "MSMETHOD trailing probe header"
        )
        if (
            probe_n_experimental_segments == 1
            and probe_start_time_raw_u64 == segments[0].start_time_raw_u64
        ):
            for segment_index in range(fixed_header.n_time_segments):
                if cursor + MSMETHOD_TIME_SEGMENT_HEADER_SIZE > len(stream_bytes):
                    raise ValueError(
                        "MSMETHOD trailing time segment header is truncated: "
                        f"segment={segment_index}, offset={cursor}"
                    )

                label = f"MSMETHOD trailing time segment {segment_index} header"
                trailing_n_fields = _u32le(stream_bytes, cursor, label)
                if trailing_n_fields <= 0:
                    raise AssertionError(
                        f"{label}: n_fields must be > 0, got {trailing_n_fields}"
                    )

                trailing_n_experimental_segments = _u32le(stream_bytes, cursor + 4, label)
                if trailing_n_experimental_segments != 1:
                    raise AssertionError(
                        f"{label}: n_experimental_segments must be 1 in trailing block, "
                        f"got {trailing_n_experimental_segments}"
                    )

                trailing_start_time_raw_u64 = _u64le(stream_bytes, cursor + 8, label)
                expected_start_time_raw_u64 = segments[segment_index].start_time_raw_u64
                if trailing_start_time_raw_u64 != expected_start_time_raw_u64:
                    raise AssertionError(
                        f"{label}: start_time_raw_u64 mismatch with primary block "
                        f"({trailing_start_time_raw_u64} vs {expected_start_time_raw_u64})"
                    )

                trailing_payload_start_offset = cursor + MSMETHOD_TIME_SEGMENT_HEADER_SIZE
                trailing_tagged_values, trailing_field_major_values, trailing_payload_end_offset = (
                    _parse_msmethod_time_segment_payload(
                        stream_bytes=stream_bytes,
                        payload_start=trailing_payload_start_offset,
                        segment_index=segment_index,
                        n_fields=trailing_n_fields,
                        n_experimental_segments=trailing_n_experimental_segments,
                    )
                )
                extension_unknown_fields = _build_unknown_fields(trailing_field_major_values)

                primary_segment = segments[segment_index]
                segments[segment_index] = MsMethodTimeSegment(
                    index=primary_segment.index,
                    header_offset=primary_segment.header_offset,
                    n_fields=primary_segment.n_fields,
                    n_experimental_segments=primary_segment.n_experimental_segments,
                    start_time_raw_u64=primary_segment.start_time_raw_u64,
                    start_time=primary_segment.start_time,
                    payload_start_offset=primary_segment.payload_start_offset,
                    payload_end_offset=primary_segment.payload_end_offset,
                    tagged_values=primary_segment.tagged_values,
                    field_major_values=primary_segment.field_major_values,
                    trailing_header_offset=cursor,
                    trailing_n_fields=trailing_n_fields,
                    trailing_n_experimental_segments=trailing_n_experimental_segments,
                    trailing_payload_start_offset=trailing_payload_start_offset,
                    trailing_payload_end_offset=trailing_payload_end_offset,
                    trailing_tagged_values=trailing_tagged_values,
                    trailing_field_major_values=trailing_field_major_values,
                    primary_unknown_fields=primary_segment.primary_unknown_fields,
                    extension_unknown_fields=extension_unknown_fields,
                )

                cursor = trailing_payload_end_offset

    return segments


def parse_msmethod_parse_result(
    stream_bytes: bytes, fixed_header: MsMethodFixedHeader | None = None
) -> MsMethodParseResult:
    if fixed_header is None:
        fixed_header = parse_msmethod_fixed_header(stream_bytes)

    time_segments = parse_msmethod_time_segments(stream_bytes, fixed_header)
    if time_segments:
        postamble_start_offset = (
            time_segments[-1].trailing_payload_end_offset or time_segments[-1].payload_end_offset
        )
    else:
        postamble_start_offset = MSMETHOD_FIXED_HEADER_SIZE

    if postamble_start_offset < 0 or postamble_start_offset > len(stream_bytes):
        raise ValueError(
            "MSMETHOD postamble start offset is out of range "
            f"(offset={postamble_start_offset}, size={len(stream_bytes)})"
        )

    postamble_cursor = postamble_start_offset
    postamble_tagged_values: list[MsMethodTaggedValue] = []
    while postamble_cursor < len(stream_bytes):
        tagged_value, next_cursor = _decode_msmethod_tagged_value(
            stream_bytes=stream_bytes,
            offset=postamble_cursor,
            payload_end=len(stream_bytes),
            segment_index=-1,
        )
        if next_cursor <= postamble_cursor:
            raise AssertionError(
                "MSMETHOD postamble decode did not advance "
                f"(offset={postamble_cursor}, next={next_cursor})"
            )
        postamble_tagged_values.append(tagged_value)
        postamble_cursor = next_cursor

    postamble_tagged_values_tuple = tuple(postamble_tagged_values)
    postamble = MsMethodPostamble(
        start_offset=postamble_start_offset,
        end_offset=postamble_cursor,
        raw_bytes=stream_bytes[postamble_start_offset:postamble_cursor],
        tagged_values=postamble_tagged_values_tuple,
        unknown_fields=_build_unknown_fields_from_tagged_values(postamble_tagged_values_tuple),
    )

    return MsMethodParseResult(
        fixed_header=fixed_header,
        time_segments=tuple(time_segments),
        postamble=postamble,
    )


def _read_ansi_length_prefixed_string(
    buffer: bytes, offset: int, label: str
) -> tuple[str | None, int]:
    if offset + 4 > len(buffer):
        raise ValueError(f"{label}: truncated before string length")
    byte_len = _u32le(buffer, offset, label)
    offset += 4
    if byte_len == 0:
        return None, offset
    end = offset + byte_len
    if end > len(buffer):
        raise ValueError(
            f"{label}: length-prefixed string extends beyond stream "
            f"(offset={offset}, len={byte_len}, size={len(buffer)})"
        )
    raw = buffer[offset:end]
    text = raw.decode("latin-1", errors="ignore").rstrip("\x00").strip()
    return (text if text else None), end


def _extract_ansi_length_prefixed_strings(buffer: bytes, min_offset: int = 0) -> list[str]:
    candidates: list[str] = []
    seen: set[str] = set()
    for offset in range(min_offset, max(min_offset, len(buffer) - 4)):
        byte_len = int.from_bytes(buffer[offset : offset + 4], "little", signed=False)
        if byte_len < 4 or byte_len > 256:
            continue
        end = offset + 4 + byte_len
        if end > len(buffer):
            continue
        raw = buffer[offset + 4 : end]
        if not raw or raw[-1] != 0:
            continue
        if any((byte < 0x20 or byte >= 0x7F) and byte != 0 for byte in raw):
            continue
        text = raw.decode("latin-1", errors="ignore").rstrip("\x00").strip()
        if not text or text in seen:
            continue
        seen.add(text)
        candidates.append(text)
    return candidates


def parse_compobj_stream(stream_bytes: bytes) -> tuple[str | None, str | None, str]:
    if len(stream_bytes) < 0x20:
        return None, None, "compobj_too_short"

    try:
        offset = 0x1C
        com_class, offset = _read_ansi_length_prefixed_string(stream_bytes, offset, "CompObj user type")

        if offset + 4 > len(stream_bytes):
            raise ValueError("CompObj stream truncated before clipboard format marker")
        clipboard_marker = _u32le(stream_bytes, offset, "CompObj clipboard marker")
        offset += 4

        if clipboard_marker == 0xFFFFFFFF:
            # Standard clipboard format id follows.
            _ = _u32le(stream_bytes, offset, "CompObj clipboard format id")
            offset += 4
        elif clipboard_marker == 0xFFFFFFFE:
            # Custom clipboard format name follows as length-prefixed ANSI.
            _, offset = _read_ansi_length_prefixed_string(
                stream_bytes, offset, "CompObj custom clipboard format name"
            )

        progid, _ = _read_ansi_length_prefixed_string(stream_bytes, offset, "CompObj ProgID")

        if com_class or progid:
            return com_class, progid, "parsed_compobj"
        return None, None, "compobj_parsed_no_strings"
    except ValueError:
        fallback_strings = _extract_ansi_length_prefixed_strings(stream_bytes, min_offset=0x10)
        if len(fallback_strings) >= 2:
            return fallback_strings[0], fallback_strings[1], "parsed_compobj_fallback_scan"
        if len(fallback_strings) == 1:
            return fallback_strings[0], None, "parsed_compobj_fallback_partial"
        return None, None, "compobj_parse_failed"


def _decode_lpstr(raw: bytes, codepage: int | None) -> str:
    if codepage in (None, 0, 1200):
        encoding = "cp1252"
    else:
        encoding = f"cp{codepage}"
    try:
        return raw.decode(encoding, errors="ignore")
    except LookupError:
        return raw.decode("latin-1", errors="ignore")


def _format_guid_le(guid_bytes: bytes) -> str:
    if len(guid_bytes) != 16:
        raise ValueError(f"GUID must be 16 bytes, got {len(guid_bytes)}")
    data1 = int.from_bytes(guid_bytes[0:4], "little", signed=False)
    data2 = int.from_bytes(guid_bytes[4:6], "little", signed=False)
    data3 = int.from_bytes(guid_bytes[6:8], "little", signed=False)
    data4 = guid_bytes[8:10].hex().upper()
    data5 = guid_bytes[10:16].hex().upper()
    return f"{{{data1:08X}-{data2:04X}-{data3:04X}-{data4}-{data5}}}"


def _parse_property_dictionary(
    section_bytes: bytes, value_offset: int, codepage: int | None
) -> dict[int, str]:
    entry_count = _u32le(section_bytes, value_offset, "PropertySet dictionary")
    cursor = value_offset + 4
    result: dict[int, str] = {}

    for _ in range(entry_count):
        prop_id = _u32le(section_bytes, cursor, "PropertySet dictionary entry")
        cursor += 4
        char_count = _u32le(section_bytes, cursor, "PropertySet dictionary entry")
        cursor += 4

        if codepage == 1200:
            byte_len = char_count * 2
            raw = section_bytes[cursor : cursor + byte_len]
            if len(raw) != byte_len:
                raise ValueError("PropertySet dictionary is truncated (UTF-16 text)")
            name = raw.decode("utf-16le", errors="ignore").rstrip("\x00")
        else:
            byte_len = char_count
            raw = section_bytes[cursor : cursor + byte_len]
            if len(raw) != byte_len:
                raise ValueError("PropertySet dictionary is truncated (ANSI text)")
            name = _decode_lpstr(raw, codepage).rstrip("\x00")

        cursor += byte_len
        while (cursor - value_offset) % 4 != 0:
            cursor += 1
            if cursor > len(section_bytes):
                raise ValueError("PropertySet dictionary alignment moved out of bounds")

        result[prop_id] = name.strip()

    return result


def _parse_typed_property_value(
    section_bytes: bytes, value_offset: int, codepage: int | None
) -> tuple[Any, int]:
    vt = _u16le(section_bytes, value_offset, "PropertySet typed value")

    if vt == 0x0002:  # VT_I2
        return int.from_bytes(section_bytes[value_offset + 4 : value_offset + 6], "little", signed=True), vt
    if vt == 0x0003:  # VT_I4
        return int.from_bytes(section_bytes[value_offset + 4 : value_offset + 8], "little", signed=True), vt
    if vt == 0x0013:  # VT_UI4
        return _u32le(section_bytes, value_offset + 4, "PropertySet VT_UI4"), vt
    if vt == 0x000B:  # VT_BOOL
        bool_word = _u16le(section_bytes, value_offset + 4, "PropertySet VT_BOOL")
        return (bool_word != 0), vt
    if vt == 0x0008:  # VT_BSTR
        byte_len = _u32le(section_bytes, value_offset + 4, "PropertySet VT_BSTR")
        raw = section_bytes[value_offset + 8 : value_offset + 8 + byte_len]
        if len(raw) != byte_len:
            raise ValueError("PropertySet VT_BSTR is truncated")
        return raw.decode("utf-16le", errors="ignore").rstrip("\x00"), vt
    if vt == 0x001E:  # VT_LPSTR
        byte_len = _u32le(section_bytes, value_offset + 4, "PropertySet VT_LPSTR")
        raw = section_bytes[value_offset + 8 : value_offset + 8 + byte_len]
        if len(raw) != byte_len:
            raise ValueError("PropertySet VT_LPSTR is truncated")
        return _decode_lpstr(raw, codepage).rstrip("\x00"), vt
    if vt == 0x001F:  # VT_LPWSTR
        char_count = _u32le(section_bytes, value_offset + 4, "PropertySet VT_LPWSTR")
        byte_len = char_count * 2
        raw = section_bytes[value_offset + 8 : value_offset + 8 + byte_len]
        if len(raw) != byte_len:
            raise ValueError("PropertySet VT_LPWSTR is truncated")
        return raw.decode("utf-16le", errors="ignore").rstrip("\x00"), vt
    if vt == 0x0048:  # VT_CLSID
        raw = section_bytes[value_offset + 4 : value_offset + 20]
        if len(raw) != 16:
            raise ValueError("PropertySet VT_CLSID is truncated")
        return _format_guid_le(raw), vt

    return None, vt


def _parse_propertyset_section(section_bytes: bytes) -> tuple[dict[int, Any], dict[int, str], int | None]:
    _ = _u32le(section_bytes, 0, "PropertySet section size")
    prop_count = _u32le(section_bytes, 4, "PropertySet section property count")

    properties: list[tuple[int, int]] = []
    for idx in range(prop_count):
        item_offset = 8 + idx * 8
        prop_id = _u32le(section_bytes, item_offset, "PropertySet property id")
        value_offset = _u32le(section_bytes, item_offset + 4, "PropertySet property value offset")
        properties.append((prop_id, value_offset))

    codepage: int | None = None
    for prop_id, value_offset in properties:
        if prop_id != 1:
            continue
        value, vt = _parse_typed_property_value(section_bytes, value_offset, None)
        if vt == 0x0002 and isinstance(value, int):
            codepage = value
        break

    values: dict[int, Any] = {}
    dictionary: dict[int, str] = {}
    for prop_id, value_offset in properties:
        if prop_id == 0:
            dictionary = _parse_property_dictionary(section_bytes, value_offset, codepage)
            continue
        value, _ = _parse_typed_property_value(section_bytes, value_offset, codepage)
        values[prop_id] = value

    return values, dictionary, codepage


def parse_propertyset_stream(stream_bytes: bytes) -> dict[str, Any]:
    result = {
        "status": "propertyset_parse_failed",
        "type_id": None,
        "method_id": None,
        "revision_number": None,
    }

    if len(stream_bytes) < 28:
        result["status"] = "propertyset_too_short"
        return result

    byte_order = _u16le(stream_bytes, 0, "PropertySet header")
    if byte_order != CFBF_BYTE_ORDER_LE:
        result["status"] = f"propertyset_invalid_byte_order_0x{byte_order:04x}"
        return result

    section_count = _u32le(stream_bytes, 24, "PropertySet header")
    if section_count == 0:
        result["status"] = "propertyset_no_sections"
        return result

    values: dict[int, Any] = {}
    dictionary: dict[int, str] = {}
    parsed_sections = 0

    for idx in range(section_count):
        section_item_offset = 28 + idx * 20
        if section_item_offset + 20 > len(stream_bytes):
            break
        section_offset = _u32le(stream_bytes, section_item_offset + 16, "PropertySet section offset")
        if section_offset >= len(stream_bytes):
            continue
        if section_offset + 8 > len(stream_bytes):
            continue

        section_size = _u32le(stream_bytes, section_offset, "PropertySet section size")
        section_end = section_offset + section_size
        if section_size < 8 or section_end > len(stream_bytes):
            continue

        section_bytes = stream_bytes[section_offset:section_end]
        try:
            section_values, section_dict, _ = _parse_propertyset_section(section_bytes)
        except ValueError:
            continue

        parsed_sections += 1
        for prop_id, value in section_values.items():
            values.setdefault(prop_id, value)
        for prop_id, name in section_dict.items():
            if name and prop_id not in dictionary:
                dictionary[prop_id] = name

    if parsed_sections == 0:
        result["status"] = "propertyset_no_readable_sections"
        return result

    name_map = {name.strip().lower(): prop_id for prop_id, name in dictionary.items() if name}
    type_pid = name_map.get("type_id", 2)
    method_pid = name_map.get("id", 3)
    revision_pid = name_map.get("revision_number", 4)

    type_id_value = values.get(type_pid)
    method_id_value = values.get(method_pid)
    revision_value = values.get(revision_pid)

    if isinstance(type_id_value, str):
        result["type_id"] = type_id_value.strip() or None
    elif type_id_value is not None:
        result["type_id"] = str(type_id_value)

    if method_id_value is not None:
        normalized_method_id = str(method_id_value).strip()
        result["method_id"] = normalized_method_id if normalized_method_id else None

    if isinstance(revision_value, int):
        result["revision_number"] = revision_value
    elif isinstance(revision_value, str):
        result["revision_number"] = int(revision_value) if revision_value.isdigit() else None

    if result["type_id"] or result["method_id"] is not None or result["revision_number"] is not None:
        result["status"] = "parsed_propertyset"
    else:
        result["status"] = "propertyset_parsed_no_target_fields"
    return result


def extract_method_metadata_from_cfbf(parsed: CfbfDecodeResult | None) -> MsMethodMetadata:
    if parsed is None:
        return MsMethodMetadata(
            parse_status="cfbf_parse_failed",
            compobj_parse_status="compobj_unavailable",
            propertyset_parse_status="propertyset_unavailable",
            com_class=None,
            progid=None,
            type_id=None,
            method_id=None,
            revision_number=None,
        )

    comp_stream = cfbf_get_stream_by_name(parsed, "\x01CompObj")
    property_stream = next((stream for stream in parsed.streams if stream.name.startswith("\x05")), None)

    compobj_parse_status = "compobj_missing"
    com_class: str | None = None
    progid: str | None = None
    if comp_stream is not None:
        com_class, progid, compobj_parse_status = parse_compobj_stream(comp_stream.data)

    propertyset_parse_status = "propertyset_missing"
    type_id: str | None = None
    method_id: str | None = None
    revision_number: int | None = None
    if property_stream is not None:
        parsed_property = parse_propertyset_stream(property_stream.data)
        propertyset_parse_status = parsed_property["status"]
        type_id = parsed_property["type_id"]
        method_id = parsed_property["method_id"]
        revision_number = parsed_property["revision_number"]

    has_compobj = bool(com_class or progid)
    has_propertyset = bool(type_id or method_id is not None or revision_number is not None)
    if has_compobj and has_propertyset:
        parse_status = "parsed_compobj_propertyset"
    elif has_compobj:
        parse_status = "parsed_compobj_only"
    elif has_propertyset:
        parse_status = "parsed_propertyset_only"
    elif comp_stream is None and property_stream is None:
        parse_status = "method_streams_missing"
    else:
        parse_status = "method_stream_parse_failed"

    return MsMethodMetadata(
        parse_status=parse_status,
        compobj_parse_status=compobj_parse_status,
        propertyset_parse_status=propertyset_parse_status,
        com_class=com_class,
        progid=progid,
        type_id=type_id,
        method_id=method_id,
        revision_number=revision_number,
    )


def apply_method_metadata(payload: dict[str, Any], metadata: MsMethodMetadata) -> None:
    payload["ms_method_parse_status"] = metadata.parse_status
    payload["ms_method_progid"] = metadata.progid
    payload["ms_method_com_class"] = metadata.com_class
    payload["ms_method_type_id"] = metadata.type_id
    payload["ms_method_profile_token"] = metadata.method_id
    payload["ms_method_model_token"] = (
        str(metadata.revision_number) if metadata.revision_number is not None else None
    )


def extract_qtof_report_texts_from_stg(
    stg_bytes: bytes, parsed: CfbfDecodeResult | None = None
) -> tuple[list[str], str, MsMethodParseResult | None]:
    parsed_local = parsed
    if parsed_local is None:
        try:
            parsed_local = parse_cfbf_structure(stg_bytes)
        except ValueError:
            return [], "cfbf_parse_failed", None

    primary_stream = cfbf_get_stream_by_name(parsed_local, "MSMETHOD")
    if primary_stream is None:
        return [], "cfbf_stream_missing:MSMETHOD", None

    # Strict decode: validate primary/extension/postamble structures from MSMETHOD.
    fixed_header = parse_msmethod_fixed_header(primary_stream.data)
    parse_result = parse_msmethod_parse_result(primary_stream.data, fixed_header=fixed_header)

    decoded = primary_stream.data.decode("utf-16le", errors="ignore")
    report_texts = QTOF_REPORT_PATTERN.findall(decoded)
    unique_reports = list(dict.fromkeys(report_texts))
    return unique_reports, "cfbf_stream:MSMETHOD", parse_result


def _msmethod_value_to_json_compatible(
    value: float | int | str | bool | MsMethodArrayValue | None,
) -> float | int | str | bool | dict[str, Any] | None:
    if isinstance(value, MsMethodArrayValue):
        if value.rank >= 3:
            raise AssertionError(
                "MSMETHOD array rank>=3 is not supported yet "
                f"(rank={value.rank}, dims={value.dims})"
            )
        return {
            "type": "array",
            "rank": value.rank,
            "dims": list(value.dims),
            "n_values": len(value.values),
            "values": list(value.values),
        }
    if isinstance(value, str) and len(value) > 400:
        preview = value[:240].replace("\r", " ").replace("\n", " ")
        return {
            "type": "text_preview",
            "length": len(value),
            "preview": preview,
        }
    return value


def _msmethod_unknown_fields_to_json_payload(
    fields: tuple[MsMethodUnknownField, ...],
) -> list[dict[str, Any]]:
    return [
        {
            "index": field.index,
            "name": field.name,
            "values": [
                _msmethod_value_to_json_compatible(tagged_value.value)
                for tagged_value in field.values
            ],
        }
        for field in fields
    ]


def msmethod_time_segments_to_json(parse_result: MsMethodParseResult | None) -> str | None:
    if parse_result is None or not parse_result.time_segments:
        return None

    payload: list[dict[str, Any]] = []
    for segment in parse_result.time_segments:
        payload.append(
            {
                "segment_index": segment.index + 1,
                "start_time_min": segment.start_time,
                "n_experimental_segments": segment.n_experimental_segments,
                "primary_unknown_fields": _msmethod_unknown_fields_to_json_payload(
                    segment.primary_unknown_fields
                ),
                "extension_unknown_fields": _msmethod_unknown_fields_to_json_payload(
                    segment.extension_unknown_fields
                ),
            }
        )
    return json.dumps(payload, ensure_ascii=True)


def msmethod_postamble_to_json(parse_result: MsMethodParseResult | None) -> str | None:
    if parse_result is None:
        return None
    payload = _msmethod_unknown_fields_to_json_payload(parse_result.postamble.unknown_fields)
    return json.dumps(payload, ensure_ascii=True)


def extract_qtof_report_fields(root: ET.Element) -> dict[str, Any]:
    expected_masses: list[float] = []
    resolutions: list[float] = []
    for mass_node in root.findall(".//TTISpectrum/MassList"):
        expected_mass = to_float(text_at(mass_node, "ExpectedMass"))
        if expected_mass is not None:
            expected_masses.append(expected_mass)
        resolution = to_float(text_at(mass_node, "Resolution"))
        if resolution is not None:
            resolutions.append(resolution)

    return {
        "ms_tune_datetime": text_at(root, "TuneDateTime"),
        "ms_model": text_at(root, "msModel"),
        "ms_serial_number": text_at(root, "SerialNumber"),
        "ms_firmware_version": text_at(root, "FirmwareVersion"),
        "ms_source_type": text_at(root, "sourceType"),
        "ms_mass_range": text_at(root, "MassRange"),
        "ms_instrument_mode": text_at(root, "InstrumentMode"),
        "ms_suremass": text_at(root, "SureMass"),
        "ms_min_range_mz": to_float(text_at(root, "TOFMassRange/MinRangeTune")),
        "ms_max_range_mz": to_float(text_at(root, "TOFMassRange/MaxRangeTune")),
        "ms_acquisition_rate_hz": to_float(text_at(root, "TOFAcquisitionRate/AcquisitionRate")),
        "ms_acquisition_time_ms": to_float(text_at(root, "TOFAcquisitionRate/AcquisitionTime")),
        "ms_set_src_gas_temp_C": to_float(text_at(root, "Setpoints/Source/GasTempTune")),
        "ms_set_src_drying_gas_L_min": to_float(text_at(root, "Setpoints/Source/DryingGasTune")),
        "ms_set_src_neb_psi": to_float(text_at(root, "Setpoints/Source/NebPresTune")),
        "ms_set_src_vcap_V": to_float(text_at(root, "Setpoints/Source/VCap")),
        "ms_set_src_nozzle_voltage_V": to_float(text_at(root, "Setpoints/Source/NozzleVoltage")),
        "ms_set_src_sheath_gas_temp_C": to_float(text_at(root, "Setpoints/Source/SheathGasTempTune")),
        "ms_set_src_sheath_gas_flow_L_min": to_float(text_at(root, "Setpoints/Source/SheathGasFlowTune")),
        "ms_act_src_gas_temp_C": to_float(text_at(root, "Actuals/Source/GasTemp")),
        "ms_act_src_vapor_temp_C": to_float(text_at(root, "Actuals/Source/Vapor")),
        "ms_act_src_drying_gas_L_min": to_float(text_at(root, "Actuals/Source/DryingGas")),
        "ms_act_src_neb_psi": to_float(text_at(root, "Actuals/Source/NebPres")),
        "ms_act_src_vcap_V": to_float(text_at(root, "Actuals/Source/VCapES")),
        "ms_set_opt1_fragment_V": to_float(text_at(root, "Setpoints/Optics1/FragmentTune")),
        "ms_set_opt1_skim1_V": to_float(text_at(root, "Setpoints/Optics1/Skim1Tune")),
        "ms_set_opt1_oct_peak_V": to_float(text_at(root, "Setpoints/Optics1/OctPeakTune")),
        "ms_set_opt1_lens1_V": to_float(text_at(root, "Setpoints/Optics1/Lens1")),
        "ms_set_opt1_lens2_V": to_float(text_at(root, "Setpoints/Optics1/Lens2")),
        "ms_set_quad_mass_dac_amu": to_float(text_at(root, "Setpoints/Quad/MassDAC")),
        "ms_set_quad_quad_dc_V": to_float(text_at(root, "Setpoints/Quad/QuadDC")),
        "ms_set_cell_cc_flow_psi": to_float(text_at(root, "Setpoints/Cell/CCFlow")),
        "ms_set_cell_hex_rf_V": to_float(text_at(root, "Setpoints/Cell/HexRF")),
        "ms_set_cell_skim2_V": to_float(text_at(root, "Setpoints/Cell/Skim2")),
        "ms_set_opt2_extractor_dc_V": to_float(text_at(root, "Setpoints/Optics2/ExtractorDC")),
        "ms_set_opt2_ion_focus_V": to_float(text_at(root, "Setpoints/Optics2/IonFocus")),
        "ms_set_opt2_slit_left_V": to_float(text_at(root, "Setpoints/Optics2/SlitLeft")),
        "ms_set_opt2_slit_right_V": to_float(text_at(root, "Setpoints/Optics2/SlitRight")),
        "ms_set_tof_vpulse_V": to_float(text_at(root, "Setpoints/TOF/VPulse")),
        "ms_set_tof_pusher_offset_mV": to_float(text_at(root, "Setpoints/TOF/PusherOffset")),
        "ms_set_tof_vliner_V": to_float(text_at(root, "Setpoints/TOF/VLiner")),
        "ms_set_acq_num_pts": int(to_float(text_at(root, "Setpoints/TOFAcquisitionRate/numPts")) or 0) or None,
        "ms_set_acq_num_trans": to_float(text_at(root, "Setpoints/TOFAcquisitionRate/numTransTune")),
        "ms_set_det_vmcpback_V": to_float(text_at(root, "Setpoints/Detector/Vmcpback")),
        "ms_set_det_preamp_offset": to_float(text_at(root, "Setpoints/Detector/PreAmpOffset")),
        "ms_set_det_dualgain_ratio": to_float(text_at(root, "Setpoints/Detector/DualGainLowHighRatio")),
        "ms_act_src_corona_vol_V": to_float(text_at(root, "Actuals/Source/CoronaVol")),
        "ms_act_src_cap_cur": to_float(text_at(root, "Actuals/Source/CapCur")),
        "ms_act_src_cham_cur": to_float(text_at(root, "Actuals/Source/ChamCur")),
        "ms_tune_report_type": text_at(root, "TuneReportType"),
        "ms_tune_report_title": text_at(root, "TuneReportTitle"),
        "ms_tune_data_path": text_at(root, "TuneDataPathName"),
        "ms_tune_user": text_at(root, "User"),
        "ms_tune_last_user": text_at(root, "lastUser"),
        "ms_tune_slicer_mode": text_at(root, "SlicerMode"),
        "ms_tune_state_save_delta": text_at(root, "InstrumentState/SaveDelta"),
        "ms_tune_state_gain_channel": text_at(root, "InstrumentState/GainChannel"),
        "ms_tune_state_high_resolution_enabled": text_at(root, "InstrumentState/HighResolutionEnabled"),
        "ms_tune_cal_a": to_float(text_at(root, "Calibration/A")),
        "ms_tune_cal_b": to_float(text_at(root, "Calibration/B")),
        "ms_tune_cal_c": to_float(text_at(root, "Calibration/C")),
        "ms_tune_cal_d": to_float(text_at(root, "Calibration/D")),
        "ms_tune_cal_e": to_float(text_at(root, "Calibration/E")),
        "ms_tune_cal_f": to_float(text_at(root, "Calibration/F")),
        "ms_tune_cal_poly_terms_flag": text_at(root, "Calibration/PolyTermsFlag"),
        "ms_tune_cal_trad_weighting": to_float(text_at(root, "Calibration/TradWeighting")),
        "ms_tune_cal_poly_weighting": to_float(text_at(root, "Calibration/PolyWeighting")),
        "ms_tune_vac_quad_temp_C": to_float(text_at(root, "Actuals/QTOF_VacuumTemp/QuadTemp")),
        "ms_tune_vac_rough_torr": to_float(text_at(root, "Actuals/QTOF_VacuumTemp/RoughVac")),
        "ms_tune_vac_high_torr": to_float(text_at(root, "Actuals/QTOF_VacuumTemp/HighVac")),
        "ms_tune_vac_octknee_torr": to_float(text_at(root, "Actuals/QTOF_VacuumTemp/OctKnee")),
        "ms_tune_vac_turbo1_pwr_w": to_float(text_at(root, "Actuals/QTOF_VacuumTemp/Turbo1Pwr")),
        "ms_tune_vac_turbo1_spd_pct": to_float(text_at(root, "Actuals/QTOF_VacuumTemp/Turbo1Spd")),
        "ms_tune_vac_turbo2_pwr_w": to_float(text_at(root, "Actuals/QTOF_VacuumTemp/Turbo2Pwr")),
        "ms_tune_vac_turbo2_spd_pct": to_float(text_at(root, "Actuals/QTOF_VacuumTemp/Turbo2Spd")),
        "ms_tune_cal_point_count": len(expected_masses),
        "ms_tune_expected_mass_min_mz": min(expected_masses) if expected_masses else None,
        "ms_tune_expected_mass_max_mz": max(expected_masses) if expected_masses else None,
        "ms_tune_resolution_mean": mean_or_none(resolutions),
        "ms_tune_resolution_min": min(resolutions) if resolutions else None,
        "ms_tune_resolution_max": max(resolutions) if resolutions else None,
    }


def summarize_qtof_polarity(entries: list[dict[str, Any]]) -> dict[str, Any]:
    hashes = [entry["hash"] for entry in entries]
    unique_hashes = sorted(set(hashes))
    consistency = "missing"
    if entries:
        consistency = "consistent" if len(unique_hashes) == 1 else "mismatch"

    first_payload: dict[str, Any] | None = None
    first_hash: str | None = None
    if entries:
        first_payload = extract_qtof_report_fields(entries[0]["root"])
        first_hash = entries[0]["hash"]

    return {
        "count": len(entries),
        "unique_hash_count": len(unique_hashes),
        "selected_hash": first_hash,
        "all_hashes": ";".join(unique_hashes) if unique_hashes else None,
        "hash_consistency": consistency,
        "json": json.dumps(first_payload, ensure_ascii=True) if first_payload is not None else None,
    }


def derive_qtof_xml_header(root: ET.Element, polarity_text: str | None) -> str:
    title = text_at(root, "TuneReportTitle")
    report_type = text_at(root, "TuneReportType")
    instrument_mode = text_at(root, "InstrumentMode")
    base = title or report_type or instrument_mode or "QTOFTuneReport"
    polarity = (polarity_text or "").strip()
    if polarity and polarity.lower() not in base.lower():
        return f"{base} ({polarity})"
    return base


def empty_ms_payload(status: str, size_bytes: int | None, md5_hex: str | None) -> dict[str, Any]:
    return {
        "ms_parse_status": status,
        "stg_size_bytes": size_bytes,
        "stg_md5": md5_hex,
        "ms_report_count": None,
        "ms_available_polarities": None,
        "ms_selected_polarity": None,
        "ms_tune_datetime": None,
        "ms_model": None,
        "ms_serial_number": None,
        "ms_firmware_version": None,
        "ms_source_type": None,
        "ms_mass_range": None,
        "ms_instrument_mode": None,
        "ms_suremass": None,
        "ms_min_range_mz": None,
        "ms_max_range_mz": None,
        "ms_acquisition_rate_hz": None,
        "ms_acquisition_time_ms": None,
        "ms_set_src_gas_temp_C": None,
        "ms_set_src_drying_gas_L_min": None,
        "ms_set_src_neb_psi": None,
        "ms_set_src_vcap_V": None,
        "ms_set_src_nozzle_voltage_V": None,
        "ms_set_src_sheath_gas_temp_C": None,
        "ms_set_src_sheath_gas_flow_L_min": None,
        "ms_act_src_gas_temp_C": None,
        "ms_act_src_vapor_temp_C": None,
        "ms_act_src_drying_gas_L_min": None,
        "ms_act_src_neb_psi": None,
        "ms_act_src_vcap_V": None,
        "ms_tune_report_type": None,
        "ms_tune_report_title": None,
        "ms_tune_data_path": None,
        "ms_tune_user": None,
        "ms_tune_last_user": None,
        "ms_tune_slicer_mode": None,
        "ms_tune_state_save_delta": None,
        "ms_tune_state_gain_channel": None,
        "ms_tune_state_high_resolution_enabled": None,
        "ms_tune_cal_a": None,
        "ms_tune_cal_b": None,
        "ms_tune_cal_c": None,
        "ms_tune_cal_d": None,
        "ms_tune_cal_e": None,
        "ms_tune_cal_f": None,
        "ms_tune_cal_poly_terms_flag": None,
        "ms_tune_cal_trad_weighting": None,
        "ms_tune_cal_poly_weighting": None,
        "ms_tune_vac_quad_temp_C": None,
        "ms_tune_vac_rough_torr": None,
        "ms_tune_vac_high_torr": None,
        "ms_tune_vac_octknee_torr": None,
        "ms_tune_vac_turbo1_pwr_w": None,
        "ms_tune_vac_turbo1_spd_pct": None,
        "ms_tune_vac_turbo2_pwr_w": None,
        "ms_tune_vac_turbo2_spd_pct": None,
        "ms_tune_cal_point_count": None,
        "ms_tune_expected_mass_min_mz": None,
        "ms_tune_expected_mass_max_mz": None,
        "ms_tune_resolution_mean": None,
        "ms_tune_resolution_min": None,
        "ms_tune_resolution_max": None,
        "ms_mode_hint": "disabled_no_binary_parse",
        "ms_set_opt1_fragment_V": None,
        "ms_set_opt1_skim1_V": None,
        "ms_set_opt1_oct_peak_V": None,
        "ms_set_opt1_lens1_V": None,
        "ms_set_opt1_lens2_V": None,
        "ms_set_quad_mass_dac_amu": None,
        "ms_set_quad_quad_dc_V": None,
        "ms_set_cell_cc_flow_psi": None,
        "ms_set_cell_hex_rf_V": None,
        "ms_set_cell_skim2_V": None,
        "ms_set_opt2_extractor_dc_V": None,
        "ms_set_opt2_ion_focus_V": None,
        "ms_set_opt2_slit_left_V": None,
        "ms_set_opt2_slit_right_V": None,
        "ms_set_tof_vpulse_V": None,
        "ms_set_tof_pusher_offset_mV": None,
        "ms_set_tof_vliner_V": None,
        "ms_set_acq_num_pts": None,
        "ms_set_acq_num_trans": None,
        "ms_set_det_vmcpback_V": None,
        "ms_set_det_preamp_offset": None,
        "ms_set_det_dualgain_ratio": None,
        "ms_act_src_corona_vol_V": None,
        "ms_act_src_cap_cur": None,
        "ms_act_src_cham_cur": None,
        "ms_method_parse_status": "disabled_no_binary_parse",
        "ms_method_progid": None,
        "ms_method_com_class": None,
        "ms_method_type_id": None,
        "ms_method_default_tune_file": None,
        "ms_method_targeted_list_file": None,
        "ms_method_profile_token": None,
        "ms_method_model_token": None,
        "ms_method_polarity": None,
        "ms_method_source_type": None,
        "ms_method_mass_range": None,
        "ms_method_instrument_mode": None,
        "ms_method_acquisition_rate_hz": None,
        "ms_method_acquisition_time_ms": None,
        "ms_method_cc_gas": None,
        "ms_method_acq_holdoff_delay_ms": None,
        "ms_method_time_segments_json": None,
        "ms_method_postamble_json": None,
        "ms_embedded_xml_count": None,
        "ms_embedded_xml_headers": None,
        "ms_embedded_xml_payload_json": None,
        "ms_bin_candidate_parse_status": "disabled_no_binary_parse",
        "ms_bin_gap_lengths": None,
        "ms_bin_tagged_double_count": None,
        "ms_bin_tagged_double_values": None,
        "ms_bin_method_extraction_note": None,
        "ms_bin_method_gas_temp_C": None,
        "ms_bin_method_drying_gas_L_min": None,
        "ms_bin_method_nebulizer_psi": None,
        "ms_bin_method_nebulizer_source": None,
        "ms_bin_method_sheath_gas_temp_C": None,
        "ms_bin_method_sheath_gas_flow_L_min": None,
        "ms_bin_method_vcap_V": None,
        "ms_bin_method_nozzle_voltage_V": None,
        "ms_bin_method_fragmentor_V": None,
        "ms_bin_method_skimmer_V": None,
        "ms_bin_method_oct1_rf_vpp_V": None,
        "ms_tune_positive_json": None,
        "ms_tune_negative_json": None,
        "ms_tune_positive_count": None,
        "ms_tune_negative_count": None,
        "ms_tune_positive_unique_hash_count": None,
        "ms_tune_negative_unique_hash_count": None,
        "ms_tune_positive_selected_hash": None,
        "ms_tune_negative_selected_hash": None,
        "ms_tune_positive_all_hashes": None,
        "ms_tune_negative_all_hashes": None,
        "ms_tune_positive_hash_consistency": None,
        "ms_tune_negative_hash_consistency": None,
    }


def parse_ms_stg(path: Path, expected_polarity: str | None) -> dict[str, Any]:
    if not path.exists():
        return empty_ms_payload("stg_missing", None, None)

    stg_bytes = path.read_bytes()
    stg_md5 = hashlib.md5(stg_bytes).hexdigest()
    stg_size = path.stat().st_size
    try:
        parsed_cfbf = parse_cfbf_structure(stg_bytes)
    except ValueError:
        parsed_cfbf = None
    method_metadata = extract_method_metadata_from_cfbf(parsed_cfbf)

    # Strict mode for MSMETHOD header validation: no fallback on parse errors.
    report_texts, mode_hint, msmethod_parse_result = extract_qtof_report_texts_from_stg(
        stg_bytes, parsed=parsed_cfbf
    )
    ms_method_time_segments_json = msmethod_time_segments_to_json(msmethod_parse_result)
    ms_method_postamble_json = msmethod_postamble_to_json(msmethod_parse_result)

    mode_hint = (
        f"{mode_hint};compobj={method_metadata.compobj_parse_status};"
        f"propertyset={method_metadata.propertyset_parse_status}"
    )

    if not report_texts:
        payload = empty_ms_payload("qtof_xml_not_found", stg_size, stg_md5)
        payload["ms_mode_hint"] = mode_hint
        payload["ms_method_time_segments_json"] = ms_method_time_segments_json
        payload["ms_method_postamble_json"] = ms_method_postamble_json
        apply_method_metadata(payload, method_metadata)
        return payload

    report_entries: list[dict[str, Any]] = []
    for report_text in report_texts:
        try:
            root = ET.fromstring(report_text)
        except ET.ParseError:
            continue
        polarity_text = text_at(root, "polarity")
        header = derive_qtof_xml_header(root, polarity_text)
        report_entries.append(
            {
                "root": root,
                "header": header,
                "xml_text": report_text,
                "polarity_text": polarity_text,
                "polarity_key": normalize_polarity_key(polarity_text),
                "hash": hashlib.md5(ET.tostring(root, encoding="utf-8")).hexdigest(),
            }
        )

    if not report_entries:
        payload = empty_ms_payload("qtof_xml_parse_failed", stg_size, stg_md5)
        payload["ms_mode_hint"] = mode_hint
        payload["ms_method_time_segments_json"] = ms_method_time_segments_json
        payload["ms_method_postamble_json"] = ms_method_postamble_json
        apply_method_metadata(payload, method_metadata)
        return payload

    grouped: dict[str, list[dict[str, Any]]] = {}
    for entry in report_entries:
        grouped.setdefault(entry["polarity_key"], []).append(entry)

    positive_summary = summarize_qtof_polarity(grouped.get("positive", []))
    negative_summary = summarize_qtof_polarity(grouped.get("negative", []))

    selected_entry: dict[str, Any] | None = None
    fallback_used = False
    if expected_polarity:
        matching = grouped.get(normalize_polarity_key(expected_polarity), [])
        if matching:
            selected_entry = matching[0]
        else:
            fallback_used = True

    if selected_entry is None:
        selected_entry = report_entries[0]

    selected_root = selected_entry["root"]
    selected_fields = extract_qtof_report_fields(selected_root)

    polarities = sorted(
        {
            (entry["polarity_text"] or "").strip()
            for entry in report_entries
            if (entry["polarity_text"] or "").strip()
        }
    )
    embedded_xml_headers_unique = list(
        dict.fromkeys(entry["header"] for entry in report_entries if entry.get("header"))
    )
    embedded_xml_payload_json = json.dumps(
        [
            {
                "header": entry["header"],
                "polarity": entry["polarity_text"],
                "hash": entry["hash"],
                "xml": entry["xml_text"],
            }
            for entry in report_entries
        ],
        ensure_ascii=True,
    )
    payload = empty_ms_payload(
        "parsed_qtof_xml_fallback_polarity" if fallback_used else "parsed_qtof_xml",
        stg_size,
        stg_md5,
    )
    payload.update(
        {
            "ms_report_count": len(report_entries),
            "ms_available_polarities": ";".join(polarities) if polarities else None,
            "ms_selected_polarity": selected_entry["polarity_text"],
            "ms_mode_hint": mode_hint,
            "ms_method_default_tune_file": selected_fields.get("ms_tune_data_path"),
            "ms_method_polarity": selected_entry["polarity_text"],
            "ms_method_source_type": selected_fields.get("ms_source_type"),
            "ms_method_mass_range": selected_fields.get("ms_mass_range"),
            "ms_method_instrument_mode": selected_fields.get("ms_instrument_mode"),
            "ms_method_acquisition_rate_hz": selected_fields.get("ms_acquisition_rate_hz"),
            "ms_method_acquisition_time_ms": selected_fields.get("ms_acquisition_time_ms"),
            "ms_method_cc_gas": text_at(selected_root, "Setpoints/Cell/CCGas"),
            "ms_method_acq_holdoff_delay_ms": to_float(text_at(selected_root, "AcqHoldOffDelayTime")),
            "ms_method_time_segments_json": ms_method_time_segments_json,
            "ms_method_postamble_json": ms_method_postamble_json,
            "ms_embedded_xml_count": len(report_entries),
            "ms_embedded_xml_headers": (
                ";".join(embedded_xml_headers_unique) if embedded_xml_headers_unique else None
            ),
            "ms_embedded_xml_payload_json": embedded_xml_payload_json,
            "ms_tune_positive_json": positive_summary["json"],
            "ms_tune_negative_json": negative_summary["json"],
            "ms_tune_positive_count": positive_summary["count"],
            "ms_tune_negative_count": negative_summary["count"],
            "ms_tune_positive_unique_hash_count": positive_summary["unique_hash_count"],
            "ms_tune_negative_unique_hash_count": negative_summary["unique_hash_count"],
            "ms_tune_positive_selected_hash": positive_summary["selected_hash"],
            "ms_tune_negative_selected_hash": negative_summary["selected_hash"],
            "ms_tune_positive_all_hashes": positive_summary["all_hashes"],
            "ms_tune_negative_all_hashes": negative_summary["all_hashes"],
            "ms_tune_positive_hash_consistency": positive_summary["hash_consistency"],
            "ms_tune_negative_hash_consistency": negative_summary["hash_consistency"],
        }
    )
    payload.update(selected_fields)
    apply_method_metadata(payload, method_metadata)
    return payload


def discover_method_dirs(root_dir: Path) -> list[Path]:
    return sorted(path for path in root_dir.glob("*.m") if path.is_dir())


def extract_gradient_points(pump_root: ET.Element) -> list[GradientPoint]:
    initial_percent_b: float | None = None
    entries: list[GradientPoint] = []

    for solvent_element in pump_root.findall(".//SolventComposition/SolventElement"):
        channel = text_at(solvent_element, "Channel")
        percentage = to_float(text_at(solvent_element, "Percentage"))
        if channel == "Channel_B":
            initial_percent_b = percentage
            break

    if initial_percent_b is None:
        initial_percent_b = 0.0

    entries.append(GradientPoint(time_min=0.0, percent_b=initial_percent_b))

    for timetable_entry in pump_root.findall(".//Timetable/TimetableEntry"):
        time_min = to_float(text_at(timetable_entry, "Time"))
        percent_b = to_float(text_at(timetable_entry, "PercentB"))
        if time_min is None or percent_b is None:
            continue
        entries.append(GradientPoint(time_min=time_min, percent_b=percent_b))

    entries.sort(key=lambda x: x.time_min)
    return entries


def build_gradient_profile(points: list[GradientPoint]) -> tuple[str, str]:
    compact = " | ".join(f"{p.time_min:g}:{p.percent_b:g}" for p in points)
    payload = [{"time_min": p.time_min, "percent_b": p.percent_b} for p in points]
    return compact, json.dumps(payload, ensure_ascii=True)


def parse_method_dir(method_dir: Path) -> MethodSummary | None:
    missing_files = [name for name in REQUIRED_XML_FILES if not (method_dir / name).exists()]
    if missing_files:
        warn(f"Skipping {method_dir.name}: missing required files {missing_files}")
        return None

    try:
        info_root = parse_xml(method_dir / "info.xml")
        pump_root = parse_xml(method_dir / "BinPump_1.xml")
        als_root = parse_xml(method_dir / "ALS_1.xml")
        tcc_root = parse_xml(method_dir / "TCC_1.xml")
    except ET.ParseError as exc:
        warn(f"Skipping {method_dir.name}: XML parse error ({exc})")
        return None
    except OSError as exc:
        warn(f"Skipping {method_dir.name}: file read error ({exc})")
        return None

    solvent_a_name = None
    solvent_b_name = None
    initial_percent_a = None
    initial_percent_b = None

    for solvent_element in pump_root.findall(".//SolventComposition/SolventElement"):
        channel = text_at(solvent_element, "Channel")
        name = text_at(solvent_element, "Channel1UserName")
        percentage = to_float(text_at(solvent_element, "Percentage"))
        if channel == "Channel_A":
            solvent_a_name = name
            initial_percent_a = percentage
        elif channel == "Channel_B":
            solvent_b_name = name
            initial_percent_b = percentage

    gradient_points = extract_gradient_points(pump_root)
    gradient_profile, gradient_profile_json = build_gradient_profile(gradient_points)
    expected_polarity = parse_expected_polarity(method_dir.name)
    ms_info = parse_ms_stg(method_dir / "191_1.stg", expected_polarity)

    return MethodSummary(
        method_dir=method_dir.name,
        description=text_at(info_root, "Description"),
        created_at=text_at(info_root, "CreationDate"),
        modified_at=text_at(info_root, "LastModificationDate"),
        estimated_run_time_min=to_float(text_at(info_root, "EstimatedRunTime")),
        flow_mL_min=to_float(text_at(pump_root, "Flow")),
        stop_time_min=to_float(text_at(pump_root, "StopTime/StopTimeValue")),
        post_time_min=to_float(text_at(pump_root, "PostTime/PostTimeValue")),
        low_pressure_limit_bar=to_float(text_at(pump_root, "LowPressureLimit")),
        high_pressure_limit_bar=to_float(text_at(pump_root, "HighPressureLimit")),
        max_flow_ramp=to_float(text_at(pump_root, "MaximumFlowRamp")),
        solvent_a_name=solvent_a_name,
        solvent_b_name=solvent_b_name,
        initial_percent_a=initial_percent_a,
        initial_percent_b=initial_percent_b,
        gradient_profile=gradient_profile,
        gradient_profile_json=gradient_profile_json,
        injection_mode=text_at(als_root, "Injection/InjectionMode"),
        injection_volume_uL=to_float(text_at(als_root, "Injection/InjectionVolume")),
        needle_wash_location=text_at(als_root, "Injection/NeedleWash/NeedleWashLocation"),
        needle_wash_time_s=to_float(text_at(als_root, "Injection/NeedleWash/WashTime")),
        draw_speed=to_float(text_at(als_root, "Auxiliary/DrawSpeed")),
        eject_speed=to_float(text_at(als_root, "Auxiliary/EjectSpeed")),
        wait_time_after_draw_s=to_float(text_at(als_root, "Auxiliary/WaitTimeAfterDraw")),
        draw_position_offset=to_float(text_at(als_root, "Auxiliary/DrawPositionOffset")),
        sample_flush_out_factor=to_float(text_at(als_root, "HighThroughput/SampleFlushOutFactor")),
        valve_to_bypass_after_injection=text_at(als_root, "HighThroughput/ValveToBypassAfterInjection"),
        column_valve_position=text_at(tcc_root, "ColumnValvePosition"),
        left_temp_mode=text_at(tcc_root, "LeftTemperatureControl/TemperatureControlMode"),
        left_temp_C=to_float(text_at(tcc_root, "LeftTemperatureControl/Temperature")),
        left_not_ready_limit=to_float(text_at(tcc_root, "LeftTemperatureControl/NotReadyLimit/NotReadyLimitValue")),
        right_temp_mode=text_at(tcc_root, "RightTemperatureControl/TemperatureControlMode"),
        ms_parse_status=ms_info["ms_parse_status"],
        stg_size_bytes=ms_info["stg_size_bytes"],
        stg_md5=ms_info["stg_md5"],
        ms_report_count=ms_info["ms_report_count"],
        ms_available_polarities=ms_info["ms_available_polarities"],
        ms_selected_polarity=ms_info["ms_selected_polarity"],
        ms_tune_datetime=ms_info["ms_tune_datetime"],
        ms_model=ms_info["ms_model"],
        ms_serial_number=ms_info["ms_serial_number"],
        ms_firmware_version=ms_info["ms_firmware_version"],
        ms_source_type=ms_info["ms_source_type"],
        ms_mass_range=ms_info["ms_mass_range"],
        ms_instrument_mode=ms_info["ms_instrument_mode"],
        ms_suremass=ms_info["ms_suremass"],
        ms_min_range_mz=ms_info["ms_min_range_mz"],
        ms_max_range_mz=ms_info["ms_max_range_mz"],
        ms_acquisition_rate_hz=ms_info["ms_acquisition_rate_hz"],
        ms_acquisition_time_ms=ms_info["ms_acquisition_time_ms"],
        ms_set_src_gas_temp_C=ms_info["ms_set_src_gas_temp_C"],
        ms_set_src_drying_gas_L_min=ms_info["ms_set_src_drying_gas_L_min"],
        ms_set_src_neb_psi=ms_info["ms_set_src_neb_psi"],
        ms_set_src_vcap_V=ms_info["ms_set_src_vcap_V"],
        ms_set_src_nozzle_voltage_V=ms_info["ms_set_src_nozzle_voltage_V"],
        ms_set_src_sheath_gas_temp_C=ms_info["ms_set_src_sheath_gas_temp_C"],
        ms_set_src_sheath_gas_flow_L_min=ms_info["ms_set_src_sheath_gas_flow_L_min"],
        ms_act_src_gas_temp_C=ms_info["ms_act_src_gas_temp_C"],
        ms_act_src_vapor_temp_C=ms_info["ms_act_src_vapor_temp_C"],
        ms_act_src_drying_gas_L_min=ms_info["ms_act_src_drying_gas_L_min"],
        ms_act_src_neb_psi=ms_info["ms_act_src_neb_psi"],
        ms_act_src_vcap_V=ms_info["ms_act_src_vcap_V"],
        ms_set_opt1_fragment_V=ms_info["ms_set_opt1_fragment_V"],
        ms_set_opt1_skim1_V=ms_info["ms_set_opt1_skim1_V"],
        ms_set_opt1_oct_peak_V=ms_info["ms_set_opt1_oct_peak_V"],
        ms_set_opt1_lens1_V=ms_info["ms_set_opt1_lens1_V"],
        ms_set_opt1_lens2_V=ms_info["ms_set_opt1_lens2_V"],
        ms_set_quad_mass_dac_amu=ms_info["ms_set_quad_mass_dac_amu"],
        ms_set_quad_quad_dc_V=ms_info["ms_set_quad_quad_dc_V"],
        ms_set_cell_cc_flow_psi=ms_info["ms_set_cell_cc_flow_psi"],
        ms_set_cell_hex_rf_V=ms_info["ms_set_cell_hex_rf_V"],
        ms_set_cell_skim2_V=ms_info["ms_set_cell_skim2_V"],
        ms_set_opt2_extractor_dc_V=ms_info["ms_set_opt2_extractor_dc_V"],
        ms_set_opt2_ion_focus_V=ms_info["ms_set_opt2_ion_focus_V"],
        ms_set_opt2_slit_left_V=ms_info["ms_set_opt2_slit_left_V"],
        ms_set_opt2_slit_right_V=ms_info["ms_set_opt2_slit_right_V"],
        ms_set_tof_vpulse_V=ms_info["ms_set_tof_vpulse_V"],
        ms_set_tof_pusher_offset_mV=ms_info["ms_set_tof_pusher_offset_mV"],
        ms_set_tof_vliner_V=ms_info["ms_set_tof_vliner_V"],
        ms_set_acq_num_pts=ms_info["ms_set_acq_num_pts"],
        ms_set_acq_num_trans=ms_info["ms_set_acq_num_trans"],
        ms_set_det_vmcpback_V=ms_info["ms_set_det_vmcpback_V"],
        ms_set_det_preamp_offset=ms_info["ms_set_det_preamp_offset"],
        ms_set_det_dualgain_ratio=ms_info["ms_set_det_dualgain_ratio"],
        ms_act_src_corona_vol_V=ms_info["ms_act_src_corona_vol_V"],
        ms_act_src_cap_cur=ms_info["ms_act_src_cap_cur"],
        ms_act_src_cham_cur=ms_info["ms_act_src_cham_cur"],
        ms_tune_report_type=ms_info["ms_tune_report_type"],
        ms_tune_report_title=ms_info["ms_tune_report_title"],
        ms_tune_data_path=ms_info["ms_tune_data_path"],
        ms_tune_user=ms_info["ms_tune_user"],
        ms_tune_last_user=ms_info["ms_tune_last_user"],
        ms_tune_slicer_mode=ms_info["ms_tune_slicer_mode"],
        ms_tune_state_save_delta=ms_info["ms_tune_state_save_delta"],
        ms_tune_state_gain_channel=ms_info["ms_tune_state_gain_channel"],
        ms_tune_state_high_resolution_enabled=ms_info["ms_tune_state_high_resolution_enabled"],
        ms_tune_cal_a=ms_info["ms_tune_cal_a"],
        ms_tune_cal_b=ms_info["ms_tune_cal_b"],
        ms_tune_cal_c=ms_info["ms_tune_cal_c"],
        ms_tune_cal_d=ms_info["ms_tune_cal_d"],
        ms_tune_cal_e=ms_info["ms_tune_cal_e"],
        ms_tune_cal_f=ms_info["ms_tune_cal_f"],
        ms_tune_cal_poly_terms_flag=ms_info["ms_tune_cal_poly_terms_flag"],
        ms_tune_cal_trad_weighting=ms_info["ms_tune_cal_trad_weighting"],
        ms_tune_cal_poly_weighting=ms_info["ms_tune_cal_poly_weighting"],
        ms_tune_vac_quad_temp_C=ms_info["ms_tune_vac_quad_temp_C"],
        ms_tune_vac_rough_torr=ms_info["ms_tune_vac_rough_torr"],
        ms_tune_vac_high_torr=ms_info["ms_tune_vac_high_torr"],
        ms_tune_vac_octknee_torr=ms_info["ms_tune_vac_octknee_torr"],
        ms_tune_vac_turbo1_pwr_w=ms_info["ms_tune_vac_turbo1_pwr_w"],
        ms_tune_vac_turbo1_spd_pct=ms_info["ms_tune_vac_turbo1_spd_pct"],
        ms_tune_vac_turbo2_pwr_w=ms_info["ms_tune_vac_turbo2_pwr_w"],
        ms_tune_vac_turbo2_spd_pct=ms_info["ms_tune_vac_turbo2_spd_pct"],
        ms_tune_cal_point_count=ms_info["ms_tune_cal_point_count"],
        ms_tune_expected_mass_min_mz=ms_info["ms_tune_expected_mass_min_mz"],
        ms_tune_expected_mass_max_mz=ms_info["ms_tune_expected_mass_max_mz"],
        ms_tune_resolution_mean=ms_info["ms_tune_resolution_mean"],
        ms_tune_resolution_min=ms_info["ms_tune_resolution_min"],
        ms_tune_resolution_max=ms_info["ms_tune_resolution_max"],
        ms_mode_hint=ms_info["ms_mode_hint"],
        ms_method_parse_status=ms_info["ms_method_parse_status"],
        ms_method_progid=ms_info["ms_method_progid"],
        ms_method_com_class=ms_info["ms_method_com_class"],
        ms_method_type_id=ms_info["ms_method_type_id"],
        ms_method_default_tune_file=ms_info["ms_method_default_tune_file"],
        ms_method_targeted_list_file=ms_info["ms_method_targeted_list_file"],
        ms_method_profile_token=ms_info["ms_method_profile_token"],
        ms_method_model_token=ms_info["ms_method_model_token"],
        ms_method_polarity=ms_info["ms_method_polarity"],
        ms_method_source_type=ms_info["ms_method_source_type"],
        ms_method_mass_range=ms_info["ms_method_mass_range"],
        ms_method_instrument_mode=ms_info["ms_method_instrument_mode"],
        ms_method_acquisition_rate_hz=ms_info["ms_method_acquisition_rate_hz"],
        ms_method_acquisition_time_ms=ms_info["ms_method_acquisition_time_ms"],
        ms_method_cc_gas=ms_info["ms_method_cc_gas"],
        ms_method_acq_holdoff_delay_ms=ms_info["ms_method_acq_holdoff_delay_ms"],
        ms_method_time_segments_json=ms_info["ms_method_time_segments_json"],
        ms_method_postamble_json=ms_info["ms_method_postamble_json"],
        ms_embedded_xml_count=ms_info["ms_embedded_xml_count"],
        ms_embedded_xml_headers=ms_info["ms_embedded_xml_headers"],
        ms_embedded_xml_payload_json=ms_info["ms_embedded_xml_payload_json"],
        ms_bin_candidate_parse_status=ms_info["ms_bin_candidate_parse_status"],
        ms_bin_gap_lengths=ms_info["ms_bin_gap_lengths"],
        ms_bin_tagged_double_count=ms_info["ms_bin_tagged_double_count"],
        ms_bin_tagged_double_values=ms_info["ms_bin_tagged_double_values"],
        ms_bin_method_extraction_note=ms_info["ms_bin_method_extraction_note"],
        ms_bin_method_gas_temp_C=ms_info["ms_bin_method_gas_temp_C"],
        ms_bin_method_drying_gas_L_min=ms_info["ms_bin_method_drying_gas_L_min"],
        ms_bin_method_nebulizer_psi=ms_info["ms_bin_method_nebulizer_psi"],
        ms_bin_method_nebulizer_source=ms_info["ms_bin_method_nebulizer_source"],
        ms_bin_method_sheath_gas_temp_C=ms_info["ms_bin_method_sheath_gas_temp_C"],
        ms_bin_method_sheath_gas_flow_L_min=ms_info["ms_bin_method_sheath_gas_flow_L_min"],
        ms_bin_method_vcap_V=ms_info["ms_bin_method_vcap_V"],
        ms_bin_method_nozzle_voltage_V=ms_info["ms_bin_method_nozzle_voltage_V"],
        ms_bin_method_fragmentor_V=ms_info["ms_bin_method_fragmentor_V"],
        ms_bin_method_skimmer_V=ms_info["ms_bin_method_skimmer_V"],
        ms_bin_method_oct1_rf_vpp_V=ms_info["ms_bin_method_oct1_rf_vpp_V"],
        ms_tune_positive_json=ms_info["ms_tune_positive_json"],
        ms_tune_negative_json=ms_info["ms_tune_negative_json"],
        ms_tune_positive_count=ms_info["ms_tune_positive_count"],
        ms_tune_negative_count=ms_info["ms_tune_negative_count"],
        ms_tune_positive_unique_hash_count=ms_info["ms_tune_positive_unique_hash_count"],
        ms_tune_negative_unique_hash_count=ms_info["ms_tune_negative_unique_hash_count"],
        ms_tune_positive_selected_hash=ms_info["ms_tune_positive_selected_hash"],
        ms_tune_negative_selected_hash=ms_info["ms_tune_negative_selected_hash"],
        ms_tune_positive_all_hashes=ms_info["ms_tune_positive_all_hashes"],
        ms_tune_negative_all_hashes=ms_info["ms_tune_negative_all_hashes"],
        ms_tune_positive_hash_consistency=ms_info["ms_tune_positive_hash_consistency"],
        ms_tune_negative_hash_consistency=ms_info["ms_tune_negative_hash_consistency"],
    )


def build_gradient_traces(df: pd.DataFrame) -> list[dict[str, Any]]:
    traces: list[dict[str, Any]] = []
    for _, row in df.iterrows():
        try:
            profile = json.loads(row["gradient_profile_json"])
        except (TypeError, json.JSONDecodeError):
            continue
        if not profile:
            continue
        x = [point["time_min"] for point in profile]
        y = [point["percent_b"] for point in profile]
        trace_color = PLOTLY_COLORWAY[len(traces) % len(PLOTLY_COLORWAY)]
        traces.append(
            {
                "x": x,
                "y": y,
                "mode": "lines+markers",
                "line": {"shape": "linear", "color": trace_color},
                "marker": {"color": trace_color},
                "name": row["method_dir"],
                "type": "scatter",
            }
        )
    return traces


def build_method_time_segment_start_map(df: pd.DataFrame) -> dict[str, list[float]]:
    result: dict[str, list[float]] = {}
    if "ms_method_time_segments_json" not in df.columns:
        return result

    for _, row in df.iterrows():
        method_name = str(row.get("method_dir") or "").strip()
        if not method_name:
            continue

        raw = row.get("ms_method_time_segments_json")
        if not isinstance(raw, str) or not raw.strip():
            continue

        try:
            payload = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if not isinstance(payload, list):
            continue

        starts: list[float] = []
        for item in payload:
            if not isinstance(item, dict):
                continue
            try:
                start_time = float(item.get("start_time_min"))
            except (TypeError, ValueError):
                continue
            starts.append(start_time)

        if starts:
            result[method_name] = starts

    return result


def build_method_time_segment_expt_count_map(df: pd.DataFrame) -> dict[str, list[int]]:
    result: dict[str, list[int]] = {}
    if "ms_method_time_segments_json" not in df.columns:
        return result

    for _, row in df.iterrows():
        method_name = str(row.get("method_dir") or "").strip()
        if not method_name:
            continue

        raw = row.get("ms_method_time_segments_json")
        if not isinstance(raw, str) or not raw.strip():
            continue

        try:
            payload = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if not isinstance(payload, list):
            continue

        expt_counts: list[int] = []
        for item in payload:
            if not isinstance(item, dict):
                continue
            raw_count = item.get("n_experimental_segments")
            if raw_count is None:
                continue
            try:
                expt_count = int(raw_count)
            except (TypeError, ValueError):
                continue
            expt_counts.append(expt_count)

        if expt_counts:
            result[method_name] = expt_counts

    return result


def _parse_msmethod_segment_payload(raw: Any) -> list[dict[str, Any]]:
    if not isinstance(raw, str) or not raw.strip():
        return []
    try:
        payload = json.loads(raw)
    except json.JSONDecodeError:
        return []
    if not isinstance(payload, list):
        return []
    return [item for item in payload if isinstance(item, dict)]


def _looks_like_xml_text(text: str) -> bool:
    return bool(MSMETHOD_XML_PREFIX_PATTERN.match(text))


def _is_msmethod_xml_like_value(value: Any) -> bool:
    if isinstance(value, str):
        return _looks_like_xml_text(value)
    if isinstance(value, dict) and value.get("type") == "text_preview":
        preview = str(value.get("preview") or "")
        return _looks_like_xml_text(preview)
    return False


def _format_array_scalar_for_display(value: Any) -> str:
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, int):
        return str(value)
    if isinstance(value, float):
        if value.is_integer():
            return str(int(value))
        return f"{value:.12g}"
    return str(value)


def _is_negative_one_like(value: Any, tol: float = 1e-9) -> bool:
    if isinstance(value, bool):
        return False
    if isinstance(value, (int, float)):
        return abs(float(value) + 1.0) <= tol
    try:
        return abs(float(str(value).strip()) + 1.0) <= tol
    except (TypeError, ValueError):
        return False


def _format_msmethod_array_value_for_display(
    array_payload: dict[str, Any],
    keep_only_first_col_negative_one: bool = False,
) -> str:
    rank_raw = array_payload.get("rank")
    dims_raw = array_payload.get("dims")
    values_raw = array_payload.get("values")
    n_values = array_payload.get("n_values")

    try:
        rank = int(rank_raw)
    except (TypeError, ValueError):
        raise AssertionError(f"MSMETHOD array has invalid rank: {rank_raw!r}") from None

    if rank >= 3:
        raise AssertionError(
            "MSMETHOD array rank>=3 is not supported yet "
            f"(rank={rank}, dims={dims_raw})"
        )
    if rank <= 0:
        return ""

    if not isinstance(dims_raw, list):
        raise AssertionError(f"MSMETHOD array has invalid dims payload: {dims_raw!r}")
    dims: list[int] = []
    for dim in dims_raw:
        try:
            dim_i = int(dim)
        except (TypeError, ValueError):
            raise AssertionError(
                f"MSMETHOD array has invalid dim value: {dim!r} (dims={dims_raw!r})"
            ) from None
        if dim_i < 0:
            raise AssertionError(f"MSMETHOD array has negative dimension: {dim_i}")
        dims.append(dim_i)

    if len(dims) != rank:
        raise AssertionError(
            "MSMETHOD array rank/dims mismatch "
            f"(rank={rank}, dims={dims!r}, raw_dims={dims_raw!r})"
        )

    if not isinstance(values_raw, list):
        raise AssertionError(f"MSMETHOD array has invalid values payload: {values_raw!r}")
    values = values_raw

    if isinstance(n_values, int) and n_values != len(values):
        raise AssertionError(
            "MSMETHOD array n_values mismatch "
            f"(n_values={n_values}, actual={len(values)})"
        )

    expected_count = 1
    for dim in dims:
        expected_count *= dim
    if expected_count != len(values):
        raise AssertionError(
            "MSMETHOD array value count mismatch "
            f"(expected={expected_count}, actual={len(values)}, dims={dims})"
        )

    if rank == 1:
        return ", ".join(_format_array_scalar_for_display(v) for v in values)

    # rank == 2
    rows, cols = dims
    if rows == 0 or cols == 0:
        return ""
    lines: list[str] = []
    for row_idx in range(rows):
        # MSMETHOD array payload looks column-major for rank=2.
        row_values = [values[row_idx + col_idx * rows] for col_idx in range(cols)]
        if keep_only_first_col_negative_one:
            if not row_values or not _is_negative_one_like(row_values[0]):
                continue
            display_values = row_values[1:]
        else:
            display_values = row_values
        lines.append(", ".join(_format_array_scalar_for_display(v) for v in display_values))
    return "\n".join(lines)


def _normalize_polarity_code(value: Any) -> int | None:
    if isinstance(value, bool):
        return 1 if value else 0
    if isinstance(value, int):
        return value
    if isinstance(value, float):
        if value.is_integer():
            return int(value)
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        if re.fullmatch(r"[+-]?\d+", text):
            try:
                return int(text)
            except ValueError:
                return None
    return None


def _format_msmethod_primary_value(
    value: Any,
    field_idx: int | None = None,
    keep_only_first_col_negative_one: bool = False,
    map_polarity_zero_one: bool = False,
    apply_primary_field_rules: bool = False,
) -> str:
    if (
        apply_primary_field_rules
        and field_idx in MSMETHOD_CALIBRATION_XML_FIELD_INDEXES
        and _is_msmethod_xml_like_value(value)
    ):
        return MSMETHOD_CALIBRATION_XML_CELL_TEXT
    if apply_primary_field_rules and map_polarity_zero_one:
        code = _normalize_polarity_code(value)
        if code == 0:
            return "positive"
        if code == 1:
            return "negative"
    if apply_primary_field_rules and field_idx in MSMETHOD_ZERO_ONE_BOOL_FIELD_INDEXES:
        code = _normalize_polarity_code(value)
        if code == 0:
            return "false"
        if code == 1:
            return "true"
    if apply_primary_field_rules and field_idx in MSMETHOD_ENUM_BY_FIELD_INDEX:
        enum_map = MSMETHOD_ENUM_BY_FIELD_INDEX[field_idx]
        code = _normalize_polarity_code(value)
        if code in enum_map:
            return enum_map[code]
        if field_idx in MSMETHOD_STRICT_ENUM_FIELD_INDEXES:
            raise AssertionError(
                f"MSMETHOD strict enum field {field_idx} has invalid value: {value!r} "
                f"(normalized={code!r}, allowed={sorted(enum_map)})"
            )
    if apply_primary_field_rules and field_idx in MSMETHOD_MASS_RANGE_TRIPLET_FIELD_INDEXES:
        if (
            isinstance(value, dict)
            and value.get("type") == "array"
            and value.get("rank") == 1
            and value.get("dims") == [3]
            and isinstance(value.get("values"), list)
            and len(value["values"]) == 3
        ):
            return _format_array_scalar_for_display(value["values"][1])
    if value is None:
        return ""
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, int):
        return str(value)
    if isinstance(value, float):
        if not value.is_integer():
            return f"{value:.12g}"
        return str(int(value))
    if isinstance(value, dict):
        if value.get("type") == "array":
            return _format_msmethod_array_value_for_display(
                value,
                keep_only_first_col_negative_one=keep_only_first_col_negative_one,
            )
        if value.get("type") == "text_preview":
            preview = str(value.get("preview") or "")
            length = value.get("length")
            return f"{preview}... [len={length}]"
        try:
            return json.dumps(value, ensure_ascii=False)
        except (TypeError, ValueError):
            return str(value)
    if isinstance(value, list):
        try:
            return json.dumps(value, ensure_ascii=False)
        except (TypeError, ValueError):
            return str(value)
    return str(value)


def _extract_msmethod_primary_values_for_selection(
    segment_payload: list[dict[str, Any]],
    time_segment_index_1based: int,
    expt_segment_index_1based: int,
    n_fields: int,
) -> list[str]:
    out = [""] * n_fields
    if time_segment_index_1based <= 0 or expt_segment_index_1based <= 0:
        return out

    segment_index = time_segment_index_1based - 1
    expt_index = expt_segment_index_1based - 1
    if segment_index < 0 or segment_index >= len(segment_payload):
        return out

    segment = segment_payload[segment_index]
    raw_fields = segment.get("primary_unknown_fields")
    if not isinstance(raw_fields, list):
        return out

    for field_idx in range(min(n_fields, len(raw_fields))):
        field = raw_fields[field_idx]
        if not isinstance(field, dict):
            continue
        raw_values = field.get("values")
        if not isinstance(raw_values, list):
            continue
        if expt_index < 0 or expt_index >= len(raw_values):
            continue
        out[field_idx] = _format_msmethod_primary_value(
            raw_values[expt_index],
            field_idx=field_idx,
            map_polarity_zero_one=field_idx in MSMETHOD_PRIMARY_POLARITY_FIELD_INDEXES,
            apply_primary_field_rules=True,
        )
    return out


def _extract_msmethod_primary_polarity_for_selection(
    segment_payload: list[dict[str, Any]],
    time_segment_index_1based: int,
    expt_segment_index_1based: int,
) -> str | None:
    if time_segment_index_1based <= 0 or expt_segment_index_1based <= 0:
        return None

    segment_index = time_segment_index_1based - 1
    expt_index = expt_segment_index_1based - 1
    if segment_index < 0 or segment_index >= len(segment_payload):
        return None

    segment = segment_payload[segment_index]
    raw_fields = segment.get("primary_unknown_fields")
    if not isinstance(raw_fields, list):
        return None

    for field_idx in sorted(MSMETHOD_PRIMARY_POLARITY_FIELD_INDEXES):
        if field_idx < 0 or field_idx >= len(raw_fields):
            continue
        field = raw_fields[field_idx]
        if not isinstance(field, dict):
            continue
        raw_values = field.get("values")
        if not isinstance(raw_values, list):
            continue
        if expt_index < 0 or expt_index >= len(raw_values):
            continue
        code = _normalize_polarity_code(raw_values[expt_index])
        if code == 0:
            return "positive"
        if code == 1:
            return "negative"
    return None


def _extract_msmethod_extension_values_for_selection(
    segment_payload: list[dict[str, Any]],
    time_segment_index_1based: int,
    n_fields: int,
) -> list[str]:
    out = [""] * n_fields
    if time_segment_index_1based <= 0:
        return out

    segment_index = time_segment_index_1based - 1
    if segment_index < 0 or segment_index >= len(segment_payload):
        return out

    segment = segment_payload[segment_index]
    raw_fields = segment.get("extension_unknown_fields")
    if not isinstance(raw_fields, list):
        return out

    for field_idx in range(min(n_fields, len(raw_fields))):
        field = raw_fields[field_idx]
        if not isinstance(field, dict):
            continue
        raw_values = field.get("values")
        if not isinstance(raw_values, list) or not raw_values:
            continue
        out[field_idx] = _format_msmethod_primary_value(raw_values[0], field_idx=field_idx)
    return out


def _parse_msmethod_postamble_payload(raw: Any) -> list[dict[str, Any]]:
    if not isinstance(raw, str) or not raw.strip():
        return []
    try:
        payload = json.loads(raw)
    except json.JSONDecodeError:
        return []
    if not isinstance(payload, list):
        return []
    return [item for item in payload if isinstance(item, dict)]


def _extract_msmethod_postamble_values(
    postamble_payload: list[dict[str, Any]],
    n_fields: int,
    ref_mass_field_indexes: set[int] | frozenset[int] | None = None,
) -> list[str]:
    if ref_mass_field_indexes is None:
        ref_mass_field_indexes = MSMETHOD_REF_MASS_FIELD_INDEXES
    out = [""] * n_fields
    for field_idx in range(min(n_fields, len(postamble_payload))):
        field = postamble_payload[field_idx]
        if not isinstance(field, dict):
            continue
        raw_values = field.get("values")
        if not isinstance(raw_values, list) or not raw_values:
            continue
        out[field_idx] = _format_msmethod_primary_value(
            raw_values[0],
            field_idx=field_idx,
            keep_only_first_col_negative_one=field_idx in ref_mass_field_indexes,
        )
    return out


def _infer_max_msmethod_extension_field_count(table_df: pd.DataFrame) -> int:
    max_fields = 0
    for _, row in table_df.iterrows():
        payload = _parse_msmethod_segment_payload(row.get("ms_method_time_segments_json"))
        for segment in payload:
            raw_fields = segment.get("extension_unknown_fields")
            if isinstance(raw_fields, list):
                max_fields = max(max_fields, len(raw_fields))
    return max_fields


def _infer_max_msmethod_postamble_field_count(table_df: pd.DataFrame) -> int:
    max_fields = 0
    for _, row in table_df.iterrows():
        payload = _parse_msmethod_postamble_payload(row.get("ms_method_postamble_json"))
        max_fields = max(max_fields, len(payload))
    return max_fields


def build_method_primary_unknown_field_map(df: pd.DataFrame) -> dict[str, list[dict[str, Any]]]:
    result: dict[str, list[dict[str, Any]]] = {}
    if "ms_method_time_segments_json" not in df.columns:
        return result

    for _, row in df.iterrows():
        method_name = str(row.get("method_dir") or "").strip()
        if not method_name:
            continue
        payload = _parse_msmethod_segment_payload(row.get("ms_method_time_segments_json"))
        if payload:
            result[method_name] = payload
    return result


def build_method_postamble_unknown_field_map(df: pd.DataFrame) -> dict[str, list[dict[str, Any]]]:
    result: dict[str, list[dict[str, Any]]] = {}
    if "ms_method_postamble_json" not in df.columns:
        return result

    for _, row in df.iterrows():
        method_name = str(row.get("method_dir") or "").strip()
        if not method_name:
            continue
        payload = _parse_msmethod_postamble_payload(row.get("ms_method_postamble_json"))
        if payload:
            result[method_name] = payload
    return result


def build_ms_method_details_df(
    table_df: pd.DataFrame,
    n_primary_fields: int = MSMETHOD_PRIMARY_FIELD_COUNT,
) -> pd.DataFrame:
    column_names = ["method_dir", *(msmethod_unknown_field_name(idx) for idx in range(n_primary_fields))]
    rows: list[dict[str, Any]] = []

    for _, row in table_df.iterrows():
        method_name = str(row.get("method_dir") or "").strip()
        if not method_name:
            continue

        payload = _parse_msmethod_segment_payload(row.get("ms_method_time_segments_json"))
        primary_values = _extract_msmethod_primary_values_for_selection(
            payload,
            time_segment_index_1based=1,
            expt_segment_index_1based=1,
            n_fields=n_primary_fields,
        )
        row_payload: dict[str, Any] = {"method_dir": method_name}
        for field_idx, value in enumerate(primary_values):
            row_payload[msmethod_unknown_field_name(field_idx)] = value
        rows.append(row_payload)

    return pd.DataFrame(rows, columns=column_names)


def build_ms_method_details_extension_df(
    table_df: pd.DataFrame,
    n_extension_fields: int | None = None,
) -> pd.DataFrame:
    if n_extension_fields is None:
        n_extension_fields = _infer_max_msmethod_extension_field_count(table_df)
    n_extension_fields = max(0, n_extension_fields)

    column_names = [
        "method_dir",
        *(msmethod_extension_unknown_field_name(idx) for idx in range(n_extension_fields)),
    ]
    rows: list[dict[str, Any]] = []

    for _, row in table_df.iterrows():
        method_name = str(row.get("method_dir") or "").strip()
        if not method_name:
            continue

        payload = _parse_msmethod_segment_payload(row.get("ms_method_time_segments_json"))
        extension_values = _extract_msmethod_extension_values_for_selection(
            payload,
            time_segment_index_1based=1,
            n_fields=n_extension_fields,
        )
        row_payload: dict[str, Any] = {"method_dir": method_name}
        for field_idx, value in enumerate(extension_values):
            row_payload[msmethod_extension_unknown_field_name(field_idx)] = value
        rows.append(row_payload)

    return pd.DataFrame(rows, columns=column_names)


def build_ms_method_details_trailing_df(
    table_df: pd.DataFrame,
    n_trailing_fields: int | None = None,
) -> pd.DataFrame:
    if n_trailing_fields is None:
        n_trailing_fields = _infer_max_msmethod_postamble_field_count(table_df)
    n_trailing_fields = max(0, n_trailing_fields)

    visible_slot_indexes = [0, 1, 2]
    for generic_slot_idx in range(MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS):
        if generic_slot_idx in MSMETHOD_TRAILING_HIDDEN_GENERIC_SLOT_INDEXES:
            continue
        visible_slot_indexes.append(MSMETHOD_TRAILING_SPECIAL_WRAP_COLUMNS + generic_slot_idx)

    column_names = [
        "method_dir",
        *(msmethod_trailing_wrapped_column_name(slot_idx) for slot_idx in visible_slot_indexes),
    ]
    rows: list[dict[str, Any]] = []

    for _, row in table_df.iterrows():
        method_name = str(row.get("method_dir") or "").strip()
        if not method_name:
            continue

        payload = _parse_msmethod_postamble_payload(row.get("ms_method_postamble_json"))
        method_n_trailing_fields = len(payload)
        if n_trailing_fields is not None:
            method_n_trailing_fields = min(method_n_trailing_fields, n_trailing_fields)
        method_n_trailing_fields = max(0, method_n_trailing_fields)
        ref_pos_index, ref_neg_index = msmethod_ref_mass_indexes_for_n_fields(
            method_n_trailing_fields
        )
        ref_mass_field_indexes = {
            idx for idx in (ref_pos_index, ref_neg_index) if idx is not None
        }
        trailing_values = _extract_msmethod_postamble_values(
            payload,
            n_fields=method_n_trailing_fields,
            ref_mass_field_indexes=ref_mass_field_indexes,
        )
        ref_pos_value = (
            trailing_values[ref_pos_index]
            if ref_pos_index is not None and 0 <= ref_pos_index < len(trailing_values)
            else ""
        )
        ref_neg_value = (
            trailing_values[ref_neg_index]
            if ref_neg_index is not None and 0 <= ref_neg_index < len(trailing_values)
            else ""
        )

        generic_values = [
            trailing_values[field_idx]
            for field_idx in range(method_n_trailing_fields)
            if field_idx not in ref_mass_field_indexes
        ]
        generic_rows = _build_msmethod_trailing_generic_rows(generic_values)

        for row_index, generic_row_values in enumerate(generic_rows):
            row_payload: dict[str, Any] = {"method_dir": method_name}
            ref_pos_col_name = msmethod_trailing_wrapped_column_name(0)
            ref_neg_col_name = msmethod_trailing_wrapped_column_name(1)
            row_payload[ref_pos_col_name] = (
                format_msmethod_trailing_wrapped_cell(
                    ref_pos_value,
                    column_name=ref_pos_col_name,
                )
                if row_index == 0
                else ""
            )
            row_payload[ref_neg_col_name] = (
                format_msmethod_trailing_wrapped_cell(
                    ref_neg_value,
                    column_name=ref_neg_col_name,
                )
                if row_index == 0
                else ""
            )
            row_payload[msmethod_trailing_wrapped_column_name(2)] = (
                msmethod_trailing_type_label_from_generic_row(generic_row_values)
            )
            for generic_slot_idx in range(MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS):
                if generic_slot_idx in MSMETHOD_TRAILING_HIDDEN_GENERIC_SLOT_INDEXES:
                    continue
                col_name = msmethod_trailing_wrapped_column_name(
                    MSMETHOD_TRAILING_SPECIAL_WRAP_COLUMNS + generic_slot_idx
                )
                value = (
                    generic_row_values[generic_slot_idx]
                    if generic_slot_idx < len(generic_row_values)
                    else ""
                )
                row_payload[col_name] = format_msmethod_trailing_wrapped_cell(
                    value,
                    column_name=col_name,
                )
            rows.append(row_payload)

    return pd.DataFrame(rows, columns=column_names)


def build_tune_table_df(table_df: pd.DataFrame, polarity: str) -> pd.DataFrame:
    json_col = f"ms_tune_{polarity}_json"
    # Keep this table focused on XML-derived fields only.
    rows: list[dict[str, Any]] = []
    extra_columns: list[str] = []
    extra_column_set: set[str] = set()

    for _, row in table_df.iterrows():
        out: dict[str, Any] = {"method_dir": row.get("method_dir")}
        raw_json = row.get(json_col)
        has_json_payload = isinstance(raw_json, str) and bool(raw_json.strip())

        # Keep all method rows. Methods without XML stay blank in this section.
        if has_json_payload:
            try:
                parsed = json.loads(raw_json)
                if isinstance(parsed, dict):
                    out.update(parsed)
                    for key in parsed:
                        if key == "method_dir":
                            continue
                        if key not in extra_column_set:
                            extra_column_set.add(key)
                            extra_columns.append(key)
            except json.JSONDecodeError:
                # Keep cell values blank on JSON parse failure.
                pass
        rows.append(out)

    if not rows:
        return pd.DataFrame(columns=["method_dir"])

    columns = ["method_dir", *extra_columns]
    out_df = pd.DataFrame(rows)
    for col in columns:
        if col not in out_df.columns:
            out_df[col] = None
    return out_df[columns].fillna("")


def build_tune_warning_html(table_df: pd.DataFrame, polarity: str) -> str:
    consistency_col = f"ms_tune_{polarity}_hash_consistency"
    if consistency_col not in table_df.columns:
        return ""

    mismatch_rows = table_df[table_df[consistency_col] == "mismatch"]
    if mismatch_rows.empty:
        return ""

    method_names = ", ".join(sorted(mismatch_rows["method_dir"].astype(str).tolist()))
    polarity_label = polarity.capitalize()
    return (
        f'<div class="warning-box">'
        f"Warning: {polarity_label} tune reports are not identical across repeats for: {method_names}. "
        f"Showing the first report for each method."
        f"</div>"
    )


def build_tune_section_heading(table_df: pd.DataFrame, polarity: str) -> str:
    json_col = f"ms_tune_{polarity}_json"
    polarity_label = polarity.capitalize()
    if json_col not in table_df.columns:
        return f"MS XML ({polarity_label})"

    for raw in table_df[json_col]:
        if not isinstance(raw, str) or not raw.strip():
            continue
        try:
            payload = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if not isinstance(payload, dict):
            continue
        base = (
            payload.get("ms_tune_report_title")
            or payload.get("ms_tune_report_type")
            or payload.get("ms_instrument_mode")
            or payload.get("ms_tune_data_path")
        )
        if isinstance(base, str) and base.strip():
            base = base.strip()
            if polarity_label.lower() not in base.lower():
                return f"{base} ({polarity_label})"
            return base

    return f"MS XML ({polarity_label})"


def format_display_column_name(column_name: str) -> str:
    if column_name == "method_dir":
        return column_name
    if column_name.startswith("unknown_"):
        return column_name
    if column_name in MSMETHOD_UNKNOWN_FIELD_NAME_OVERRIDES.values():
        return column_name
    if column_name in MSMETHOD_TRAILING_UNKNOWN_FIELD_NAME_OVERRIDES.values():
        return column_name

    suffix_units: tuple[tuple[str, str], ...] = (
        ("_mL_min", "mL/min"),
        ("_L_min", "L/min"),
        ("_vpp_V", "Vpp"),
        ("_mV", "mV"),
        ("_uL", "uL"),
        ("_amu", "amu"),
        ("_torr", "Torr"),
        ("_bar", "bar"),
        ("_psi", "psi"),
        ("_Hz", "Hz"),
        ("_ms", "ms"),
        ("_min", "min"),
        ("_mz", "m/z"),
        ("_pct", "%"),
        ("_C", "ºC"),
        ("_V", "V"),
        ("_s", "s"),
    )
    for suffix, unit in suffix_units:
        if column_name.endswith(suffix):
            base = column_name[: -len(suffix)].rstrip("_")
            if not base:
                return column_name
            return f"{base} [{unit}]"
    return column_name


def with_display_column_names(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns={col: format_display_column_name(col) for col in df.columns})


def msmethod_primary_column_category_name(column_name: str) -> str:
    if column_name == "method_dir":
        return MSMETHOD_PRIMARY_METHOD_CATEGORY_NAME
    for category_name, category_columns in MSMETHOD_PRIMARY_COLUMN_CATEGORY_RULES:
        if column_name in category_columns:
            return category_name
    return MSMETHOD_PRIMARY_OTHER_CATEGORY_NAME


def with_msmethod_primary_grouped_display_columns(df: pd.DataFrame) -> pd.DataFrame:
    columns = list(df.columns)
    category_by_column = {col: msmethod_primary_column_category_name(col) for col in columns}

    ordered_columns: list[str] = []
    if "method_dir" in columns:
        ordered_columns.append("method_dir")

    # Category/column order is primarily defined by MSMETHOD_PRIMARY_COLUMN_CATEGORY_RULES.
    for category_name, category_columns in MSMETHOD_PRIMARY_COLUMN_CATEGORY_RULES:
        for col in category_columns:
            if col in columns and col not in ordered_columns:
                ordered_columns.append(col)
        # Keep any additional columns in this category that were not listed explicitly.
        ordered_columns.extend(
            col
            for col in columns
            if category_by_column.get(col) == category_name and col not in ordered_columns
        )

    # Other-category columns keep source order after explicitly listed category columns.
    ordered_columns.extend(
        col
        for col in columns
        if category_by_column.get(col) == MSMETHOD_PRIMARY_OTHER_CATEGORY_NAME
        and col not in ordered_columns
    )
    # Fallback for any unmatched/missed column.
    ordered_columns.extend(col for col in columns if col not in ordered_columns)

    grouped_columns = [
        (category_by_column[col], format_display_column_name(col)) for col in ordered_columns
    ]
    out = df[ordered_columns].copy() if not df.empty else df.copy()
    out.columns = pd.MultiIndex.from_tuples(grouped_columns)
    return out


def to_csv(df: pd.DataFrame, output_path: Path, generated_at: str) -> None:
    with output_path.open("w", encoding="utf-8", newline="") as f:
        f.write(f"generated_at,{generated_at},viewer_version,{VIEWER_VERSION}\n")
        df.to_csv(f, index=False)


def to_html(
    df: pd.DataFrame, traces: list[dict[str, Any]], output_path: Path, generated_at: str
) -> None:
    table_df = df.drop(columns=["gradient_profile_json"])
    ms_column_set = {"ms_parse_status", "stg_size_bytes", "stg_md5"}
    ms_column_set.update(col for col in table_df.columns if col.startswith("ms_"))

    lc_columns = [col for col in table_df.columns if col not in ms_column_set]

    lc_table_df = table_df[lc_columns]
    ms_method_details_primary_df = build_ms_method_details_df(table_df)
    ms_method_details_extension_df = build_ms_method_details_extension_df(table_df)
    ms_method_trailing_field_count = _infer_max_msmethod_postamble_field_count(table_df)
    ms_method_details_trailing_df = build_ms_method_details_trailing_df(
        table_df,
        n_trailing_fields=ms_method_trailing_field_count,
    )
    ms_tune_positive_df = build_tune_table_df(table_df, "positive")
    ms_tune_negative_df = build_tune_table_df(table_df, "negative")
    ms_tune_positive_heading = build_tune_section_heading(table_df, "positive")
    ms_tune_negative_heading = build_tune_section_heading(table_df, "negative")

    table_dfs_by_id: dict[str, pd.DataFrame] = {
        "lc_method_table": lc_table_df,
        "ms_method_details_table": ms_method_details_primary_df,
        "ms_method_details_extension_table": ms_method_details_extension_df,
        "ms_method_details_trailing_table": ms_method_details_trailing_df,
        "ms_tune_positive_table": ms_tune_positive_df,
        "ms_tune_negative_table": ms_tune_negative_df,
    }
    display_name_by_table_and_column: dict[str, dict[str, str]] = {
        table_id: {col: format_display_column_name(col) for col in table.columns}
        for table_id, table in table_dfs_by_id.items()
    }

    lc_table_display_df = with_display_column_names(lc_table_df)
    ms_method_details_primary_display_df = with_msmethod_primary_grouped_display_columns(
        ms_method_details_primary_df
    )
    ms_method_details_extension_display_df = with_display_column_names(ms_method_details_extension_df)
    ms_method_details_trailing_display_df = with_display_column_names(ms_method_details_trailing_df)
    ms_tune_positive_display_df = with_display_column_names(ms_tune_positive_df)
    ms_tune_negative_display_df = with_display_column_names(ms_tune_negative_df)

    lc_table_html = lc_table_display_df.to_html(
        index=False,
        escape=True,
        table_id="lc_method_table",
        classes=["method-table"],
    )
    ms_method_details_primary_html = ms_method_details_primary_display_df.to_html(
        index=False,
        escape=True,
        table_id="ms_method_details_table",
        classes=["method-table"],
    )
    ms_method_details_primary_panel_html = f"""
  <details class="panel collapsible-panel" open>
    <summary class="panel-summary"><span class="section-title">MS Method Details (primary)</span></summary>
    <div class="panel-body">
      <div class="table-controls column-controls">
        <div class="controls-title">Visible Columns</div>
        <div id="ms_method_details_column_selector" class="column-selector"></div>
      </div>
      <div class="table-wrap">
        {ms_method_details_primary_html}
      </div>
    </div>
  </details>"""
    ms_method_details_extension_html = ms_method_details_extension_display_df.to_html(
        index=False,
        escape=True,
        table_id="ms_method_details_extension_table",
        classes=["method-table"],
    )
    ms_method_details_extension_panel_html = f"""
  <details class="panel collapsible-panel" open>
    <summary class="panel-summary"><span class="section-title">MS Method Details (extension)</span></summary>
    <div class="panel-body">
      <div class="table-controls column-controls">
        <div class="controls-title">Visible Columns</div>
        <div id="ms_method_details_extension_column_selector" class="column-selector"></div>
      </div>
      <div class="table-wrap">
        {ms_method_details_extension_html}
      </div>
    </div>
  </details>"""
    ms_method_details_trailing_html = ms_method_details_trailing_display_df.to_html(
        index=False,
        escape=True,
        table_id="ms_method_details_trailing_table",
        classes=["method-table"],
    )
    ms_method_details_trailing_panel_html = f"""
  <details class="panel collapsible-panel" open>
    <summary class="panel-summary"><span class="section-title">MS Method Details (trailing)</span></summary>
    <div class="panel-body">
      <div class="table-wrap">
        {ms_method_details_trailing_html}
      </div>
    </div>
  </details>"""
    ms_tune_positive_panel_html = ""
    if not ms_tune_positive_df.empty:
        ms_tune_positive_html = ms_tune_positive_display_df.to_html(
            index=False,
            escape=True,
            table_id="ms_tune_positive_table",
            classes=["method-table"],
        )
        ms_tune_positive_warning_html = build_tune_warning_html(table_df, "positive")
        ms_tune_positive_panel_html = f"""
  <details class="panel collapsible-panel" open>
    <summary class="panel-summary"><span class="section-title">{html.escape(ms_tune_positive_heading)}</span></summary>
    <div class="panel-body">
      {ms_tune_positive_warning_html}
      <div class="table-controls column-controls">
        <div class="controls-title">Visible Columns</div>
        <div id="ms_tune_positive_column_selector" class="column-selector"></div>
      </div>
      <div class="table-wrap">
        {ms_tune_positive_html}
      </div>
    </div>
  </details>"""

    ms_tune_negative_panel_html = ""
    if not ms_tune_negative_df.empty:
        ms_tune_negative_html = ms_tune_negative_display_df.to_html(
            index=False,
            escape=True,
            table_id="ms_tune_negative_table",
            classes=["method-table"],
        )
        ms_tune_negative_warning_html = build_tune_warning_html(table_df, "negative")
        ms_tune_negative_panel_html = f"""
  <details class="panel collapsible-panel" open>
    <summary class="panel-summary"><span class="section-title">{html.escape(ms_tune_negative_heading)}</span></summary>
    <div class="panel-body">
      {ms_tune_negative_warning_html}
      <div class="table-controls column-controls">
        <div class="controls-title">Visible Columns</div>
        <div id="ms_tune_negative_column_selector" class="column-selector"></div>
      </div>
      <div class="table-wrap">
        {ms_tune_negative_html}
      </div>
    </div>
  </details>"""

    traces_json = json.dumps(traces, ensure_ascii=True)
    unchecked_columns_by_table_json = json.dumps(
        {
            table_id: sorted(
                display_name_by_table_and_column.get(table_id, {}).get(column_name, column_name)
                for column_name in columns
            )
            for table_id, columns in DEFAULT_UNCHECKED_BY_TABLE.items()
        },
        ensure_ascii=True,
    )
    method_time_segment_map_json = json.dumps(
        build_method_time_segment_start_map(df), ensure_ascii=True
    )
    method_expt_segment_map_json = json.dumps(
        build_method_time_segment_expt_count_map(df), ensure_ascii=True
    )
    method_primary_unknown_field_map_json = json.dumps(
        build_method_primary_unknown_field_map(df), ensure_ascii=True
    )
    method_postamble_unknown_field_map_json = json.dumps(
        build_method_postamble_unknown_field_map(df), ensure_ascii=True
    )
    ms_method_primary_field_count = max(0, len(ms_method_details_primary_df.columns) - 1)
    ms_method_extension_field_count = max(0, len(ms_method_details_extension_df.columns) - 1)
    ms_method_primary_field_index_by_display_name: dict[str, int] = {}
    for field_idx, column_name in enumerate(ms_method_details_primary_df.columns[1:]):
        display_name = format_display_column_name(column_name)
        if display_name not in ms_method_primary_field_index_by_display_name:
            ms_method_primary_field_index_by_display_name[display_name] = field_idx
    ms_method_primary_field_index_by_display_name_json = json.dumps(
        ms_method_primary_field_index_by_display_name,
        ensure_ascii=True,
    )

    html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>LCMS Method Summary</title>
  <script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
  <style>
    body {{
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif;
      margin: 24px;
      color: #222;
      background: #fafafa;
    }}
    h1 {{
      margin-bottom: 4px;
    }}
    .meta {{
      color: #555;
      margin-bottom: 16px;
    }}
    .panel {{
      background: #fff;
      border: 1px solid #ddd;
      border-radius: 8px;
      padding: 0;
      margin-bottom: 20px;
      box-shadow: 0 1px 3px rgba(0, 0, 0, 0.06);
      overflow: hidden;
    }}
    .panel-summary {{
      list-style: none;
      cursor: pointer;
      display: flex;
      align-items: center;
      justify-content: space-between;
      padding: 12px 16px;
      background: #fff;
      user-select: none;
    }}
    .panel-summary:hover {{
      background: #f7faff;
    }}
    .panel-summary:focus-visible {{
      outline: 2px solid #4e86c4;
      outline-offset: -2px;
    }}
    .panel-summary::-webkit-details-marker {{
      display: none;
    }}
    .panel-summary::after {{
      content: "▾";
      color: #5a6a7a;
      font-size: 14px;
      line-height: 1;
      transition: transform 0.15s ease;
    }}
    .collapsible-panel:not([open]) .panel-summary::after {{
      transform: rotate(-90deg);
    }}
    .panel-body {{
      padding: 0 16px 16px;
    }}
    .section-title {{
      margin: 0;
      font-size: 18px;
    }}
    .column-filter-actions {{
      margin-bottom: 8px;
      display: flex;
      gap: 8px;
      flex-wrap: wrap;
    }}
    .column-filter-actions button {{
      font-size: 12px;
      padding: 4px 8px;
      border: 1px solid #bbb;
      border-radius: 6px;
      background: #fff;
      cursor: pointer;
    }}
    .table-controls {{
      margin-bottom: 12px;
      border: 1px solid #ddd;
      border-radius: 8px;
      background: #f9f9f9;
      padding: 10px 12px;
    }}
    .method-config-controls {{
      margin-bottom: 20px;
    }}
    .method-config-wrap {{
      border: none;
      border-radius: 0;
      background: transparent;
      overflow-x: auto;
    }}
    #method_config_table {{
      width: 100%;
      min-width: 960px;
      border-collapse: separate;
      border-spacing: 0 2px;
      font-size: 12px;
      margin: 0;
      background: transparent;
    }}
    #method_config_table td {{
      border: none;
      padding: 3px 8px;
      text-align: left;
      vertical-align: middle;
      background: #fff;
    }}
    #method_config_table tr td:first-child {{
      border-radius: 8px 0 0 8px;
    }}
    #method_config_table tr td:last-child {{
      border-radius: 0 8px 8px 0;
    }}
    #method_config_table .master-row td {{
      background: #dfeefe;
      font-weight: 600;
    }}
    #method_config_table .method-row td {{
      background: #fbfbfb;
    }}
    #method_config_table .checkbox-cell {{
      width: 56px;
      text-align: center;
    }}
    #method_config_table .method-color-cell {{
      width: 72px;
      text-align: center;
    }}
    #method_config_table .method-color-bar {{
      display: inline-block;
      width: 44px;
      height: 6px;
      border-radius: 999px;
      background: #9aa3ad;
      border: 1px solid rgba(0, 0, 0, 0.08);
      vertical-align: middle;
    }}
    #method_config_table .master-row .method-color-bar {{
      background: #3f73ab;
      border-color: rgba(0, 0, 0, 0.15);
    }}
    #method_config_table #method_master_visible {{
      cursor: pointer;
    }}
    #method_config_table .method-name-cell {{
      width: auto;
      font-weight: 500;
    }}
    #method_config_table .segment-info-cell {{
      width: 220px;
      color: #4e5d6c;
      font-size: 11px;
      white-space: nowrap;
      overflow: hidden;
      text-overflow: ellipsis;
    }}
    #method_config_table .expt-segment-info-cell {{
      width: 180px;
      color: #4e5d6c;
      font-size: 11px;
      white-space: nowrap;
      overflow: hidden;
      text-overflow: ellipsis;
    }}
    #method_config_table .master-segment-info-cell {{
      color: #2f4f76;
      font-weight: 600;
    }}
    #method_config_table .select-cell {{
      width: 180px;
    }}
    #method_config_table select {{
      width: 100%;
      font-size: 12px;
      border: 1px solid #bfc6ce;
      border-radius: 6px;
      padding: 1px 6px;
      background: #fff;
      min-height: 22px;
    }}
    #method_config_table #method_master_time,
    #method_config_table #method_master_experimental {{
      background: #cfe4ff;
      border-color: #4e86c4;
      color: inherit;
      font-weight: 400;
      font-family: inherit;
    }}
    #method_config_table #method_master_time:focus,
    #method_config_table #method_master_experimental:focus {{
      outline: 2px solid #4e86c4;
      outline-offset: 1px;
    }}
    #method_config_table .majority-outlier {{
      color: #c62828 !important;
      font-weight: 600;
    }}
    #method_config_table select.majority-outlier {{
      border-color: #d84343;
      box-shadow: inset 0 0 0 1px rgba(216, 67, 67, 0.18);
    }}
    .column-controls {{
      display: grid;
      grid-template-columns: 1fr auto;
      grid-template-areas:
        "title actions"
        "selector selector";
      column-gap: 10px;
      row-gap: 8px;
      align-items: center;
    }}
    .controls-title {{
      font-size: 12px;
      font-weight: 600;
      margin-bottom: 8px;
      color: #444;
    }}
    .column-controls .controls-title {{
      grid-area: title;
      margin-bottom: 0;
    }}
    .column-selector {{
      display: flex;
      flex-wrap: wrap;
      gap: 8px 12px;
      max-height: 140px;
      overflow-y: auto;
      padding-right: 4px;
    }}
    .column-controls .column-selector {{
      grid-area: selector;
    }}
    .column-filter-actions {{
      grid-area: actions;
      margin-bottom: 0;
      justify-content: flex-end;
    }}
    .column-selector label {{
      display: inline-flex;
      align-items: center;
      font-size: 12px;
      color: #333;
      gap: 4px;
      white-space: nowrap;
    }}
    .column-selector-group {{
      display: block;
      border: 1px solid #dde3ea;
      border-radius: 8px;
      padding: 6px 8px;
      background: #fff;
      min-width: 220px;
    }}
    .column-selector-group-title {{
      display: flex;
      align-items: center;
      gap: 8px;
      margin-bottom: 6px;
      padding-bottom: 4px;
      border-bottom: 1px dashed #e3e8ef;
    }}
    .column-selector-group-title label {{
      font-weight: 600;
      color: #2f4f76;
    }}
    .column-selector-group-columns {{
      display: flex;
      flex-wrap: wrap;
      gap: 6px 10px;
    }}
    .column-selector-group-columns label {{
      font-size: 12px;
      font-weight: 400;
      color: #333;
    }}
    #ms_method_details_table thead tr:first-child th {{
      background: #e9eef5;
      top: 0;
      z-index: 4;
    }}
    #ms_method_details_table thead tr:last-child th {{
      top: var(--ms-method-primary-header-offset, 28px);
      z-index: 4;
    }}
    #ms_method_details_table thead tr:first-child th:first-child,
    #ms_method_details_table thead tr:last-child th:first-child {{
      z-index: 6;
    }}
    @media (max-width: 860px) {{
      #method_config_table {{
        min-width: 0;
      }}
      #method_config_table .select-cell {{
        width: 140px;
      }}
      #method_config_table .segment-info-cell {{
        width: 170px;
      }}
      #method_config_table .expt-segment-info-cell {{
        width: 130px;
      }}
    }}
    .table-wrap {{
      overflow-x: auto;
      border: 1px solid #ddd;
      border-radius: 8px;
      background: #fff;
    }}
    .warning-box {{
      margin: 0 0 12px 0;
      border: 1px solid #f0b429;
      background: #fff7e6;
      color: #6b4e00;
      border-radius: 8px;
      padding: 8px 10px;
      font-size: 12px;
    }}
    .empty-state {{
      border: 1px dashed #c9c9c9;
      border-radius: 8px;
      background: #fafafa;
      color: #666;
      font-size: 12px;
      padding: 10px 12px;
    }}
    table {{
      width: max-content;
      min-width: 100%;
      border-collapse: collapse;
      font-size: 12px;
    }}
    th, td {{
      border: 1px solid #ddd;
      padding: 6px 8px;
      text-align: left;
      vertical-align: top;
    }}
    th {{
      background: #f2f2f2;
      position: sticky;
      top: 0;
      z-index: 1;
    }}
    .method-table th:first-child {{
      position: sticky;
      left: 0;
      z-index: 3;
      background: #f2f2f2;
      box-shadow: 2px 0 0 #ddd;
    }}
    .method-table td:first-child {{
      position: sticky;
      left: 0;
      z-index: 2;
      background: #fff;
      box-shadow: 2px 0 0 #ddd;
    }}
    #ms_method_details_trailing_table td:first-child {{
      position: static;
      left: auto;
      z-index: auto;
      box-shadow: none;
    }}
    #ms_method_details_trailing_table th.method-dir-header {{
      position: sticky;
      left: var(--ms-trailing-left-method, 0px);
      z-index: 4;
      background: #f2f2f2;
      box-shadow: 2px 0 0 #ddd;
    }}
    #ms_method_details_trailing_table th.ref-mass-pos-header {{
      position: sticky;
      left: var(--ms-trailing-left-ref-pos, 0px);
      z-index: 4;
      background: #f2f2f2;
      box-shadow: 2px 0 0 #ddd;
    }}
    #ms_method_details_trailing_table th.ref-mass-neg-header {{
      position: sticky;
      left: var(--ms-trailing-left-ref-neg, 0px);
      z-index: 4;
      background: #f2f2f2;
      box-shadow: 2px 0 0 #ddd;
    }}
    #ms_method_details_trailing_table td.method-dir-cell {{
      position: sticky;
      left: var(--ms-trailing-left-method, 0px);
      z-index: 3;
      background: #fff;
      box-shadow: 2px 0 0 #ddd;
    }}
    #ms_method_details_trailing_table td.ref-mass-pos-cell {{
      position: sticky;
      left: var(--ms-trailing-left-ref-pos, 0px);
      z-index: 3;
      background: #fff;
      box-shadow: 2px 0 0 #ddd;
    }}
    #ms_method_details_trailing_table td.ref-mass-neg-cell {{
      position: sticky;
      left: var(--ms-trailing-left-ref-neg, 0px);
      z-index: 3;
      background: #fff;
      box-shadow: 2px 0 0 #ddd;
    }}
    #ms_method_details_table td,
    #ms_method_details_extension_table td,
    #ms_method_details_trailing_table td {{
      white-space: pre-line;
    }}
    .method-table td.numeric-diff-outlier {{
      color: #c62828;
      font-weight: 600;
    }}
    .auto-hidden-column {{
      display: none !important;
    }}
  </style>
</head>
<body>
  <h1>LC Method Summary</h1>
  <div class="meta">Generated at: {generated_at} | Version: {VIEWER_VERSION}</div>
  <details class="panel method-config-controls collapsible-panel" open>
    <summary class="panel-summary"><span class="section-title">Method Files</span></summary>
    <div class="panel-body">
      <div class="method-config-wrap">
        <table id="method_config_table">
          <tbody></tbody>
        </table>
      </div>
    </div>
  </details>
  <details class="panel collapsible-panel" open>
    <summary class="panel-summary"><span class="section-title">LC Gradient Summary</span></summary>
    <div class="panel-body">
      <div id="gradient_plot" style="width:100%;height:480px;"></div>
    </div>
  </details>
  <details class="panel collapsible-panel" open>
    <summary class="panel-summary"><span class="section-title">LC Method Details</span></summary>
    <div class="panel-body">
      <div class="table-controls column-controls">
        <div class="controls-title">Visible Columns</div>
        <div id="column_selector" class="column-selector"></div>
      </div>
      <div class="table-wrap">
        {lc_table_html}
      </div>
    </div>
  </details>
  {ms_method_details_primary_panel_html}
  {ms_method_details_extension_panel_html}
  {ms_method_details_trailing_panel_html}
  {ms_tune_positive_panel_html}
  {ms_tune_negative_panel_html}
  <script>
    const traces = {traces_json};
    const defaultUncheckedColumnsByTable = {unchecked_columns_by_table_json};
    const methodTimeSegmentStartsByName = {method_time_segment_map_json};
    const methodExptSegmentCountsByName = {method_expt_segment_map_json};
    const methodPrimaryUnknownFieldsByName = {method_primary_unknown_field_map_json};
    const methodPostambleUnknownFieldsByName = {method_postamble_unknown_field_map_json};
    const methodTraceColorByName = (() => {{
      const colorByName = {{}};
      traces.forEach((trace) => {{
        const methodName = String(trace?.name || "").trim();
        if (!methodName || colorByName[methodName]) {{
          return;
        }}
        const lineColor = trace?.line && typeof trace.line.color === "string" ? trace.line.color.trim() : "";
        const markerColor =
          trace?.marker && typeof trace.marker.color === "string" ? trace.marker.color.trim() : "";
        colorByName[methodName] = lineColor || markerColor || "#9aa3ad";
      }});
      return colorByName;
    }})();
    const BASE_PLOT_SHAPES = [
      {{
        type: "line",
        xref: "paper",
        x0: 0,
        x1: 1,
        yref: "y",
        y0: 0,
        y1: 0,
        line: {{ color: "rgba(120, 120, 120, 0.55)", width: 1 }}
      }},
      {{
        type: "line",
        xref: "paper",
        x0: 0,
        x1: 1,
        yref: "y",
        y0: 100,
        y1: 100,
        line: {{ color: "rgba(120, 120, 120, 0.55)", width: 1 }}
      }}
    ];
    const layout = {{
      title: "%B vs Time",
      xaxis: {{ title: "Time (min)" }},
      yaxis: {{
        title: "%B",
        autorange: true,
      }},
      template: "plotly_white",
      showlegend: false,
      margin: {{ l: 40, r: 40, t: 60, b: 40 }},
      shapes: BASE_PLOT_SHAPES
    }};
    const methodConfigTableBody = document.querySelector("#method_config_table tbody");

    const uniqueMethodsFromTable = () => {{
      const table = document.getElementById("lc_method_table");
      if (!table) return [];
      const names = Array.from(table.querySelectorAll("tbody tr td:first-child"))
        .map((cell) => (cell.textContent || "").trim())
        .filter((name) => name.length > 0);
      return Array.from(new Set(names));
    }};

    const optionValuesByCount = (count) => {{
      const normalizedCount = Number.isFinite(count) ? Math.trunc(count) : 0;
      if (normalizedCount <= 0) {{
        return [];
      }}
      const out = [];
      for (let i = 1; i <= normalizedCount; i += 1) {{
        out.push(String(i));
      }}
      return out;
    }};

    const timeSegmentOptionValuesForMethod = (methodName) => {{
      const starts = methodTimeSegmentStartsByName[methodName];
      if (!Array.isArray(starts) || starts.length <= 0) {{
        return ["1"];
      }}
      const out = [];
      for (let i = 1; i <= starts.length; i += 1) {{
        out.push(String(i));
      }}
      return out;
    }};

    const masterTimeSegmentOptionValues = () => {{
      const counts = Object.values(methodTimeSegmentStartsByName)
        .filter((value) => Array.isArray(value))
        .map((value) => value.length)
        .filter((count) => Number.isFinite(count) && count > 0);
      const maxCount = counts.length ? Math.max(...counts) : 1;
      const out = [];
      for (let i = 1; i <= maxCount; i += 1) {{
        out.push(String(i));
      }}
      return out;
    }};

    const exptSegmentCountForMethodTime = (methodName, timeSegmentValue) => {{
      if (timeSegmentValue === UNSELECTED_SEGMENT_VALUE) {{
        return 0;
      }}
      const counts = methodExptSegmentCountsByName[methodName];
      if (!Array.isArray(counts) || counts.length === 0) {{
        return 1;
      }}
      const timeIndex = Number.parseInt(String(timeSegmentValue || "1"), 10) - 1;
      if (!Number.isFinite(timeIndex) || timeIndex < 0 || timeIndex >= counts.length) {{
        return 1;
      }}
      const rawCount = Number(counts[timeIndex]);
      if (!Number.isFinite(rawCount) || rawCount <= 0) {{
        return 1;
      }}
      return Math.max(1, Math.trunc(rawCount));
    }};

    const exptSegmentOptionValuesForMethod = (methodName, timeSegmentValue) => {{
      const count = exptSegmentCountForMethodTime(methodName, timeSegmentValue);
      return count > 0 ? optionValuesByCount(count) : [];
    }};

    const formatPreviewList = (values, formatter) => {{
      if (!Array.isArray(values) || values.length === 0) {{
        return "n/a";
      }}
      const previewValues = values.slice(0, 6).map((value) => formatter(value)).join(", ");
      const suffix = values.length > 6 ? ", ..." : "";
      return `[${{previewValues}}${{suffix}}]`;
    }};

    const formatTimeSegmentInfo = (methodName) => {{
      const starts = methodTimeSegmentStartsByName[methodName];
      return formatPreviewList(starts, (value) => {{
        const num = Number(value);
        if (!Number.isFinite(num)) return "?";
        return Number.isInteger(num) ? String(num) : num.toFixed(2).replace(/\\.00$/, "");
      }});
    }};

    const formatExptSegmentInfo = (methodName) => {{
      const exptCounts = methodExptSegmentCountsByName[methodName];
      return formatPreviewList(exptCounts, (value) => {{
        const num = Number(value);
        return Number.isFinite(num) ? String(Math.trunc(num)) : "?";
      }});
    }};

    const traceColorForMethod = (methodName) => methodTraceColorByName[methodName] || "#9aa3ad";

    const MASTER_MIXED_VALUE = "__mixed__";
    const UNSELECTED_SEGMENT_VALUE = "__unselected__";
    const UNSELECTED_SEGMENT_LABEL = "—";
    const MSMETHOD_PRIMARY_FIELD_COUNT = {ms_method_primary_field_count};
    const MSMETHOD_EXTENSION_FIELD_COUNT = {ms_method_extension_field_count};
    const MSMETHOD_PRIMARY_FIELD_INDEX_BY_COLUMN_NAME = {ms_method_primary_field_index_by_display_name_json};
    const MSMETHOD_TRAILING_FIELD_COUNT = {ms_method_trailing_field_count};
    const MSMETHOD_TRAILING_WRAP_COLUMNS = {MSMETHOD_TRAILING_WRAP_COLUMNS};
    const MSMETHOD_TRAILING_SPECIAL_WRAP_COLUMNS = {MSMETHOD_TRAILING_SPECIAL_WRAP_COLUMNS};
    const MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS = {MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS};
    const MSMETHOD_TRAILING_HIDDEN_GENERIC_SLOT_INDEXES = new Set({json.dumps(sorted(MSMETHOD_TRAILING_HIDDEN_GENERIC_SLOT_INDEXES), ensure_ascii=True)});
    const MSMETHOD_TRAILING_TYPE_BY_CODE = {json.dumps(MSMETHOD_TRAILING_TYPE_BY_CODE, ensure_ascii=True)};
    const MSMETHOD_TRAILING_EXPT_TYPE_BY_CODE = {json.dumps(MSMETHOD_TRAILING_EXPT_TYPE_BY_CODE, ensure_ascii=True)};
    const MSMETHOD_TRAILING_POLARITY_BY_CODE = {json.dumps(MSMETHOD_TRAILING_POLARITY_BY_CODE, ensure_ascii=True)};
    const MSMETHOD_TRAILING_COMPACTION_RULES = {json.dumps(
        [
            {
                "signature": list(signature),
                "insert_after_n_values": insert_after_n_values,
                "blank_cell_count": blank_cell_count,
            }
            for signature, insert_after_n_values, blank_cell_count in MSMETHOD_TRAILING_COMPACTION_RULES
        ],
        ensure_ascii=True,
    )};
    const MSMETHOD_PRIMARY_POLARITY_FIELD_INDEXES = new Set({json.dumps(sorted(MSMETHOD_PRIMARY_POLARITY_FIELD_INDEXES), ensure_ascii=True)});
    const MSMETHOD_ZERO_ONE_BOOL_FIELD_INDEXES = new Set({json.dumps(sorted(MSMETHOD_ZERO_ONE_BOOL_FIELD_INDEXES), ensure_ascii=True)});
    const MSMETHOD_ENUM_BY_FIELD_INDEX = {json.dumps(MSMETHOD_ENUM_BY_FIELD_INDEX, ensure_ascii=True)};
    const MSMETHOD_STRICT_ENUM_FIELD_INDEXES = new Set({json.dumps(sorted(MSMETHOD_STRICT_ENUM_FIELD_INDEXES), ensure_ascii=True)});
    const MSMETHOD_CALIBRATION_XML_FIELD_INDEXES = new Set({json.dumps(sorted(MSMETHOD_CALIBRATION_XML_FIELD_INDEXES), ensure_ascii=True)});
    const MSMETHOD_MASS_RANGE_TRIPLET_FIELD_INDEXES = new Set({json.dumps(sorted(MSMETHOD_MASS_RANGE_TRIPLET_FIELD_INDEXES), ensure_ascii=True)});
    const MSMETHOD_CALIBRATION_XML_CELL_TEXT = {json.dumps(MSMETHOD_CALIBRATION_XML_CELL_TEXT, ensure_ascii=True)};

    const fillSegmentSelect = (
      selectEl,
      initialValue = "1",
      labelBuilder = (value) => value,
      optionValues = ["1"],
      includeUnselectedOption = false,
      preferUnselectedOnInvalid = false
    ) => {{
      if (!selectEl) return;
      const values = optionValues.length ? optionValues : includeUnselectedOption ? [] : ["1"];
      selectEl.innerHTML = "";
      const selectableValues = [...values];
      if (includeUnselectedOption) {{
        const unselectedOption = document.createElement("option");
        unselectedOption.value = UNSELECTED_SEGMENT_VALUE;
        unselectedOption.textContent = UNSELECTED_SEGMENT_LABEL;
        selectEl.appendChild(unselectedOption);
        selectableValues.push(UNSELECTED_SEGMENT_VALUE);
      }}
      values.forEach((value) => {{
        const option = document.createElement("option");
        option.value = value;
        option.textContent = labelBuilder(value);
        selectEl.appendChild(option);
      }});
      let nextValue = values.length > 0 ? values[0] : includeUnselectedOption ? UNSELECTED_SEGMENT_VALUE : "1";
      if (selectableValues.includes(initialValue)) {{
        nextValue = initialValue;
      }} else if (includeUnselectedOption && preferUnselectedOnInvalid) {{
        nextValue = UNSELECTED_SEGMENT_VALUE;
      }}
      selectEl.value = nextValue;
    }};

    const getMethodConfigRows = () => {{
      if (!methodConfigTableBody) return [];
      return Array.from(methodConfigTableBody.querySelectorAll("tr[data-method]"));
    }};

    const getMethodSegmentSelectionMap = () => {{
      const out = new Map();
      getMethodConfigRows().forEach((row) => {{
        const methodName = row.getAttribute("data-method") || "";
        if (!methodName) {{
          return;
        }}
        const timeSelect = row.querySelector(".method-time-select");
        const exptSelect = row.querySelector(".method-experimental-select");
        out.set(methodName, {{
          time: timeSelect instanceof HTMLSelectElement ? timeSelect.value : "1",
          expt: exptSelect instanceof HTMLSelectElement ? exptSelect.value : "1",
        }});
      }});
      return out;
    }};

    const getRowMethodName = (row) => {{
      if (!(row instanceof HTMLTableRowElement)) {{
        return "";
      }}
      const attrName = String(row.getAttribute("data-method-dir") || "").trim();
      if (attrName) {{
        return attrName;
      }}
      const firstCell = row.querySelector("td:first-child");
      const textName = String(firstCell?.textContent || "").trim();
      if (textName) {{
        row.setAttribute("data-method-dir", textName);
      }}
      return textName;
    }};

    const mergeMethodDirCellsByRowspan = (tableId) => {{
      const table = document.getElementById(tableId);
      if (!(table instanceof HTMLTableElement)) {{
        return;
      }}
      if (table.dataset.methodDirMerged === "1") {{
        return;
      }}
      const rows = Array.from(table.querySelectorAll("tbody tr")).filter(
        (row) => row instanceof HTMLTableRowElement
      );
      if (!rows.length) {{
        return;
      }}

      rows.forEach((row) => {{
        const firstCell = row.querySelector("td:first-child");
        if (!(firstCell instanceof HTMLTableCellElement)) {{
          return;
        }}
        const methodName = String(firstCell.textContent || "").trim();
        if (methodName) {{
          row.setAttribute("data-method-dir", methodName);
        }}
      }});

      let cursor = 0;
      while (cursor < rows.length) {{
        const methodName = getRowMethodName(rows[cursor]);
        if (!methodName) {{
          cursor += 1;
          continue;
        }}
        let groupEnd = cursor + 1;
        while (groupEnd < rows.length && getRowMethodName(rows[groupEnd]) === methodName) {{
          groupEnd += 1;
        }}
        const groupCount = groupEnd - cursor;
        const firstRow = rows[cursor];
        const methodCell = firstRow.children[0];
        const refPosCell = firstRow.children[1];
        const refNegCell = firstRow.children[2];

        if (methodCell instanceof HTMLTableCellElement) {{
          methodCell.classList.add("method-dir-cell");
          methodCell.textContent = methodName;
          methodCell.rowSpan = groupCount;
        }}
        if (refPosCell instanceof HTMLTableCellElement) {{
          refPosCell.classList.add("ref-mass-pos-cell");
          refPosCell.rowSpan = groupCount;
        }}
        if (refNegCell instanceof HTMLTableCellElement) {{
          refNegCell.classList.add("ref-mass-neg-cell");
          refNegCell.rowSpan = groupCount;
        }}

        for (let rowIdx = cursor + 1; rowIdx < groupEnd; rowIdx += 1) {{
          const row = rows[rowIdx];
          for (let removeIdx = 2; removeIdx >= 0; removeIdx -= 1) {{
            const cell = row.children[removeIdx];
            if (cell instanceof HTMLTableCellElement) {{
              cell.remove();
            }}
          }}
        }}
        cursor = groupEnd;
      }}
      table.dataset.methodDirMerged = "1";
    }};

    const updateMsMethodTrailingStickyOffsets = () => {{
      const table = document.getElementById("ms_method_details_trailing_table");
      if (!(table instanceof HTMLTableElement)) {{
        return;
      }}
      const methodHeader = table.querySelector("thead th:nth-child(1)");
      const refPosHeader = table.querySelector("thead th:nth-child(2)");
      const refNegHeader = table.querySelector("thead th:nth-child(3)");

      if (methodHeader instanceof HTMLTableCellElement) {{
        methodHeader.classList.add("method-dir-header");
      }}
      if (refPosHeader instanceof HTMLTableCellElement) {{
        refPosHeader.classList.add("ref-mass-pos-header");
      }}
      if (refNegHeader instanceof HTMLTableCellElement) {{
        refNegHeader.classList.add("ref-mass-neg-header");
      }}

      const methodWidth =
        methodHeader instanceof HTMLElement && methodHeader.style.display !== "none"
          ? methodHeader.getBoundingClientRect().width
          : 0;
      const refPosWidth =
        refPosHeader instanceof HTMLElement && refPosHeader.style.display !== "none"
          ? refPosHeader.getBoundingClientRect().width
          : 0;

      table.style.setProperty("--ms-trailing-left-method", "0px");
      table.style.setProperty("--ms-trailing-left-ref-pos", `${{methodWidth}}px`);
      table.style.setProperty(
        "--ms-trailing-left-ref-neg",
        `${{methodWidth + refPosWidth}}px`
      );
    }};

    const updateMsMethodPrimaryStickyOffsets = () => {{
      const table = document.getElementById("ms_method_details_table");
      if (!(table instanceof HTMLTableElement)) {{
        return;
      }}
      const headerRows = getTableHeaderRows(table);
      if (headerRows.length < 2) {{
        table.style.setProperty("--ms-method-primary-header-offset", "0px");
        return;
      }}
      const firstRow = headerRows[0];
      const firstRowHeight =
        firstRow instanceof HTMLElement ? firstRow.getBoundingClientRect().height : 0;
      table.style.setProperty("--ms-method-primary-header-offset", `${{firstRowHeight}}px`);
    }};

    const getColumnVisibilityMapFromSelector = (selectorId) => {{
      const selector = document.getElementById(selectorId);
      const out = new Map();
      if (!(selector instanceof HTMLElement)) {{
        return out;
      }}
      const labels = Array.from(selector.querySelectorAll("label[data-column-name]"));
      labels.forEach((label) => {{
        const checkbox = label.querySelector("input[type='checkbox']");
        if (!(checkbox instanceof HTMLInputElement)) {{
          return;
        }}
        const name = String(label.getAttribute("data-column-name") || "").trim();
        if (!name) {{
          return;
        }}
        out.set(name, checkbox.checked);
      }});
      return out;
    }};

    const applyColumnVisibilityMapToTable = (tableId, visibilityByName) => {{
      const table = document.getElementById(tableId);
      if (!(table instanceof HTMLTableElement)) {{
        return;
      }}
      const headers = getTableLeafHeaderCells(table);
      headers.forEach((header, idx) => {{
        const name = (header.textContent || "").trim() || `column_${{idx + 1}}`;
        const isVisible =
          name === "method_dir"
            ? true
            : visibilityByName.has(name)
              ? Boolean(visibilityByName.get(name))
              : true;
        const headerCell = headers[idx];
        if (headerCell instanceof HTMLElement) {{
          headerCell.style.display = isVisible ? "" : "none";
        }}
        const nth = idx + 1;
        const bodyCells = table.querySelectorAll(`tbody tr > *:nth-child(${{nth}})`);
        bodyCells.forEach((cell) => {{
          cell.style.display = isVisible ? "" : "none";
        }});
      }});
    }};

    const getTableHeaderRows = (table) => {{
      if (!(table instanceof HTMLTableElement)) {{
        return [];
      }}
      return Array.from(table.querySelectorAll("thead tr")).filter(
        (row) => row instanceof HTMLTableRowElement
      );
    }};

    const getTableLeafHeaderCells = (table) => {{
      const headerRows = getTableHeaderRows(table);
      if (!headerRows.length) {{
        return [];
      }}
      const leafRow = headerRows[headerRows.length - 1];
      return Array.from(leafRow.querySelectorAll("th"));
    }};

    const getTableLeafHeaderNames = (table) => {{
      const leafHeaders = getTableLeafHeaderCells(table);
      return leafHeaders.map(
        (header, idx) => (header.textContent || "").trim() || `column_${{idx + 1}}`
      );
    }};

    const looksLikeXmlText = (text) => /^\\s*(<\\?xml\\b|<[A-Za-z_][\\w:.-]*)/.test(String(text || ""));

    const isMsMethodXmlLikeValue = (value) => {{
      if (typeof value === "string") {{
        return looksLikeXmlText(value);
      }}
      if (value && typeof value === "object" && value.type === "text_preview") {{
        return looksLikeXmlText(value.preview || "");
      }}
      return false;
    }};

    const formatArrayScalarForDisplay = (value) => {{
      if (typeof value === "boolean") {{
        return value ? "true" : "false";
      }}
      if (typeof value === "number") {{
        if (!Number.isFinite(value)) {{
          return "";
        }}
        return Number.isInteger(value) ? String(value) : String(value);
      }}
      return String(value);
    }};

    const isNegativeOneLike = (value) => {{
      if (typeof value === "boolean") {{
        return false;
      }}
      const num = Number(value);
      return Number.isFinite(num) && Math.abs(num + 1) <= 1e-9;
    }};

    const formatMsMethodArrayValueForDisplay = (arrayPayload, keepOnlyFirstColNegativeOne = false) => {{
      const rank = Number.parseInt(String(arrayPayload?.rank), 10);
      const dims = Array.isArray(arrayPayload?.dims) ? arrayPayload.dims.map((dim) => Number.parseInt(String(dim), 10)) : null;
      const values = Array.isArray(arrayPayload?.values) ? arrayPayload.values : null;
      const nValues = Number.parseInt(String(arrayPayload?.n_values), 10);

      if (!Number.isFinite(rank)) {{
        throw new Error(`MSMETHOD array has invalid rank: ${{String(arrayPayload?.rank)}}`);
      }}
      if (rank >= 3) {{
        throw new Error(`MSMETHOD array rank>=3 is not supported yet (rank=${{rank}}, dims=${{JSON.stringify(arrayPayload?.dims)}})`);
      }}
      if (rank <= 0) {{
        return "";
      }}
      if (!Array.isArray(dims)) {{
        throw new Error(`MSMETHOD array has invalid dims payload: ${{JSON.stringify(arrayPayload?.dims)}}`);
      }}
      if (dims.length !== rank || dims.some((dim) => !Number.isFinite(dim) || dim < 0)) {{
        throw new Error(`MSMETHOD array rank/dims mismatch (rank=${{rank}}, dims=${{JSON.stringify(dims)}})`);
      }}
      if (!Array.isArray(values)) {{
        throw new Error(`MSMETHOD array has invalid values payload: ${{JSON.stringify(arrayPayload?.values)}}`);
      }}
      if (Number.isFinite(nValues) && nValues !== values.length) {{
        throw new Error(`MSMETHOD array n_values mismatch (n_values=${{nValues}}, actual=${{values.length}})`);
      }}
      const expectedCount = dims.reduce((acc, dim) => acc * dim, 1);
      if (expectedCount !== values.length) {{
        throw new Error(
          `MSMETHOD array value count mismatch (expected=${{expectedCount}}, actual=${{values.length}}, dims=${{JSON.stringify(dims)}})`
        );
      }}

      if (rank === 1) {{
        return values.map((v) => formatArrayScalarForDisplay(v)).join(", ");
      }}

      const [rows, cols] = dims;
      if (rows === 0 || cols === 0) {{
        return "";
      }}
      const lines = [];
      for (let rowIdx = 0; rowIdx < rows; rowIdx += 1) {{
        // MSMETHOD array payload looks column-major for rank=2.
        const rowValues = [];
        for (let colIdx = 0; colIdx < cols; colIdx += 1) {{
          rowValues.push(values[rowIdx + colIdx * rows]);
        }}
        let displayValues = rowValues;
        if (keepOnlyFirstColNegativeOne) {{
          if (!rowValues.length || !isNegativeOneLike(rowValues[0])) {{
            continue;
          }}
          displayValues = rowValues.slice(1);
        }}
        lines.push(displayValues.map((v) => formatArrayScalarForDisplay(v)).join(", "));
      }}
      return lines.join("\\n");
    }};

    const normalizePolarityCode = (value) => {{
      if (typeof value === "boolean") {{
        return value ? 1 : 0;
      }}
      if (typeof value === "number") {{
        if (!Number.isFinite(value) || !Number.isInteger(value)) {{
          return null;
        }}
        return value;
      }}
      if (typeof value === "string") {{
        const text = value.trim();
        if (!text.length || !/^[+-]?\\d+$/.test(text)) {{
          return null;
        }}
        const parsed = Number.parseInt(text, 10);
        return Number.isFinite(parsed) ? parsed : null;
      }}
      return null;
    }};

    const formatMsMethodPrimaryValue = (
      value,
      fieldIdx = null,
      keepOnlyFirstColNegativeOne = false,
      mapPolarityZeroOne = false,
      applyPrimaryFieldRules = false
    ) => {{
      if (
        applyPrimaryFieldRules &&
        fieldIdx !== null &&
        MSMETHOD_CALIBRATION_XML_FIELD_INDEXES.has(fieldIdx) &&
        isMsMethodXmlLikeValue(value)
      ) {{
        return MSMETHOD_CALIBRATION_XML_CELL_TEXT;
      }}
      if (applyPrimaryFieldRules && mapPolarityZeroOne) {{
        const code = normalizePolarityCode(value);
        if (code === 0) {{
          return "positive";
        }}
        if (code === 1) {{
          return "negative";
        }}
      }}
      if (
        applyPrimaryFieldRules &&
        fieldIdx !== null &&
        MSMETHOD_ZERO_ONE_BOOL_FIELD_INDEXES.has(fieldIdx)
      ) {{
        const code = normalizePolarityCode(value);
        if (code === 0) {{
          return "false";
        }}
        if (code === 1) {{
          return "true";
        }}
      }}
      if (applyPrimaryFieldRules && fieldIdx !== null) {{
        const enumMap = MSMETHOD_ENUM_BY_FIELD_INDEX[String(fieldIdx)];
        if (enumMap && typeof enumMap === "object") {{
          const code = normalizePolarityCode(value);
          if (code !== null) {{
            const label = enumMap[String(code)];
            if (typeof label === "string" && label.length > 0) {{
              return label;
            }}
          }}
          if (MSMETHOD_STRICT_ENUM_FIELD_INDEXES.has(fieldIdx)) {{
            throw new Error(
              `MSMETHOD strict enum field ${{fieldIdx}} has invalid value: ${{JSON.stringify(value)}} `
              + `(normalized=${{String(code)}}, allowed=${{Object.keys(enumMap).join(",")}})`
            );
          }}
        }}
      }}
      if (
        applyPrimaryFieldRules &&
        fieldIdx !== null &&
        MSMETHOD_MASS_RANGE_TRIPLET_FIELD_INDEXES.has(fieldIdx) &&
        value &&
        typeof value === "object" &&
        value.type === "array" &&
        Number(value.rank) === 1 &&
        Array.isArray(value.dims) &&
        value.dims.length === 1 &&
        Number(value.dims[0]) === 3 &&
        Array.isArray(value.values) &&
        value.values.length === 3
      ) {{
        return formatArrayScalarForDisplay(value.values[1]);
      }}
      if (value === null || value === undefined) {{
        return "";
      }}
      if (typeof value === "boolean") {{
        return value ? "true" : "false";
      }}
      if (typeof value === "number") {{
        return Number.isFinite(value) ? String(value) : "";
      }}
      if (typeof value === "string") {{
        return value;
      }}
      if (Array.isArray(value)) {{
        try {{
          return JSON.stringify(value);
        }} catch (_err) {{
          return String(value);
        }}
      }}
      if (typeof value === "object") {{
        if (value.type === "array") {{
          return formatMsMethodArrayValueForDisplay(value, keepOnlyFirstColNegativeOne);
        }}
        if (value.type === "text_preview") {{
          const preview = String(value.preview || "");
          return `${{preview}}... [len=${{value.length}}]`;
        }}
        try {{
          return JSON.stringify(value);
        }} catch (_err) {{
          return String(value);
        }}
      }}
      return String(value);
    }};

    const extractMsMethodPrimaryValues = (methodName, timeSegmentValue, exptSegmentValue) => {{
      const out = Array.from({{ length: MSMETHOD_PRIMARY_FIELD_COUNT }}, () => "");
      if (
        timeSegmentValue === UNSELECTED_SEGMENT_VALUE ||
        exptSegmentValue === UNSELECTED_SEGMENT_VALUE
      ) {{
        return out;
      }}
      const timeIndex = Number.parseInt(String(timeSegmentValue || "1"), 10) - 1;
      const exptIndex = Number.parseInt(String(exptSegmentValue || "1"), 10) - 1;
      if (!Number.isFinite(timeIndex) || !Number.isFinite(exptIndex) || timeIndex < 0 || exptIndex < 0) {{
        return out;
      }}

      const segments = methodPrimaryUnknownFieldsByName[methodName];
      if (!Array.isArray(segments) || timeIndex >= segments.length) {{
        return out;
      }}
      const segment = segments[timeIndex];
      const fields = Array.isArray(segment?.primary_unknown_fields) ? segment.primary_unknown_fields : [];
      const n = Math.min(MSMETHOD_PRIMARY_FIELD_COUNT, fields.length);
      for (let idx = 0; idx < n; idx += 1) {{
        const values = Array.isArray(fields[idx]?.values) ? fields[idx].values : [];
        if (exptIndex < values.length) {{
          out[idx] = formatMsMethodPrimaryValue(
            values[exptIndex],
            idx,
            false,
            MSMETHOD_PRIMARY_POLARITY_FIELD_INDEXES.has(idx),
            true
          );
        }}
      }}
      return out;
    }};

    const extractMsMethodPrimaryPolarityForSelection = (methodName, timeSegmentValue, exptSegmentValue) => {{
      if (
        timeSegmentValue === UNSELECTED_SEGMENT_VALUE ||
        exptSegmentValue === UNSELECTED_SEGMENT_VALUE
      ) {{
        return null;
      }}
      const timeIndex = Number.parseInt(String(timeSegmentValue || "1"), 10) - 1;
      const exptIndex = Number.parseInt(String(exptSegmentValue || "1"), 10) - 1;
      if (!Number.isFinite(timeIndex) || !Number.isFinite(exptIndex) || timeIndex < 0 || exptIndex < 0) {{
        return null;
      }}

      const segments = methodPrimaryUnknownFieldsByName[methodName];
      if (!Array.isArray(segments) || timeIndex >= segments.length) {{
        return null;
      }}
      const segment = segments[timeIndex];
      const fields = Array.isArray(segment?.primary_unknown_fields) ? segment.primary_unknown_fields : [];
      const polarityFieldIndexes = Array.from(MSMETHOD_PRIMARY_POLARITY_FIELD_INDEXES).sort((a, b) => a - b);
      for (const idx of polarityFieldIndexes) {{
        if (idx < 0 || idx >= fields.length) {{
          continue;
        }}
        const values = Array.isArray(fields[idx]?.values) ? fields[idx].values : [];
        if (exptIndex < 0 || exptIndex >= values.length) {{
          continue;
        }}
        const code = normalizePolarityCode(values[exptIndex]);
        if (code === 0) {{
          return "positive";
        }}
        if (code === 1) {{
          return "negative";
        }}
      }}
      return null;
    }};

    const extractMsMethodExtensionValues = (methodName, timeSegmentValue) => {{
      const out = Array.from({{ length: MSMETHOD_EXTENSION_FIELD_COUNT }}, () => "");
      if (timeSegmentValue === UNSELECTED_SEGMENT_VALUE) {{
        return out;
      }}
      const timeIndex = Number.parseInt(String(timeSegmentValue || "1"), 10) - 1;
      if (!Number.isFinite(timeIndex) || timeIndex < 0) {{
        return out;
      }}

      const segments = methodPrimaryUnknownFieldsByName[methodName];
      if (!Array.isArray(segments) || timeIndex >= segments.length) {{
        return out;
      }}
      const segment = segments[timeIndex];
      const fields = Array.isArray(segment?.extension_unknown_fields) ? segment.extension_unknown_fields : [];
      const n = Math.min(MSMETHOD_EXTENSION_FIELD_COUNT, fields.length);
      for (let idx = 0; idx < n; idx += 1) {{
        const values = Array.isArray(fields[idx]?.values) ? fields[idx].values : [];
        if (values.length > 0) {{
          out[idx] = formatMsMethodPrimaryValue(values[0], idx);
        }}
      }}
      return out;
    }};

    const refreshMsMethodDetailsPrimaryTableValues = () => {{
      const table = document.getElementById("ms_method_details_table");
      if (!(table instanceof HTMLTableElement)) {{
        return;
      }}
      const leafHeaderNames = getTableLeafHeaderNames(table);
      const fieldIndexByColumnPosition = leafHeaderNames.map((columnName, columnIdx) => {{
        if (columnIdx === 0) {{
          return null;
        }}
        const mapped = MSMETHOD_PRIMARY_FIELD_INDEX_BY_COLUMN_NAME[String(columnName || "")];
        const fieldIdx = Number(mapped);
        if (
          !Number.isInteger(fieldIdx) ||
          fieldIdx < 0 ||
          fieldIdx >= MSMETHOD_PRIMARY_FIELD_COUNT
        ) {{
          return null;
        }}
        return fieldIdx;
      }});
      const selectionMap = getMethodSegmentSelectionMap();
      const rows = Array.from(table.querySelectorAll("tbody tr"));
      rows.forEach((row) => {{
        if (!(row instanceof HTMLTableRowElement)) {{
          return;
        }}
        const methodName = getRowMethodName(row);
        if (!methodName) {{
          return;
        }}
        const selection = selectionMap.get(methodName) || {{ time: "1", expt: "1" }};
        const values = extractMsMethodPrimaryValues(methodName, selection.time, selection.expt);
        for (let colIdx = 1; colIdx < fieldIndexByColumnPosition.length; colIdx += 1) {{
          const cell = row.children[colIdx];
          if (!(cell instanceof HTMLTableCellElement)) {{
            continue;
          }}
          const fieldIdx = fieldIndexByColumnPosition[colIdx];
          if (!Number.isInteger(fieldIdx)) {{
            cell.textContent = "";
            continue;
          }}
          cell.textContent = values[fieldIdx];
        }}
      }});
    }};

    const refreshMsMethodDetailsExtensionTableValues = () => {{
      const table = document.getElementById("ms_method_details_extension_table");
      if (!(table instanceof HTMLTableElement)) {{
        return;
      }}
      const selectionMap = getMethodSegmentSelectionMap();
      const rows = Array.from(table.querySelectorAll("tbody tr"));
      rows.forEach((row) => {{
        if (!(row instanceof HTMLTableRowElement)) {{
          return;
        }}
        const methodName = getRowMethodName(row);
        if (!methodName) {{
          return;
        }}
        const selection = selectionMap.get(methodName) || {{ time: "1", expt: "1" }};
        const values = extractMsMethodExtensionValues(methodName, selection.time);
        for (let fieldIdx = 0; fieldIdx < MSMETHOD_EXTENSION_FIELD_COUNT; fieldIdx += 1) {{
          const cell = row.children[fieldIdx + 1];
          if (!(cell instanceof HTMLTableCellElement)) {{
            continue;
          }}
          cell.textContent = values[fieldIdx];
        }}
      }});
    }};

    const msmethodRefMassIndexesForTrailingFieldCount = (nTrailingFieldsRaw) => {{
      const nTrailingFields = Number.isFinite(Number(nTrailingFieldsRaw))
        ? Math.max(0, Math.trunc(Number(nTrailingFieldsRaw)))
        : 0;
      if (nTrailingFields <= 0) {{
        return {{ pos: null, neg: null }};
      }}
      if (nTrailingFields === 1) {{
        return {{ pos: null, neg: 0 }};
      }}
      return {{
        pos: nTrailingFields - 2,
        neg: nTrailingFields - 1,
      }};
    }};

    const matchMsMethodTrailingCompactionRule = (values, startIndex) => {{
      if (!Array.isArray(values)) {{
        return null;
      }}
      if (startIndex < 0) {{
        return null;
      }}
      for (const rule of MSMETHOD_TRAILING_COMPACTION_RULES) {{
        const signature = Array.isArray(rule?.signature) ? rule.signature : [];
        if (startIndex + signature.length > values.length) {{
          continue;
        }}
        let matched = true;
        for (let offset = 0; offset < signature.length; offset += 1) {{
          const expected = signature[offset];
          const actual = normalizePolarityCode(values[startIndex + offset]);
          if (actual !== expected) {{
            matched = false;
            break;
          }}
        }}
        if (matched) {{
          return rule;
        }}
      }}
      return null;
    }};

    const msMethodTrailingTypeLabelFromGenericRow = (rowValues) => {{
      if (!Array.isArray(rowValues) || rowValues.length <= 1) {{
        return "";
      }}
      const code = normalizePolarityCode(rowValues[1]);
      if (code === null) {{
        return "";
      }}
      const label = MSMETHOD_TRAILING_TYPE_BY_CODE[String(code)];
      return typeof label === "string" ? label : "";
    }};

    const extractMsMethodTrailingDisplay = (methodName, timeSegmentValue, exptSegmentValue) => {{
      const out = {{
        refPos: "",
        refNeg: "",
        genericRows: [[]],
      }};
      const fields = methodPostambleUnknownFieldsByName[methodName];
      if (!Array.isArray(fields) || !fields.length) {{
        return out;
      }}

      const n = Math.min(MSMETHOD_TRAILING_FIELD_COUNT, fields.length);
      const refIndexes = msmethodRefMassIndexesForTrailingFieldCount(n);
      const refPosIndex = refIndexes.pos;
      const refNegIndex = refIndexes.neg;

      const genericItems = [];
      for (let idx = 0; idx < n; idx += 1) {{
        const values = Array.isArray(fields[idx]?.values) ? fields[idx].values : [];
        const rawValue = values.length > 0 ? values[0] : "";
        const displayValue = formatMsMethodPrimaryValue(
          rawValue,
          idx,
          idx === refPosIndex || idx === refNegIndex
        );
        if (idx === refPosIndex) {{
          out.refPos = displayValue;
          continue;
        }}
        if (idx === refNegIndex) {{
          out.refNeg = displayValue;
          continue;
        }}
        genericItems.push({{ raw: rawValue, display: displayValue }});
      }}

      const genericRawValues = genericItems.map((item) => item.raw);
      const genericRows = [];
      let cursor = 0;
      while (cursor < genericItems.length) {{
        const rowValues = [];
        // Per requested display rule, first value in each row is a row index and skipped.
        cursor += 1;
        let pendingBlankInsertions = [];

        while (
          rowValues.length < MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS &&
          cursor < genericItems.length
        ) {{
          const compactRule = matchMsMethodTrailingCompactionRule(genericRawValues, cursor);
          if (compactRule) {{
            const signature = Array.isArray(compactRule.signature) ? compactRule.signature : [];
            const insertAfter = Number.isFinite(Number(compactRule.insert_after_n_values))
              ? Math.max(0, Math.trunc(Number(compactRule.insert_after_n_values)))
              : 0;
            const blankCount = Number.isFinite(Number(compactRule.blank_cell_count))
              ? Math.max(0, Math.trunc(Number(compactRule.blank_cell_count)))
              : 0;
            cursor += signature.length;
            pendingBlankInsertions.push({{ remaining: insertAfter, blankCount }});
            continue;
          }}

          rowValues.push(genericItems[cursor].display);
          cursor += 1;

          if (pendingBlankInsertions.length) {{
            pendingBlankInsertions = pendingBlankInsertions.map((item) => ({{
              ...item,
              remaining: item.remaining - 1,
            }}));
            const ready = pendingBlankInsertions.filter((item) => item.remaining <= 0);
            pendingBlankInsertions = pendingBlankInsertions.filter((item) => item.remaining > 0);
            for (const item of ready) {{
              for (let blankIdx = 0; blankIdx < item.blankCount; blankIdx += 1) {{
                if (rowValues.length >= MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS) {{
                  break;
                }}
                rowValues.push("");
              }}
            }}
          }}
        }}
        genericRows.push(rowValues);
      }}
      out.genericRows = genericRows.length ? genericRows : [[]];
      return out;
    }};

    const formatMsMethodTrailingEnumByColumnName = (columnName, value) => {{
      const name = String(columnName || "").trim();
      const code = normalizePolarityCode(value);
      if (code === null) {{
        return String(value ?? "");
      }}
      if (name === "expt_type") {{
        const mapped = MSMETHOD_TRAILING_EXPT_TYPE_BY_CODE[String(code)];
        if (typeof mapped === "string") {{
          return mapped;
        }}
      }}
      if (name === "polarity") {{
        const mapped = MSMETHOD_TRAILING_POLARITY_BY_CODE[String(code)];
        if (typeof mapped === "string") {{
          return mapped;
        }}
      }}
      return String(value ?? "");
    }};

    const formatMsMethodTrailingWrappedCell = (fieldIdx, value, columnName = "") => {{
      if (!Number.isFinite(fieldIdx) || fieldIdx < 0 || fieldIdx >= MSMETHOD_TRAILING_FIELD_COUNT) {{
        return "";
      }}
      const mapped = formatMsMethodTrailingEnumByColumnName(columnName, value);
      return mapped.replace(/\\\\n/g, String.fromCharCode(10));
    }};

    const msMethodTrailingDisplayColumnNames = (table) => {{
      if (!(table instanceof HTMLTableElement)) {{
        return [];
      }}
      return getTableLeafHeaderNames(table);
    }};

    const buildMsMethodTrailingRowModels = (methodNames, selectionMap, columnCount) => {{
      const rowModels = [];
      methodNames.forEach((methodName) => {{
        const selection = selectionMap.get(methodName) || {{ time: "1", expt: "1" }};
        const selectedPolarity = extractMsMethodPrimaryPolarityForSelection(
          methodName,
          selection.time,
          selection.expt
        );
        const display = extractMsMethodTrailingDisplay(
          methodName,
          selection.time,
          selection.expt
        );
        const genericRows = Array.isArray(display.genericRows) && display.genericRows.length
          ? display.genericRows
          : [[]];
        const refPosValue = selectedPolarity === "negative" ? "" : display.refPos;
        const refNegValue = selectedPolarity === "positive" ? "" : display.refNeg;

        genericRows.forEach((rowValues, rowIndex) => {{
          const valuesByColumnIndex = Array.from({{ length: columnCount }}, () => "");
          if (columnCount > 0) {{
            valuesByColumnIndex[0] = methodName;
          }}
          if (columnCount > 1) {{
            valuesByColumnIndex[1] = rowIndex === 0 ? refPosValue : "";
          }}
          if (columnCount > 2) {{
            valuesByColumnIndex[2] = rowIndex === 0 ? refNegValue : "";
          }}
          if (columnCount > 3) {{
            valuesByColumnIndex[3] = msMethodTrailingTypeLabelFromGenericRow(rowValues);
          }}

          let columnIndex = 4;
          for (let genericSlotIdx = 0; genericSlotIdx < MSMETHOD_TRAILING_GENERIC_WRAP_COLUMNS; genericSlotIdx += 1) {{
            if (MSMETHOD_TRAILING_HIDDEN_GENERIC_SLOT_INDEXES.has(genericSlotIdx)) {{
              continue;
            }}
            if (columnIndex >= columnCount) {{
              break;
            }}
            const value = genericSlotIdx < rowValues.length ? rowValues[genericSlotIdx] : "";
            valuesByColumnIndex[columnIndex] = value;
            columnIndex += 1;
          }}

          rowModels.push({{
            methodName,
            rowIndex,
            valuesByColumnIndex,
          }});
        }});
      }});
      return rowModels;
    }};

    const renderMsMethodTrailingRowModels = (tbody, rowModels, columnNames) => {{
      const fragment = document.createDocumentFragment();
      rowModels.forEach((rowModel) => {{
        const row = document.createElement("tr");
        row.setAttribute("data-method-dir", rowModel.methodName);
        row.setAttribute("data-trailing-row-index", String(rowModel.rowIndex));

        rowModel.valuesByColumnIndex.forEach((value, columnIndex) => {{
          const cell = document.createElement("td");
          const columnName = Array.isArray(columnNames) && columnIndex < columnNames.length
            ? String(columnNames[columnIndex] || "")
            : `column_${{columnIndex + 1}}`;
          cell.dataset.columnIndex = String(columnIndex);
          if (columnIndex === 0) {{
            cell.textContent = String(value ?? "");
          }} else {{
            cell.textContent = formatMsMethodTrailingWrappedCell(
              columnIndex - 1,
              value,
              columnName
            );
          }}
          row.appendChild(cell);
        }});
        fragment.appendChild(row);
      }});

      tbody.innerHTML = "";
      tbody.appendChild(fragment);
    }};

    const refreshMsMethodDetailsTrailingTableValues = () => {{
      const table = document.getElementById("ms_method_details_trailing_table");
      if (!(table instanceof HTMLTableElement)) {{
        return;
      }}
      const tbody = table.querySelector("tbody");
      if (!(tbody instanceof HTMLTableSectionElement)) {{
        return;
      }}

      const columnNames = msMethodTrailingDisplayColumnNames(table);
      const columnCount = columnNames.length;
      const selectionMap = getMethodSegmentSelectionMap();
      const methodNames = uniqueMethodsFromTable();
      const rowModels = buildMsMethodTrailingRowModels(methodNames, selectionMap, columnCount);
      renderMsMethodTrailingRowModels(tbody, rowModels, columnNames);
      delete table.dataset.methodDirMerged;

      mergeMethodDirCellsByRowspan("ms_method_details_trailing_table");
      updateMsMethodTrailingStickyOffsets();
    }};

    const compareMajorityCandidates = (leftRaw, rightRaw) => {{
      const left = String(leftRaw ?? "").trim();
      const right = String(rightRaw ?? "").trim();
      const leftNum = Number(left);
      const rightNum = Number(right);
      const leftIsNum = Number.isFinite(leftNum);
      const rightIsNum = Number.isFinite(rightNum);

      if (leftIsNum && rightIsNum && leftNum !== rightNum) {{
        return leftNum - rightNum;
      }}
      if (leftIsNum !== rightIsNum) {{
        return leftIsNum ? -1 : 1;
      }}
      if (left.length !== right.length) {{
        return left.length - right.length;
      }}
      return left.localeCompare(right);
    }};

    const pickMajorityValue = (values) => {{
      if (!Array.isArray(values) || values.length === 0) {{
        return null;
      }}
      const counts = new Map();
      values.forEach((value) => {{
        const key = String(value ?? "");
        counts.set(key, (counts.get(key) || 0) + 1);
      }});

      let bestValue = null;
      let bestCount = -1;
      counts.forEach((count, value) => {{
        if (count > bestCount) {{
          bestCount = count;
          bestValue = value;
          return;
        }}
        if (count === bestCount && bestValue !== null) {{
          if (compareMajorityCandidates(value, bestValue) < 0) {{
            bestValue = value;
          }}
        }}
      }});
      return bestValue;
    }};

    const applyMajorityHighlightByExtractor = (extractor) => {{
      const entries = getMethodConfigRows()
        .map((row) => extractor(row))
        .filter((entry) => entry && entry.element && entry.value !== undefined && entry.value !== null);
      if (!entries.length) {{
        return;
      }}

      const majorityValue = pickMajorityValue(entries.map((entry) => entry.value));
      entries.forEach((entry) => {{
        const element = entry.element;
        if (!(element instanceof HTMLElement)) return;
        const isOutlier = majorityValue !== null && String(entry.value) !== String(majorityValue);
        element.classList.toggle("majority-outlier", isOutlier);
      }});
    }};

    const applyMethodConfigMajorityHighlight = () => {{
      applyMajorityHighlightByExtractor((row) => {{
        const element = row.querySelector(".segment-info-cell:not(.expt-segment-info-cell)");
        return {{
          element,
          value: (element?.textContent || "").trim(),
        }};
      }});

      applyMajorityHighlightByExtractor((row) => {{
        const element = row.querySelector(".expt-segment-info-cell");
        return {{
          element,
          value: (element?.textContent || "").trim(),
        }};
      }});

      applyMajorityHighlightByExtractor((row) => {{
        const element = row.querySelector(".method-time-select");
        return {{
          element,
          value: element instanceof HTMLSelectElement ? element.value : "",
        }};
      }});

      applyMajorityHighlightByExtractor((row) => {{
        const element = row.querySelector(".method-experimental-select");
        return {{
          element,
          value: element instanceof HTMLSelectElement ? element.value : "",
        }};
      }});
    }};

    const exptSegmentOptionValuesForMaster = () => {{
      let maxCount = 1;
      const rows = getMethodConfigRows();
      if (!rows.length) {{
        return optionValuesByCount(maxCount);
      }}
      rows.forEach((row) => {{
        const methodName = row.getAttribute("data-method") || "";
        if (!methodName) return;
        const timeSelect = row.querySelector(".method-time-select");
        const timeValue = timeSelect instanceof HTMLSelectElement ? timeSelect.value : "1";
        const exptCount = exptSegmentCountForMethodTime(methodName, timeValue);
        if (exptCount > maxCount) {{
          maxCount = exptCount;
        }}
      }});
      return optionValuesByCount(maxCount);
    }};

    const refreshMethodRowExptOptions = (methodRow) => {{
      if (!(methodRow instanceof HTMLElement)) return;
      const methodName = methodRow.getAttribute("data-method") || "";
      if (!methodName) return;
      const timeSelect = methodRow.querySelector(".method-time-select");
      const exptSelect = methodRow.querySelector(".method-experimental-select");
      if (!(exptSelect instanceof HTMLSelectElement)) return;
      const timeValue = timeSelect instanceof HTMLSelectElement ? timeSelect.value : "1";
      const previousValue = exptSelect.value || "1";
      fillSegmentSelect(
        exptSelect,
        previousValue,
        (value) => value,
        exptSegmentOptionValuesForMethod(methodName, timeValue),
        true,
        true
      );
    }};

    const refreshMasterExperimentSelectOptions = () => {{
      const masterSelect = document.getElementById("method_master_experimental");
      if (!(masterSelect instanceof HTMLSelectElement)) return;
      const previousValue =
        masterSelect.value && masterSelect.value !== MASTER_MIXED_VALUE ? masterSelect.value : "1";
      fillSegmentSelect(
        masterSelect,
        previousValue,
        (value) => `All: Expt Seg ${{value}}`,
        exptSegmentOptionValuesForMaster()
      );
    }};

    const syncMasterSelectFromRows = (masterId, rowSelectClass, mixedLabel) => {{
      const masterSelect = document.getElementById(masterId);
      if (!(masterSelect instanceof HTMLSelectElement)) return;

      const rowValues = getMethodConfigRows()
        .map((row) => row.querySelector(rowSelectClass))
        .filter((el) => el instanceof HTMLSelectElement)
        .map((el) => el.value);
      if (!rowValues.length) return;

      const uniqueValues = Array.from(new Set(rowValues));
      const mixedOption = masterSelect.querySelector(`option[value="${{MASTER_MIXED_VALUE}}"]`);

      if (uniqueValues.length === 1) {{
        const soleValue = uniqueValues[0];
        const hasMatchingMasterOption = Boolean(
          masterSelect.querySelector(`option[value="${{soleValue}}"]`)
        );
        if (!hasMatchingMasterOption) {{
          if (!mixedOption) {{
            const option = document.createElement("option");
            option.value = MASTER_MIXED_VALUE;
            option.textContent = mixedLabel;
            option.selected = true;
            masterSelect.appendChild(option);
          }} else {{
            mixedOption.textContent = mixedLabel;
            mixedOption.selected = true;
          }}
          masterSelect.value = MASTER_MIXED_VALUE;
          return;
        }}
        if (mixedOption) {{
          mixedOption.remove();
        }}
        masterSelect.value = soleValue;
        return;
      }}

      if (!mixedOption) {{
        const option = document.createElement("option");
        option.value = MASTER_MIXED_VALUE;
        option.textContent = mixedLabel;
        option.selected = true;
        masterSelect.appendChild(option);
      }} else {{
        mixedOption.textContent = mixedLabel;
        mixedOption.selected = true;
      }}
      masterSelect.value = MASTER_MIXED_VALUE;
    }};

    const syncMasterSegmentSelectors = () => {{
      refreshMasterExperimentSelectOptions();
      syncMasterSelectFromRows("method_master_time", ".method-time-select", "Mixed: Time Seg");
      syncMasterSelectFromRows(
        "method_master_experimental",
        ".method-experimental-select",
        "Mixed: Expt Seg"
      );
      applyMethodConfigMajorityHighlight();
    }};

    const selectedMethods = () => {{
      const selected = getMethodConfigRows()
        .filter((row) => {{
          const checkbox = row.querySelector(".method-visible-checkbox");
          return checkbox instanceof HTMLInputElement && checkbox.checked;
        }})
        .map((row) => row.getAttribute("data-method") || "")
        .filter((name) => name.length > 0);
      return new Set(selected);
    }};

    const syncMasterVisibleCheckboxFromRows = () => {{
      const masterCheckbox = document.getElementById("method_master_visible");
      if (!(masterCheckbox instanceof HTMLInputElement)) return;

      const rows = getMethodConfigRows();
      if (!rows.length) {{
        masterCheckbox.checked = false;
        masterCheckbox.indeterminate = false;
        return;
      }}

      const checkedCount = rows.filter((row) => {{
        const checkbox = row.querySelector(".method-visible-checkbox");
        return checkbox instanceof HTMLInputElement && checkbox.checked;
      }}).length;

      if (checkedCount === rows.length) {{
        masterCheckbox.checked = true;
        masterCheckbox.indeterminate = false;
      }} else if (checkedCount === 0) {{
        masterCheckbox.checked = false;
        masterCheckbox.indeterminate = false;
      }} else {{
        masterCheckbox.checked = false;
        masterCheckbox.indeterminate = true;
      }}
    }};

    const parseTableCellComparisonValue = (text) => {{
      const raw = String(text ?? "").trim();
      if (!raw) {{
        return null;
      }}

      const normalized = raw.replace(/,/g, "");
      const numericPattern = /^[+-]?(?:\\d+\\.?\\d*|\\.\\d+)(?:[eE][+-]?\\d+)?$/;
      if (numericPattern.test(normalized)) {{
        const numericValue = Number(normalized);
        if (Number.isFinite(numericValue)) {{
          return {{
            key: `n:${{numericValue}}`,
            comparableValue: numericValue,
          }};
        }}
      }}

      // String columns are compared by exact (trimmed) text.
      return {{
        key: `s:${{raw}}`,
        comparableValue: raw,
      }};
    }};

    const applyValueDiffHighlightForTable = (tableId) => {{
      const table = document.getElementById(tableId);
      if (!(table instanceof HTMLTableElement)) return;
      const isTrailingTable = tableId === "ms_method_details_trailing_table";

      const allBodyCells = Array.from(table.querySelectorAll("tbody td"));
      allBodyCells.forEach((cell) => {{
        cell.classList.remove("numeric-diff-outlier");
      }});

      const getComparableCellForColumn = (row, colIdx) => {{
        const mappedCell = row.querySelector(`td[data-column-index="${{colIdx}}"]`);
        if (mappedCell instanceof HTMLTableCellElement) {{
          return mappedCell;
        }}
        if (isTrailingTable) {{
          return null;
        }}
        const fallbackCell = row.children[colIdx];
        return fallbackCell instanceof HTMLTableCellElement ? fallbackCell : null;
      }};

      const rows = Array.from(table.querySelectorAll("tbody tr")).filter(
        (row) => row instanceof HTMLTableRowElement && row.style.display !== "none"
      );
      if (rows.length <= 1) {{
        return;
      }}

      const leafHeaderCells = getTableLeafHeaderCells(table);
      const columnCount = leafHeaderCells.length || (rows[0]?.querySelectorAll("td").length ?? 0);
      if (columnCount <= 1) {{
        return;
      }}

      for (let colIdx = 1; colIdx < columnCount; colIdx += 1) {{
        const entries = [];
        rows.forEach((row) => {{
          const cell = getComparableCellForColumn(row, colIdx);
          if (!(cell instanceof HTMLTableCellElement)) {{
            return;
          }}
          if (cell.style.display === "none" || cell.classList.contains("auto-hidden-column")) {{
            return;
          }}
          const parsed = parseTableCellComparisonValue(cell.textContent);
          if (!parsed) {{
            return;
          }}
          entries.push({{
            cell,
            key: parsed.key,
            comparableValue: parsed.comparableValue,
            methodName: getRowMethodName(row),
          }});
        }});

        if (entries.length <= 1) {{
          continue;
        }}

        const valueCounts = new Map();
        if (isTrailingTable) {{
          // For trailing rows, one method may contribute multiple wrapped rows.
          // Count each (method, value) at most once so a single method does not dominate.
          const seenMethodValuePairs = new Set();
          entries.forEach((entry) => {{
            const methodName = String(entry.methodName || "");
            const dedupeKey = `${{methodName}}\\u0000${{entry.key}}`;
            if (methodName && seenMethodValuePairs.has(dedupeKey)) {{
              return;
            }}
            if (methodName) {{
              seenMethodValuePairs.add(dedupeKey);
            }}
            const existing = valueCounts.get(entry.key);
            if (existing) {{
              existing.count += 1;
              return;
            }}
            valueCounts.set(entry.key, {{
              count: 1,
              comparableValue: entry.comparableValue,
            }});
          }});
        }} else {{
          entries.forEach((entry) => {{
            const existing = valueCounts.get(entry.key);
            if (existing) {{
              existing.count += 1;
              return;
            }}
            valueCounts.set(entry.key, {{
              count: 1,
              comparableValue: entry.comparableValue,
            }});
          }});
        }}
        if (valueCounts.size <= 1) {{
          continue;
        }}

        let majorityKey = null;
        let bestCount = -1;
        let majorityComparableValue = null;
        valueCounts.forEach((valueInfo, key) => {{
          if (valueInfo.count > bestCount) {{
            bestCount = valueInfo.count;
            majorityKey = key;
            majorityComparableValue = valueInfo.comparableValue;
            return;
          }}
          if (
            valueInfo.count === bestCount &&
            majorityComparableValue !== null &&
            compareMajorityCandidates(valueInfo.comparableValue, majorityComparableValue) < 0
          ) {{
            majorityKey = key;
            majorityComparableValue = valueInfo.comparableValue;
          }}
        }});

        if (majorityKey === null) {{
          continue;
        }}

        entries.forEach((entry) => {{
          if (entry.key !== majorityKey) {{
            entry.cell.classList.add("numeric-diff-outlier");
          }}
        }});
      }}
    }};

    const applyAllNumericDiffHighlights = () => {{
      applyValueDiffHighlightForTable("lc_method_table");
      applyValueDiffHighlightForTable("ms_method_details_table");
      applyValueDiffHighlightForTable("ms_method_details_extension_table");
      applyValueDiffHighlightForTable("ms_method_details_trailing_table");
      applyValueDiffHighlightForTable("ms_tune_positive_table");
      applyValueDiffHighlightForTable("ms_tune_negative_table");
    }};

    const filterTableRowsByMethod = (tableId, selected) => {{
      const table = document.getElementById(tableId);
      if (!table) return;
      const rows = table.querySelectorAll("tbody tr");
      rows.forEach((row) => {{
        const methodName = getRowMethodName(row);
        row.style.display = selected.has(methodName) ? "" : "none";
      }});
    }};

    const applyMethodFilter = () => {{
      const selected = selectedMethods();
      const filteredTraces = traces.filter((trace) => selected.has(String(trace.name || "")));
      Plotly.react("gradient_plot", filteredTraces, layout, {{ responsive: true }});
      refreshMsMethodDetailsPrimaryTableValues();
      refreshMsMethodDetailsExtensionTableValues();
      refreshMsMethodDetailsTrailingTableValues();
      filterTableRowsByMethod("lc_method_table", selected);
      filterTableRowsByMethod("ms_method_details_table", selected);
      filterTableRowsByMethod("ms_method_details_extension_table", selected);
      filterTableRowsByMethod("ms_method_details_trailing_table", selected);
      filterTableRowsByMethod("ms_tune_positive_table", selected);
      filterTableRowsByMethod("ms_tune_negative_table", selected);
      syncMasterVisibleCheckboxFromRows();
      applyAllNumericDiffHighlights();
    }};

    const createMasterMethodRow = () => {{
      const row = document.createElement("tr");
      row.className = "master-row";

      const visibleCell = document.createElement("td");
      visibleCell.className = "checkbox-cell";
      const visibleCheckbox = document.createElement("input");
      visibleCheckbox.type = "checkbox";
      visibleCheckbox.id = "method_master_visible";
      visibleCheckbox.checked = true;
      visibleCheckbox.title = "Toggle all visible methods";
      visibleCheckbox.addEventListener("change", () => {{
        visibleCheckbox.indeterminate = false;
        setAllMethodVisibility(visibleCheckbox.checked);
      }});
      visibleCell.appendChild(visibleCheckbox);

      const colorCell = document.createElement("td");
      colorCell.className = "method-color-cell";
      colorCell.textContent = "Color";

      const methodCell = document.createElement("td");
      methodCell.className = "method-name-cell";
      methodCell.textContent = "Method";

      const segmentInfoCell = document.createElement("td");
      segmentInfoCell.className = "segment-info-cell master-segment-info-cell";
      segmentInfoCell.textContent = "Time Seg [min]";

      const exptSegmentInfoCell = document.createElement("td");
      exptSegmentInfoCell.className = "segment-info-cell expt-segment-info-cell master-segment-info-cell";
      exptSegmentInfoCell.textContent = "Expt Seg [#]";

      const timeCell = document.createElement("td");
      timeCell.className = "select-cell";
      const timeSelect = document.createElement("select");
      timeSelect.id = "method_master_time";
      fillSegmentSelect(
        timeSelect,
        "1",
        (value) => `All: Time Seg ${{value}}`,
        masterTimeSegmentOptionValues(),
        true,
        false
      );
      timeSelect.addEventListener("change", () => {{
        if (timeSelect.value === MASTER_MIXED_VALUE) {{
          return;
        }}
        getMethodConfigRows().forEach((methodRow) => {{
          const selectEl = methodRow.querySelector(".method-time-select");
          if (selectEl instanceof HTMLSelectElement) {{
            const matching = Array.from(selectEl.options).find(
              (option) => option.value === timeSelect.value
            );
            if (matching) {{
              selectEl.value = timeSelect.value;
            }} else {{
              const unselectedOption = Array.from(selectEl.options).find(
                (option) => option.value === UNSELECTED_SEGMENT_VALUE
              );
              if (unselectedOption) {{
                selectEl.value = UNSELECTED_SEGMENT_VALUE;
              }}
            }}
          }}
          refreshMethodRowExptOptions(methodRow);
        }});
        syncMasterSegmentSelectors();
        applyMethodFilter();
      }});
      timeCell.appendChild(timeSelect);

      const experimentalCell = document.createElement("td");
      experimentalCell.className = "select-cell";
      const experimentalSelect = document.createElement("select");
      experimentalSelect.id = "method_master_experimental";
      fillSegmentSelect(
        experimentalSelect,
        "1",
        (value) => `All: Expt Seg ${{value}}`,
        exptSegmentOptionValuesForMaster()
      );
      experimentalSelect.addEventListener("change", () => {{
        if (experimentalSelect.value === MASTER_MIXED_VALUE) {{
          return;
        }}
        getMethodConfigRows().forEach((methodRow) => {{
          const selectEl = methodRow.querySelector(".method-experimental-select");
          if (selectEl instanceof HTMLSelectElement) {{
            const matching = Array.from(selectEl.options).find(
              (option) => option.value === experimentalSelect.value
            );
            if (matching) {{
              selectEl.value = experimentalSelect.value;
            }} else {{
              const unselectedOption = Array.from(selectEl.options).find(
                (option) => option.value === UNSELECTED_SEGMENT_VALUE
              );
              if (unselectedOption) {{
                selectEl.value = UNSELECTED_SEGMENT_VALUE;
              }}
            }}
          }}
        }});
        syncMasterSegmentSelectors();
        applyMethodFilter();
      }});
      experimentalCell.appendChild(experimentalSelect);

      row.appendChild(visibleCell);
      row.appendChild(colorCell);
      row.appendChild(methodCell);
      row.appendChild(segmentInfoCell);
      row.appendChild(exptSegmentInfoCell);
      row.appendChild(timeCell);
      row.appendChild(experimentalCell);
      return row;
    }};

    const createMethodConfigRow = (methodName) => {{
      const row = document.createElement("tr");
      row.className = "method-row";
      row.setAttribute("data-method", methodName);

      const visibleCell = document.createElement("td");
      visibleCell.className = "checkbox-cell";
      const visibleCheckbox = document.createElement("input");
      visibleCheckbox.type = "checkbox";
      visibleCheckbox.className = "method-visible-checkbox";
      visibleCheckbox.value = methodName;
      visibleCheckbox.checked = true;
      visibleCheckbox.addEventListener("change", applyMethodFilter);
      visibleCell.appendChild(visibleCheckbox);

      const colorCell = document.createElement("td");
      colorCell.className = "method-color-cell";
      const colorBar = document.createElement("span");
      colorBar.className = "method-color-bar";
      colorBar.style.backgroundColor = traceColorForMethod(methodName);
      colorBar.title = `Method color: ${{methodName}}`;
      colorCell.appendChild(colorBar);

      const methodCell = document.createElement("td");
      methodCell.className = "method-name-cell";
      methodCell.textContent = methodName;

      const segmentInfoCell = document.createElement("td");
      segmentInfoCell.className = "segment-info-cell";
      segmentInfoCell.textContent = formatTimeSegmentInfo(methodName);
      segmentInfoCell.title = segmentInfoCell.textContent;

      const exptSegmentInfoCell = document.createElement("td");
      exptSegmentInfoCell.className = "segment-info-cell expt-segment-info-cell";
      exptSegmentInfoCell.textContent = formatExptSegmentInfo(methodName);
      exptSegmentInfoCell.title = exptSegmentInfoCell.textContent;

      const timeCell = document.createElement("td");
      timeCell.className = "select-cell";
      const timeSelect = document.createElement("select");
      timeSelect.className = "method-time-select";
      fillSegmentSelect(
        timeSelect,
        "1",
        (value) => value,
        timeSegmentOptionValuesForMethod(methodName),
        true,
        false
      );
      timeSelect.addEventListener("change", () => {{
        refreshMethodRowExptOptions(row);
        syncMasterSegmentSelectors();
        applyMethodFilter();
      }});
      timeCell.appendChild(timeSelect);

      const experimentalCell = document.createElement("td");
      experimentalCell.className = "select-cell";
      const experimentalSelect = document.createElement("select");
      experimentalSelect.className = "method-experimental-select";
      fillSegmentSelect(
        experimentalSelect,
        "1",
        (value) => value,
        exptSegmentOptionValuesForMethod(methodName, "1"),
        true,
        false
      );
      experimentalSelect.addEventListener("change", () => {{
        syncMasterSegmentSelectors();
        applyMethodFilter();
      }});
      experimentalCell.appendChild(experimentalSelect);

      row.appendChild(visibleCell);
      row.appendChild(colorCell);
      row.appendChild(methodCell);
      row.appendChild(segmentInfoCell);
      row.appendChild(exptSegmentInfoCell);
      row.appendChild(timeCell);
      row.appendChild(experimentalCell);
      return row;
    }};

    const setAllMethodVisibility = (checked) => {{
      getMethodConfigRows().forEach((row) => {{
        const checkbox = row.querySelector(".method-visible-checkbox");
        if (checkbox instanceof HTMLInputElement) {{
          checkbox.checked = checked;
        }}
      }});
      applyMethodFilter();
    }};

    const initMethodConfigTable = () => {{
      if (!methodConfigTableBody) return;
      methodConfigTableBody.innerHTML = "";
      methodConfigTableBody.appendChild(createMasterMethodRow());

      const methods = uniqueMethodsFromTable();
      methods.forEach((methodName) => {{
        methodConfigTableBody.appendChild(createMethodConfigRow(methodName));
      }});

      getMethodConfigRows().forEach((row) => {{
        refreshMethodRowExptOptions(row);
      }});
      syncMasterSegmentSelectors();
      applyMethodFilter();
    }};

    const initColumnSelector = (tableId, selectorId) => {{
      const table = document.getElementById(tableId);
      const selector = document.getElementById(selectorId);
      if (!table || !selector) return;
      const defaultUncheckedColumns = new Set(
        Array.isArray(defaultUncheckedColumnsByTable[tableId])
          ? defaultUncheckedColumnsByTable[tableId]
          : []
      );

      const leafHeaders = getTableLeafHeaderCells(table);
      if (!leafHeaders.length) {{
        return;
      }}
      const headerRows = getTableHeaderRows(table);
      const isPrimaryGroupedSelector =
        tableId === "ms_method_details_table" && headerRows.length >= 2;
      const topHeaderCells = isPrimaryGroupedSelector
        ? Array.from(headerRows[0].querySelectorAll("th"))
        : [];

      const columnControls = [];
      const columnControlByIndex = new Map();
      const categoryControls = [];

      const setColumnVisibility = (colIndex, isVisible) => {{
        const leafHeaderCell = leafHeaders[colIndex];
        if (leafHeaderCell instanceof HTMLElement) {{
          leafHeaderCell.style.display = isVisible ? "" : "none";
        }}
        const nth = colIndex + 1;
        const bodyCells = table.querySelectorAll(`tbody tr > *:nth-child(${{nth}})`);
        bodyCells.forEach((cell) => {{
          cell.style.display = isVisible ? "" : "none";
        }});
        if (tableId === "ms_method_details_table") {{
          updateMsMethodPrimaryStickyOffsets();
        }}
        if (tableId === "ms_method_details_trailing_table") {{
          updateMsMethodTrailingStickyOffsets();
        }}
      }};

      const refreshPrimaryGroupedTopHeader = () => {{
        if (!isPrimaryGroupedSelector) {{
          return;
        }}
        let cursor = 0;
        topHeaderCells.forEach((cell) => {{
          if (!(cell instanceof HTMLTableCellElement)) {{
            return;
          }}
          const rawSpan = Number.parseInt(
            String(cell.dataset.originalColspan || cell.getAttribute("colspan") || "1"),
            10
          );
          const span = Number.isFinite(rawSpan) && rawSpan > 0 ? rawSpan : 1;
          if (!cell.dataset.originalColspan) {{
            cell.dataset.originalColspan = String(span);
          }}

          let visibleCount = 0;
          for (let offset = 0; offset < span && cursor + offset < leafHeaders.length; offset += 1) {{
            const headerCell = leafHeaders[cursor + offset];
            if (headerCell instanceof HTMLElement && headerCell.style.display !== "none") {{
              visibleCount += 1;
            }}
          }}
          cursor += span;

          if (visibleCount <= 0) {{
            cell.style.display = "none";
            cell.colSpan = 1;
            return;
          }}
          cell.style.display = "";
          cell.colSpan = visibleCount;
        }});
      }};

      const syncCategoryCheckboxStates = () => {{
        categoryControls.forEach((category) => {{
          const active = category.indexes
            .map((idx) => columnControlByIndex.get(idx))
            .filter((control) => Boolean(control));
          if (!active.length) {{
            category.checkbox.checked = false;
            category.checkbox.indeterminate = false;
            return;
          }}
          const checkedCount = active.filter((control) => control.checkbox.checked).length;
          if (checkedCount === active.length) {{
            category.checkbox.checked = true;
            category.checkbox.indeterminate = false;
          }} else if (checkedCount === 0) {{
            category.checkbox.checked = false;
            category.checkbox.indeterminate = false;
          }} else {{
            category.checkbox.checked = false;
            category.checkbox.indeterminate = true;
          }}
        }});
      }};

      const createColumnCheckbox = (header, idx, container) => {{
        const name = (header.textContent || "").trim() || `column_${{idx + 1}}`;
        if (name === "method_dir") {{
          // Keep method_dir always visible and exclude it from selector UI.
          setColumnVisibility(idx, true);
          return null;
        }}
        const label = document.createElement("label");
        label.setAttribute("data-column-name", name);
        const checkbox = document.createElement("input");
        checkbox.type = "checkbox";
        const defaultChecked = !defaultUncheckedColumns.has(name);
        checkbox.checked = defaultChecked;
        checkbox.addEventListener("change", () => {{
          setColumnVisibility(idx, checkbox.checked);
          refreshPrimaryGroupedTopHeader();
          syncCategoryCheckboxStates();
          applyAllNumericDiffHighlights();
        }});
        label.appendChild(checkbox);
        label.appendChild(document.createTextNode(name));
        container.appendChild(label);
        setColumnVisibility(idx, checkbox.checked);
        const control = {{ checkbox, idx, defaultChecked }};
        columnControls.push(control);
        columnControlByIndex.set(idx, control);
        return control;
      }};

      if (isPrimaryGroupedSelector) {{
        let cursor = 0;
        topHeaderCells.forEach((cell, groupIdx) => {{
          if (!(cell instanceof HTMLTableCellElement)) {{
            return;
          }}
          const rawSpan = Number.parseInt(String(cell.getAttribute("colspan") || "1"), 10);
          const span = Number.isFinite(rawSpan) && rawSpan > 0 ? rawSpan : 1;
          cell.dataset.originalColspan = String(span);

          const indexes = [];
          for (let offset = 0; offset < span && cursor < leafHeaders.length; offset += 1) {{
            indexes.push(cursor);
            cursor += 1;
          }}
          const selectableIndexes = indexes.filter((idx) => {{
            const header = leafHeaders[idx];
            const name = (header?.textContent || "").trim() || `column_${{idx + 1}}`;
            return name !== "method_dir";
          }});
          if (!selectableIndexes.length) {{
            indexes.forEach((idx) => {{
              const header = leafHeaders[idx];
              const name = (header?.textContent || "").trim() || `column_${{idx + 1}}`;
              if (name === "method_dir") {{
                setColumnVisibility(idx, true);
              }}
            }});
            return;
          }}

          const categoryName = (cell.textContent || "").trim() || `group_${{groupIdx + 1}}`;
          const groupRoot = document.createElement("div");
          groupRoot.className = "column-selector-group";

          const titleWrap = document.createElement("div");
          titleWrap.className = "column-selector-group-title";
          const titleLabel = document.createElement("label");
          const categoryCheckbox = document.createElement("input");
          categoryCheckbox.type = "checkbox";
          titleLabel.appendChild(categoryCheckbox);
          titleLabel.appendChild(document.createTextNode(categoryName));
          titleWrap.appendChild(titleLabel);
          groupRoot.appendChild(titleWrap);

          const columnsWrap = document.createElement("div");
          columnsWrap.className = "column-selector-group-columns";
          selectableIndexes.forEach((idx) => {{
            const header = leafHeaders[idx];
            if (!(header instanceof HTMLTableCellElement)) {{
              return;
            }}
            createColumnCheckbox(header, idx, columnsWrap);
          }});
          groupRoot.appendChild(columnsWrap);
          selector.appendChild(groupRoot);

          categoryCheckbox.addEventListener("change", () => {{
            selectableIndexes.forEach((idx) => {{
              const control = columnControlByIndex.get(idx);
              if (!control) {{
                return;
              }}
              control.checkbox.checked = categoryCheckbox.checked;
              setColumnVisibility(idx, control.checkbox.checked);
            }});
            refreshPrimaryGroupedTopHeader();
            syncCategoryCheckboxStates();
            applyAllNumericDiffHighlights();
          }});
          categoryControls.push({{ checkbox: categoryCheckbox, indexes: selectableIndexes }});
        }});
      }} else {{
        leafHeaders.forEach((header, idx) => {{
          if (!(header instanceof HTMLTableCellElement)) {{
            return;
          }}
          createColumnCheckbox(header, idx, selector);
        }});
      }}

      const actions = document.createElement("div");
      actions.className = "column-filter-actions";

      const createActionButton = (label, onClick) => {{
        const button = document.createElement("button");
        button.type = "button";
        button.textContent = label;
        button.addEventListener("click", onClick);
        return button;
      }};

      const applyState = (resolver) => {{
        columnControls.forEach((control) => {{
          const nextChecked = resolver(control);
          control.checkbox.checked = nextChecked;
          setColumnVisibility(control.idx, nextChecked);
        }});
        refreshPrimaryGroupedTopHeader();
        syncCategoryCheckboxStates();
        applyAllNumericDiffHighlights();
      }};

      const defaultButton = createActionButton("Default", () => {{
        applyState((control) => control.defaultChecked);
      }});
      const selectAllButton = createActionButton("Select All", () => {{
        applyState(() => true);
      }});
      const clearAllButton = createActionButton("Clear All", () => {{
        applyState(() => false);
      }});

      actions.appendChild(defaultButton);
      actions.appendChild(selectAllButton);
      actions.appendChild(clearAllButton);
      selector.parentNode?.insertBefore(actions, selector);
      refreshPrimaryGroupedTopHeader();
      syncCategoryCheckboxStates();
    }};

    const initCollapsiblePanels = () => {{
      const collapsiblePanels = Array.from(document.querySelectorAll("details.collapsible-panel"));
      collapsiblePanels.forEach((panel) => {{
        panel.addEventListener("toggle", () => {{
          if (!panel.open) {{
            return;
          }}
          const gradientPlot = panel.querySelector("#gradient_plot");
          if (!(gradientPlot instanceof HTMLElement)) {{
            return;
          }}
          if (typeof Plotly === "undefined" || !Plotly.Plots || typeof Plotly.Plots.resize !== "function") {{
            return;
          }}
          Plotly.Plots.resize(gradientPlot);
        }});
      }});
    }};

    initCollapsiblePanels();
    initMethodConfigTable();
    initColumnSelector("lc_method_table", "column_selector");
    initColumnSelector("ms_method_details_table", "ms_method_details_column_selector");
    initColumnSelector("ms_method_details_extension_table", "ms_method_details_extension_column_selector");
    initColumnSelector("ms_tune_positive_table", "ms_tune_positive_column_selector");
    initColumnSelector("ms_tune_negative_table", "ms_tune_negative_column_selector");
    updateMsMethodPrimaryStickyOffsets();
    updateMsMethodTrailingStickyOffsets();
    window.addEventListener("resize", updateMsMethodPrimaryStickyOffsets);
    window.addEventListener("resize", updateMsMethodTrailingStickyOffsets);
    applyAllNumericDiffHighlights();
  </script>
</body>
</html>
"""
    output_path.write_text(html_content, encoding="utf-8")


def main() -> None:
    root_dir = Path.cwd()
    method_dirs = discover_method_dirs(root_dir)
    if not method_dirs:
        warn(f"No method directories matching *.m found in {root_dir}")
        raise SystemExit(1)

    summaries: list[MethodSummary] = []
    for method_dir in method_dirs:
        summary = parse_method_dir(method_dir)
        if summary is not None:
            summaries.append(summary)

    if not summaries:
        warn("No readable methods were found after parsing.")
        raise SystemExit(1)

    df = pd.DataFrame(asdict(summary) for summary in summaries)
    df.sort_values("method_dir", inplace=True)
    generated_at = datetime.now().isoformat(timespec="seconds")
    to_csv(df, root_dir / OUTPUT_CSV, generated_at)

    traces = build_gradient_traces(df)
    to_html(df, traces, root_dir / OUTPUT_HTML, generated_at)

    print(f"Wrote {len(df)} rows to {OUTPUT_CSV} and {OUTPUT_HTML}")


# Run viewer logic against uploaded ZIP(s), recursively discovering *.m folders.
OUTPUT_DIR = Path('/content/sample_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Click "Choose Files" and upload one or more .zip files containing *.m method folders...')
uploaded = files.upload()

zip_items = [(name, data) for name, data in uploaded.items() if name.lower().endswith('.zip')]
if not zip_items:
    raise RuntimeError('Please upload at least one .zip file.')

summaries: list[MethodSummary] = []
with tempfile.TemporaryDirectory(prefix='lcms_zip_', dir='/content') as tmp_dir:
    tmp_root = Path(tmp_dir)
    extract_root = tmp_root / 'extracted'
    extract_root.mkdir(parents=True, exist_ok=True)

    for zip_name, zip_bytes in zip_items:
        zip_path = tmp_root / zip_name
        zip_path.write_bytes(zip_bytes)
        dst = extract_root / Path(zip_name).stem
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(dst)

    method_dirs = sorted(p for p in extract_root.rglob('*.m') if p.is_dir())
    print(f'Found *.m dirs recursively: {len(method_dirs)}')
    for p in method_dirs[:30]:
        print(' -', p.relative_to(extract_root))

    if not method_dirs:
        raise RuntimeError('No *.m directories found in uploaded ZIP(s).')

    for method_dir in method_dirs:
        summary = parse_method_dir(method_dir)
        if summary is None:
            continue
        summary.method_dir = str(method_dir.relative_to(extract_root))
        summaries.append(summary)

# Temporary extracted data has been removed automatically here.
if not summaries:
    raise RuntimeError('No readable methods were found after parsing.')

df = pd.DataFrame(asdict(s) for s in summaries)
df.sort_values('method_dir', inplace=True)

CSV_PATH = OUTPUT_DIR / OUTPUT_CSV
HTML_PATH = OUTPUT_DIR / OUTPUT_HTML

generated_at = datetime.now().isoformat(timespec='seconds')
to_csv(df, CSV_PATH, generated_at)
traces = build_gradient_traces(df)
to_html(df, traces, HTML_PATH, generated_at)

print(f'Wrote {len(df)} rows')
print(f'CSV: {CSV_PATH}')
print(f'HTML: {HTML_PATH}')

from IPython.display import HTML as _HTML, display  # noqa: E402

display(_HTML(f'<p><b>CSV:</b> {CSV_PATH}</p>'))
if HTML_PATH.exists():
    display(_HTML(HTML_PATH.read_text(encoding='utf-8')))

files.download(str(CSV_PATH))
files.download(str(HTML_PATH))
